**Feature Extraction**


**Miedema Mixing Enthalpy + Standard Deviation σ**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

# --- Step 1: Upload Input File ---
uploaded = files.upload()
input_filename = next(iter(uploaded))
df_input = pd.read_excel(input_filename)

# --- Step 2: Complete Miedema Parameters (kJ/mol) ---
deltaH = {
    ('Cr', 'Fe'): -1,    ('Cr', 'Mn'): 0,     ('Cr', 'Nb'): -5,
    ('Cr', 'Sn'): -6,    ('Cr', 'Ti'): -7,    ('Cr', 'Zr'): -4,
    ('Fe', 'Mn'): 0,     ('Fe', 'Nb'): -16,   ('Fe', 'Sn'): -11,
    ('Fe', 'Ti'): -17,   ('Fe', 'Zr'): -10,   ('Mn', 'Nb'): 1,
    ('Mn', 'Sn'): -5,    ('Mn', 'Ti'): -8,    ('Mn', 'Zr'): -5,
    ('Nb', 'Sn'): 3,     ('Nb', 'Ti'): 2,     ('Nb', 'Zr'): 4,
    ('Sn', 'Ti'): -4,    ('Sn', 'Zr'): -2,    ('Ti', 'Zr'): 0
}

# --- Step 3: Prepare Calculation ---
required_columns = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
if not all(col in df_input.columns for col in required_columns):
    missing = [col for col in required_columns if col not in df_input.columns]
    raise ValueError(f"Missing columns: {missing}")

elements = ['Ti', 'Nb', 'Zr', 'Cr', 'Sn', 'Mn', 'Fe']

# --- Step 4: Calculate ΔH_mix and σΔH ---
deltaH_mix = []
deltaH_std = []  # To store standard deviations

for _, row in df_input.iterrows():
    comp = [row[col]/100 for col in required_columns]
    H_contributions = []  # Store all pairwise enthalpy contributions
    H_total = 0

    for i in range(len(elements)):
        for j in range(i+1, len(elements)):
            if comp[i] > 0 and comp[j] > 0:
                elem1, elem2 = sorted((elements[i], elements[j]))
                contribution = comp[i] * comp[j] * deltaH[(elem1, elem2)]
                H_contributions.append(contribution)
                H_total += contribution

    deltaH_mix.append(np.round(H_total, 4))
    # Calculate standard deviation of pairwise contributions
    deltaH_std.append(np.round(np.std(H_contributions), 4) if H_contributions else 0)

# --- Step 5: Save Results ---
df_output = df_input.copy()
df_output['ΔH_mix (kJ/mol)'] = deltaH_mix
df_output['σΔH (kJ/mol)'] = deltaH_std  # Add standard deviation column
output_filename = 'Miedema_Results_with_STD.xlsx'
df_output.to_excel(output_filename, index=False)

# --- Step 6: Download & Plot ---
files.download(output_filename)

# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# ΔH_mix plot
colors = ['red' if x < 0 else 'blue' for x in deltaH_mix]
ax1.bar(range(len(deltaH_mix)), deltaH_mix, color=colors)
ax1.set_ylabel('ΔH_mix (kJ/mol)')
ax1.set_title('Mixing Enthalpy and its Standard Deviation')
ax1.axhline(0, color='black', linestyle='--')
ax1.grid(axis='y', alpha=0.3)

# σΔH plot
ax2.bar(range(len(deltaH_std)), deltaH_std, color='purple')
ax2.set_xlabel('Alloy Sample')
ax2.set_ylabel('σΔH (kJ/mol)')
ax2.set_xticks(range(len(deltaH_std)))
ax2.set_xticklabels(range(1, len(deltaH_std)+1), rotation=45)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# --- Verification ---
print("CALCULATION COMPLETE")
print(f"Results saved to {output_filename}")
print("\nFirst 5 results with standard deviation:")
print(df_output.head())

print("\nKey Statistics:")
print(f"Average σΔH: {np.mean(deltaH_std):.4f} kJ/mol")
print(f"Max σΔH: {np.max(deltaH_std):.4f} kJ/mol (Sample {np.argmax(deltaH_std)+1})")

**Mixing Entropy**

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

# --- Step 1: Upload Input File ---
uploaded = files.upload()
input_filename = next(iter(uploaded))
df = pd.read_excel(input_filename)

# --- Step 2: Calculate Mixing Entropy (ΔS_mix) ---
def calculate_mixing_entropy(row):
    """
    Calculate ideal mixing entropy (ΔS_mix) for a given composition
    Formula: ΔS_mix = -R Σ [x_i * ln(x_i)]
    Where R = 8.314 J/(mol·K), x_i = atomic fraction
    """
    R = 8.314  # Gas constant in J/(mol·K)
    composition = row[['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']].values / 100
    composition = composition[composition > 0]  # Only consider elements present
    return -R * np.sum(composition * np.log(composition))

# Apply to all alloys
df['ΔS_mix (J/mol·K)'] = df.apply(calculate_mixing_entropy, axis=1)

# --- Step 3: Save Results ---
output_filename = 'Mixing_Entropy_Results.xlsx'
df.to_excel(output_filename, index=False)
files.download(output_filename)

# --- Step 4: Display Results ---
print("First 5 results:")
print(df.head())

# --- Optional Visualization ---
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.bar(range(len(df)), df['ΔS_mix (J/mol·K)'], color='green')
plt.xlabel('Alloy Sample')
plt.ylabel('ΔS_mix (J/mol·K)')
plt.title('Mixing Entropy of Alloys')
plt.xticks(range(len(df)), range(1, len(df)+1), rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Melting Point and Standard deviation of Melting Temperature **

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

# --- Step 1: Upload Input File ---
uploaded = files.upload()
input_filename = next(iter(uploaded))
df = pd.read_excel(input_filename)

# --- Step 2: Elemental Melting Points (K) ---
melting_points = {
    'Ti': 1941,  # K
    'Nb': 2750,
    'Zr': 2128,
    'Cr': 2180,
    'Sn': 505,
    'Mn': 1519,
    'Fe': 1811
}

# --- Step 3: Corrected σTm Calculation (Population Std) ---
def calculate_melting_features(row):
    comp = row[['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']].values / 100
    elements = ['Ti', 'Nb', 'Zr', 'Cr', 'Sn', 'Mn', 'Fe']

    # Filter elements present in alloy
    present_elements = [elem for elem, percent in zip(elements, comp) if percent > 0]
    present_percents = [percent for percent in comp if percent > 0]
    Tm_values = [melting_points[elem] for elem in present_elements]
    weights = present_percents

    # Weighted mean (unchanged)
    weighted_Tm = np.average(Tm_values, weights=weights)

    # CORRECTED: Population standard deviation (ddof=0)
    variance = np.average((Tm_values - weighted_Tm)**2, weights=weights)
    weighted_std = np.sqrt(variance)

    return weighted_Tm, weighted_std

# Apply to all alloys
df[['Tm (K)', 'σTm (K)']] = df.apply(calculate_melting_features, axis=1, result_type='expand')

# --- Step 4: Save Results ---
output_filename = 'Corrected_Melting_Temperature_Results.xlsx'
df.to_excel(output_filename, index=False)
files.download(output_filename)

# --- Verification ---
print("First 5 results:")
print(df.head())

**Ω Parameter**

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

def calculate_omega():
    """
    Calculate Ω parameter from uploaded file with ΔH_mix, ΔS_mix, and Tm columns.
    Works in Google Colab with file upload.
    """
    # Upload the file
    print("Please upload your input Excel file:")
    uploaded = files.upload()
    input_filename = next(iter(uploaded))

    # Load the data
    df = pd.read_excel(input_filename)

    # Verify required columns exist
    required_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)',
                    'ΔH_mix (kJ/mol)', 'ΔS_mix (J/mol·K)', 'Tm (K)']

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Calculate Ω parameter
    df['Ω'] = (df['Tm (K)'] * df['ΔS_mix (J/mol·K)'] / 1000) / abs(df['ΔH_mix (kJ/mol)'])

    # Classify phase stability
    df['Phase_Stability'] = np.where(
        df['Ω'] > 1, 'Solid Solution',
        np.where(df['Ω'] > 0, 'Intermetallic/Amorphous', 'Phase Separation')
    )

    # Save results
    output_filename = 'alloy_properties_with_omega.xlsx'
    df.to_excel(output_filename, index=False)

    # Download the results
    files.download(output_filename)

    print("\nCalculation complete. First 5 rows with Ω:")
    print(df[['Ti (%)', 'Nb (%)', 'Zr (%)', 'ΔH_mix (kJ/mol)',
              'ΔS_mix (J/mol·K)', 'Tm (K)', 'Ω', 'Phase_Stability']].head())

# Run the function
calculate_omega()

**Atomic and Electronic Features (δ, a, VEC, σVEC, χ, Δχ)**

In [ ]:

import pandas as pd
import numpy as np
from collections import defaultdict

# Atomic properties for each element (atomic radius in Å, electronegativity (Pauling scale), valence electrons)

element_properties = {
    'Ti': {'radius': 1.47, 'electroneg': 1.54, 'valence': 4},
    'Nb': {'radius': 1.45, 'electroneg': 1.6, 'valence': 5},
    'Zr': {'radius': 1.60, 'electroneg': 1.33, 'valence': 4},
    'Cr': {'radius': 1.28, 'electroneg': 1.66, 'valence': 6},
    'Sn': {'radius': 1.45, 'electroneg': 1.96, 'valence': 4},
    'Mn': {'radius': 1.27, 'electroneg': 1.55, 'valence': 7},
    'Fe': {'radius': 1.26, 'electroneg': 1.83, 'valence': 8}
}

def calculate_features(df):
    """
    Calculate alloy features from composition data
    """
    results = defaultdict(list)

    for _, row in df.iterrows():
        elements = ['Ti', 'Nb', 'Zr', 'Cr', 'Sn', 'Mn', 'Fe']
        compositions = [row[f'{e} (%)']/100 for e in elements]  # Convert % to fraction

        # Filter elements that are actually present in the alloy
        present_elements = [e for e, c in zip(elements, compositions) if c > 0]
        present_compositions = [c for c in compositions if c > 0]

        # Get properties for present elements
        radii = [element_properties[e]['radius'] for e in present_elements]
        valences = [element_properties[e]['valence'] for e in present_elements]
        electronegs = [element_properties[e]['electroneg'] for e in present_elements]

        # 1. Atomic size difference (δ)
        mean_radius = np.sum(np.array(radii) * np.array(present_compositions))
        sum_term = np.sum([present_compositions[i] * (1 - radii[i]/mean_radius)**2
                          for i in range(len(present_elements))])
        delta = np.sqrt(sum_term) * 100  # as percentage

        # 2. Mean atomic radius (a)
        a = mean_radius

        # 3. Valence electron concentration (VEC)
        vec = np.sum(np.array(valences) * np.array(present_compositions))

        # 4. Standard deviation of VEC
        vec_std = np.sqrt(np.sum([present_compositions[i] * (valences[i] - vec)**2
                                for i in range(len(present_elements))]))

        # 5. Mean electronegativity
        mean_electroneg = np.sum(np.array(electronegs) * np.array(present_compositions))

        # 6. Electronegativity difference (Δχ)
        delta_electroneg = np.sqrt(np.sum([present_compositions[i] * (electronegs[i] - mean_electroneg)**2
                                         for i in range(len(present_elements))]))

        # Store results
        results['δ (%)'].append(delta)
        results['a (Å)'].append(a)
        results['VEC'].append(vec)
        results['σVEC'].append(vec_std)
        results['Electronegativity'].append(mean_electroneg)
        results['Δχ'].append(delta_electroneg)

    return pd.DataFrame(results)

# Main execution
from google.colab import files

# Upload your data file
uploaded = files.upload()
filename = next(iter(uploaded))

# Read the input file
df = pd.read_excel(filename) if filename.endswith('.xlsx') else pd.read_csv(filename)

# Calculate features
feature_df = calculate_features(df)

# Combine with original data
result_df = pd.concat([df, feature_df], axis=1)

# Save results
output_filename = 'alloy_features_' + filename
result_df.to_excel(output_filename, index=False) if filename.endswith('.xlsx') else result_df.to_csv(output_filename, index=False)

# Download the results
files.download(output_filename)

print("Feature extraction complete Downloaded file:", output_filename)

**Machine Learning and Deep Learning Models**

**Random Forest**

In [ ]:
# Random Forest Model for Predicting Mechanical Properties


!pip install -q shap openpyxl scikit-learn pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# Train function - NO GRIDSEARCH, use fixed optimal params
def train_model(X, y, target_name, params):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Use FIXED hyperparameters - no search
    model = RandomForestRegressor(**params, random_state=42, n_jobs=1)
    model.fit(X_train_sc, y_train)

    # CV scores
    cv = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"  CV R2: {cv.mean():.4f} ± {cv.std():.4f}")

    y_tr_pred = model.predict(X_train_sc)
    y_te_pred = model.predict(X_test_sc)

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    print(f"  Train R2: {tr_met['R2']:.4f} | Test R2: {te_met['R2']:.4f}")

    return {
        'model': model, 'scaler': scaler,
        'train_metrics': tr_met, 'test_metrics': te_met,
        'y_train': y_train.values, 'y_test': y_test.values,
        'y_train_pred': y_tr_pred, 'y_test_pred': y_te_pred
    }

# OPTIMAL HYPERPARAMETERS (from your previous results)
# Use best parameters for each target based on your analysis
optimal_params = {
    'Youngs Modulus (MPa)': {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    'Yield Strength (MPa)': {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    'Ultimate Compressive Strength (MPa)': {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    'Ultimate Strain': {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    'Toughness (MJ/m³)': {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
}

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL")
print("="*80)

results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    results_before[target] = train_model(X_all, y_all[target], target, optimal_params[target])
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL")
print("="*80)

results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    results_after[target] = train_model(X_clean, y_clean[target], target, optimal_params[target])
    gc.collect()

# Smart selection
# Young's Modulus and Yield Strength AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness BEFORE outlier removal
targets_after = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
targets_before = ['Ultimate Compressive Strength (MPa)', 'Ultimate Strain', 'Toughness (MJ/m³)']
results_viz = {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols}

# PRINT METRICS
print("\n" + "="*80)
print("METRICS - BEFORE")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("METRICS - AFTER")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

# 1. Actual vs Predicted - Combined
print("\n1. Actual vs Predicted - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('Random Forest', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_pred_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution - Combined
print("\n2. Error Distribution - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=15, alpha=0.7, edgecolor='black', color='coral', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=2.5, label=f'Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2, label='Zero')
    ax.set_xlabel('Error', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('error_dist_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2b. Error Distribution - Individual plots
print("\n2b. Error Distribution - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.hist(errors, bins=20, alpha=0.7, edgecolor='black', color='mediumpurple', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=3, label=f'Normal Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2.5, label='Zero Error')
    ax.set_xlabel('Prediction Error', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'Random Forest: {t}', fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'error_dist_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Error distribution plots saved")

# 3. Residuals - Combined
print("\n3. Residuals - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], resid, alpha=0.6, s=50)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)

    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {np.mean(resid):.2f}, Std: {std_resid:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

if len(target_cols) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_analysis_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3b. Residuals - Individual plots
print("\n3b. Residuals - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], resid, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Residual')

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8, label=f'±2σ ({2*std_resid:.2f})')
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Residuals', fontsize=12, fontweight='bold')
    ax.set_title(f'Random Forest: {t}',
                 fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'residual_analysis_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Residual analysis plots saved")

# 4. Percentage Error Analysis - Individual plots ONLY
print("\n4. Percentage Error Analysis - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    percent_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], percent_errors, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')

    # Add horizontal lines at ±20%
    ax.axhline(y=20, color='orange', linestyle=':', linewidth=2, alpha=0.8, label='±20%')
    ax.axhline(y=-20, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    mean_pe = np.mean(percent_errors)
    std_pe = np.std(percent_errors)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Percentage Error (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Percentage Error Analysis - {t}\nMean: {mean_pe:.2f}%, Std: {std_pe:.2f}%',
                 fontsize=14, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'percentage_error_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Percentage error analysis plots saved")

# 5. Radar Chart - Combined (Test Set Only with 5 metrics)
print("\n5. Radar Charts - Combined (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            # Lower error = better, so invert: 1 - (error/100)
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='orange')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='orange')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=12, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('radar_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5b. Radar Charts - Individual plots
print("\n5b. Radar Charts - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    metric_values_text = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        metric_values_text.append(f"{val:.3f}")
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=3, color='cadetblue', markersize=10)
    ax.fill(angles_plot, test_vals_plot, alpha=0.3, color='cadetblue')
    ax.set_xticks(angles)
    ax.set_xticklabels([f'{label}\n({val})' for label, val in zip(metrics_labels, metric_values_text)],
                       fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
    ax.set_title(f'Random Forest: {t}', fontsize=12, fontweight='bold', pad=25)
    ax.grid(True, linewidth=1.2, alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'radar_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Radar chart plots saved")

# MINIMAL SHAP (using smart selected models)
print("\n6. SHAP Analysis (Using Smart Selected Models)")
shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    # Get the correct training data based on selection
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)

    # Ultra-minimal SHAP (only 15 samples from FULL feature set)
    n_samples = min(15, len(X_scaled))
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    # SHAP with tiny background
    bg = shap.sample(X_sample, min(8, len(X_sample)))
    exp = shap.TreeExplainer(r['model'], bg)

    # Compute SHAP on full feature set
    sv = exp.shap_values(X_sample, check_additivity=False)

    # Store full SHAP values for summary plots
    shap_values_all[t] = sv[:, len(input_cols):]  # Only engineered features

    # Extract only engineered features importance
    shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

    del exp, bg
    gc.collect()

# Normalize SHAP values for heatmap (0-1 scale)
print("\n7. SHAP Heatmap (Normalized)")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)

# Normalize each column to 0-1
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='viridis', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('Random Forest',
            fontsize=12, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('shap_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Rankings with color palette
print("\n8. SHAP Rankings (Color Palette) - Combined")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()

# Create color palette
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]

    # Use color palette
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=12, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Ranking', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_rankings_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Rankings - Individual plots for each target
print("\n8b. SHAP Rankings - Individual Plots")

# Create color palette
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for t in target_cols:
    print(f"  {t}...")
    fig, ax = plt.subplots(figsize=(10, 8))

    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]

    # Use color palette
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.8)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'Random Forest: {t}', fontsize=12, fontweight='bold', pad=15)

    # Add value labels on bars
    for i, (idx, val) in enumerate(zip(sidx, imp[sidx])):
        ax.text(val, i, f' {val:.4f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'shap_ranking_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP ranking plots saved")


# SHAP Summary Plots (individual for each target)
print("\n9. SHAP Summary Plots")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]

    # Get the sample data again for plotting based on selection
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]

    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'Random Forest: {t}', fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

# SHAP Dependence Plots (top 3 features for each target)
print("\n10. SHAP Dependence Plots (Top 3 Features)")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    # Get the sample data again for plotting based on selection
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]

    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    # Get top 3 features by SHAP importance
    imp = shap_imp[t]
    top3_idx = np.argsort(imp)[::-1][:3]
    top3_features = [feature_cols[i] for i in top3_idx]

    # Create 1x3 subplot for top 3 features
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for plot_idx, feat_idx in enumerate(top3_idx):
        ax = axes[plot_idx]

        # SHAP dependence plot
        shap.dependence_plot(
            feat_idx,
            shap_values_all[t],
            X_sample_feat,
            feature_names=feature_cols,
            show=False,
            ax=ax
        )
        ax.set_title(f'{feature_cols[feat_idx]}', fontsize=12, fontweight='bold')
        ax.set_xlabel(f'{feature_cols[feat_idx]} (scaled)', fontsize=10, fontweight='bold')
        ax.set_ylabel('SHAP value', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Random Forest: {t}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'shap_dependence_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP dependence plots saved")

print("\n" + "="*80)
print("✓ COMPLETE!")
print("="*80)
print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nModel Selection Summary:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")

print("\nSelected Model Performance:")
for t in target_cols:
    selection = 'AFTER' if t in targets_after else 'BEFORE'
    r2_score_val = results_viz[t]['test_metrics']['R2']
    print(f"  {t} ({selection}): R²={r2_score_val:.4f}")

print("\n✓ All plots saved!")

# ============================================================================
# NEW SECTION: SAVE AND DOWNLOAD SELECTED MODELS
# Young's Modulus and Yield Strength: AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness: BEFORE outlier removal
# ============================================================================

print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save individual models based on selection criteria
saved_files = []

print("\nSelected models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")
print()

for target in target_cols:
    target_clean = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

    # Determine which result set to use
    if target in targets_after:
        result_source = results_after[target]
        selection_type = 'after'
    else:
        result_source = results_before[target]
        selection_type = 'before'

    # Create a comprehensive save package
    model_package = {
        'target_name': target,
        'selection_type': selection_type,
        'model': result_source['model'],
        'scaler': result_source['scaler'],
        'params': optimal_params[target],
        'train_metrics': result_source['train_metrics'],
        'test_metrics': result_source['test_metrics'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp,
        'outliers_removed': outliers if selection_type == 'after' else []
    }

    # Save as pickle
    filename_pkl = f'randomforest_model_{target_clean}_{selection_type}_outliers_{timestamp}.pkl'
    with open(filename_pkl, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename_pkl)
    print(f"  ✓ {target} ({selection_type.upper()}): R²={result_source['test_metrics']['R2']:.4f} → {filename_pkl}")

# Save all models together
all_models_package = {
    'models': {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols},
    'selection_criteria': {
        'after_outlier_removal': targets_after,
        'before_outlier_removal': targets_before
    },
    'optimal_params': optimal_params,
    'input_cols': input_cols,
    'feature_cols': feature_cols,
    'target_cols': target_cols,
    'outliers_removed': outliers,
    'timestamp': timestamp,
    'X_clean': X_clean,
    'y_clean': y_clean,
    'X_all': X_all,
    'y_all': y_all
}

all_models_filename = f'randomforest_all_selected_models_{timestamp}.pkl'
with open(all_models_filename, 'wb') as f:
    pickle.dump(all_models_package, f)
saved_files.append(all_models_filename)
print(f"\n✓ Saved all models package: {all_models_filename}")

# Create a summary document
summary_filename = f'randomforest_model_summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("RANDOM FOREST MODEL SUMMARY - SELECTIVE MODEL SELECTION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Outliers removed: {len(outliers)} samples\n")
    f.write(f"Outlier indices: {sorted(outliers)}\n\n")

    f.write("MODEL SELECTION CRITERIA:\n")
    f.write("-"*80 + "\n")
    f.write(f"AFTER outlier removal:\n")
    for t in targets_after:
        f.write(f"  • {t}\n")
    f.write(f"\nBEFORE outlier removal:\n")
    for t in targets_before:
        f.write(f"  • {t}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("MODEL DETAILS:\n")
    f.write("-"*80 + "\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        r2_score_val = result_source['test_metrics']['R2']
        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write(f"  Test R²: {r2_score_val:.4f}\n")
        f.write(f"  Parameters: {optimal_params[target]}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED METRICS:\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write("  Train Metrics:\n")
        for k, v in result_source['train_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")
        f.write("  Test Metrics:\n")
        for k, v in result_source['test_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("HOW TO LOAD AND USE THE MODELS:\n")
    f.write("="*80 + "\n\n")
    f.write("# Load a single model:\n")
    f.write("import pickle\n")
    f.write("import numpy as np\n\n")
    f.write("with open('randomforest_model_[target]_[after/before]_outliers_[timestamp].pkl', 'rb') as f:\n")
    f.write("    model_package = pickle.load(f)\n\n")
    f.write("model = model_package['model']\n")
    f.write("scaler = model_package['scaler']\n")
    f.write("selection_type = model_package['selection_type']  # 'after' or 'before'\n\n")
    f.write("# For prediction:\n")
    f.write("# 1. Prepare your input data (same features as training)\n")
    f.write("# 2. Scale the data\n")
    f.write("X_scaled = scaler.transform(X_new)\n\n")
    f.write("# 3. Make predictions\n")
    f.write("predictions = model.predict(X_scaled)\n\n")
    f.write("# Load all models at once:\n")
    f.write("with open('randomforest_all_selected_models_[timestamp].pkl', 'rb') as f:\n")
    f.write("    all_models = pickle.load(f)\n\n")
    f.write("# Check selection criteria\n")
    f.write("print('After outlier removal:', all_models['selection_criteria']['after_outlier_removal'])\n")
    f.write("print('Before outlier removal:', all_models['selection_criteria']['before_outlier_removal'])\n\n")
    f.write("# Access specific model\n")
    f.write("youngs_modulus_model = all_models['models']['Youngs Modulus (MPa)']['model']\n")
    f.write("youngs_modulus_scaler = all_models['models']['Youngs Modulus (MPa)']['scaler']\n")

saved_files.append(summary_filename)
print(f"✓ Saved summary: {summary_filename}")

# DOWNLOAD FILES
print("\n" + "="*80)
print("DOWNLOADING FILES")
print("="*80)

for filename in saved_files:
    print(f"  Downloading: {filename}")
    files.download(filename)

print("\n" + "="*80)
print("MODEL SAVING COMPLETE!")
print("="*80)
print(f"\nTotal files saved and downloaded: {len(saved_files)}")
print("\nFiles include:")
print(f"  • {len(target_cols)} individual model files (.pkl)")
print(f"  • 1 combined models file (.pkl)")
print(f"  • 1 summary file (.txt)")

print("\n" + "="*80)
print("SELECTION SUMMARY:")
print("="*80)
print("\nModels selected AFTER outlier removal:")
for target in targets_after:
    print(f"  • {target}: R²={results_after[target]['test_metrics']['R2']:.4f}")
print("\nModels selected BEFORE outlier removal:")
for target in targets_before:
    print(f"  • {target}: R²={results_before[target]['test_metrics']['R2']:.4f}")

print("\n" + "="*80)
print("✓ ALL DONE! Random Forest models saved and downloaded.")
print("="*80)

**XGBoost**

In [ ]:
# XGBoost Model for Predicting Mechanical Properties


!pip install -q xgboost shap openpyxl scikit-learn pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - XGBoost Model")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# Train function - XGBoost with fixed optimal params
def train_model(X, y, target_name, params):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Use FIXED hyperparameters for XGBoost
    model = XGBRegressor(**params, random_state=42, n_jobs=1, verbosity=0)
    model.fit(X_train_sc, y_train)

    # CV scores
    cv = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"  CV R2: {cv.mean():.4f} ± {cv.std():.4f}")

    y_tr_pred = model.predict(X_train_sc)
    y_te_pred = model.predict(X_test_sc)

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    print(f"  Train R2: {tr_met['R2']:.4f} | Test R2: {te_met['R2']:.4f}")

    return {
        'model': model, 'scaler': scaler,
        'train_metrics': tr_met, 'test_metrics': te_met,
        'y_train': y_train.values, 'y_test': y_test.values,
        'y_train_pred': y_tr_pred, 'y_test_pred': y_te_pred
    }

# FINAL OPTIMIZED HYPERPARAMETERS FOR XGBOOST
# Analysis: Current results show Young's Modulus (0.548) and Yield Strength (0.317) need more work
# UCS (0.970), Toughness (0.985), Ultimate Strain (0.925) are excellent - keep their params
optimal_params_xgb = {
    'Youngs Modulus (MPa)': {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8},
    'Yield Strength (MPa)': {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8},
    'Ultimate Compressive Strength (MPa)': {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8},
    'Ultimate Strain': {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8},
    'Toughness (MJ/m³)': {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}
}

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - XGBoost")
print("="*80)

results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    results_before[target] = train_model(X_all, y_all[target], target, optimal_params_xgb[target])
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - XGBoost")
print("="*80)

results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    results_after[target] = train_model(X_clean, y_clean[target], target, optimal_params_xgb[target])
    gc.collect()

# Smart selection
# Young's Modulus and Yield Strength AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness BEFORE outlier removal
targets_after = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
targets_before = ['Ultimate Compressive Strength (MPa)', 'Ultimate Strain', 'Toughness (MJ/m³)']
results_viz = {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols}

# PRINT METRICS
print("\n" + "="*80)
print("METRICS - BEFORE OUTLIER REMOVAL (XGBoost)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("METRICS - AFTER OUTLIER REMOVAL (XGBoost)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS - XGBoost")
print("="*80)

# 1. Actual vs Predicted
print("\n1. Actual vs Predicted")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('XGBoost', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('xgb_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution (Histogram style)
print("2. Error Distribution")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax.axvline(x=mean_error, color='green', linestyle='-', linewidth=2,
               label=f'Mean: {mean_error:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}\nError Distribution', fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('xgb_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3. Residuals
print("3. Residuals")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=60, color='darkorange', edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    ax.axhline(residuals.mean(), color='blue', linestyle=':', linewidth=2,
              label=f'Mean: {residuals.mean():.2f}')
    std_resid = np.std(residuals)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {residuals.mean():.2f}, Std: {std_resid:.2f}',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('xgb_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3b. Percentage Error Analysis
print("3b. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    r = results_viz[t]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / r['y_test']) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    ax.set_title(f'{t}\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                 fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('xgb_percentage_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 4. Radar Chart (Test Set Only with 5 metrics)
print("4. Radar Charts (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)
    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]
    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='orange')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='orange')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts - XGBoost (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('xgb_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# MINIMAL SHAP - COMPLETE FIX FOR XGBOOST COMPATIBILITY
print("\n5. SHAP Analysis - XGBoost (Using Smart Selected Models)")
shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    n_samples = min(15, len(X_scaled))
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    # FIX: Create a wrapper function for XGBoost model to avoid base_score issue
    def model_predict(X):
        return r['model'].predict(X)

    # Use KernelExplainer instead of TreeExplainer for XGBoost compatibility
    # This is slower but always works
    background = shap.sample(X_sample, min(5, len(X_sample)))
    exp = shap.KernelExplainer(model_predict, background)
    sv = exp.shap_values(X_sample)

    # Handle 2D vs 1D SHAP values
    if len(sv.shape) == 1:
        sv = sv.reshape(-1, 1)
    if sv.shape[1] != X_sample.shape[1]:
        sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

    shap_values_all[t] = sv[:, len(input_cols):]
    shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

    del exp
    gc.collect()

# Normalize SHAP values for heatmap
print("\n6. SHAP Heatmap (Normalized) - XGBoost")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('SHAP Feature Importance (Normalized) - XGBoost\n(Engineered Features)',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('xgb_shap_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Rankings with color palette
print("7. SHAP Rankings (Color Palette) - XGBoost")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Rankings - XGBoost', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('xgb_shap_rankings.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Summary Plots
print("\n8. SHAP Summary Plots - XGBoost")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'SHAP Summary - XGBoost: {t}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'xgb_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

print("\n" + "="*80)
print("✓ COMPLETE - XGBoost!")
print("="*80)
print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nModel Selection Summary:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")

print("\nSelected Model Performance:")
for t in target_cols:
    selection = 'AFTER' if t in targets_after else 'BEFORE'
    r2_score = results_viz[t]['test_metrics']['R2']
    print(f"  {t} ({selection}): R²={r2_score:.4f}")

print("\n✓ All XGBoost plots saved!")
print("\nGenerated Files:")
print("  • xgb_actual_vs_pred.png")
print("  • xgb_error_distribution.png")
print("  • xgb_residuals.png")
print("  • xgb_percentage_error_analysis.png")
print("  • xgb_radar.png")
print("  • xgb_shap_heatmap.png")
print("  • xgb_shap_rankings.png")
print("  • xgb_shap_summary_[target].png (5 files)")

# ============================================================================
# NEW SECTION: SAVE AND DOWNLOAD SELECTED MODELS
# Young's Modulus and Yield Strength: AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness: BEFORE outlier removal
# ============================================================================

print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save individual models based on selection criteria
saved_files = []

print("\nSelected models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")
print()

for target in target_cols:
    target_clean = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

    # Determine which result set to use
    if target in targets_after:
        result_source = results_after[target]
        selection_type = 'after'
    else:
        result_source = results_before[target]
        selection_type = 'before'

    # Create a comprehensive save package
    model_package = {
        'target_name': target,
        'selection_type': selection_type,
        'model': result_source['model'],
        'scaler': result_source['scaler'],
        'params': optimal_params_xgb[target],
        'train_metrics': result_source['train_metrics'],
        'test_metrics': result_source['test_metrics'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp,
        'outliers_removed': outliers if selection_type == 'after' else []
    }

    # Save as pickle
    filename_pkl = f'xgboost_model_{target_clean}_{selection_type}_outliers_{timestamp}.pkl'
    with open(filename_pkl, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename_pkl)
    print(f"  ✓ {target} ({selection_type.upper()}): R²={result_source['test_metrics']['R2']:.4f} → {filename_pkl}")

# Save all models together
all_models_package = {
    'models': {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols},
    'selection_criteria': {
        'after_outlier_removal': targets_after,
        'before_outlier_removal': targets_before
    },
    'optimal_params': optimal_params_xgb,
    'input_cols': input_cols,
    'feature_cols': feature_cols,
    'target_cols': target_cols,
    'outliers_removed': outliers,
    'timestamp': timestamp,
    'X_clean': X_clean,
    'y_clean': y_clean,
    'X_all': X_all,
    'y_all': y_all
}

all_models_filename = f'xgboost_all_selected_models_{timestamp}.pkl'
with open(all_models_filename, 'wb') as f:
    pickle.dump(all_models_package, f)
saved_files.append(all_models_filename)
print(f"\n✓ Saved all models package: {all_models_filename}")

# Create a summary document
summary_filename = f'xgboost_model_summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("XGBOOST MODEL SUMMARY - SELECTIVE MODEL SELECTION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Outliers removed: {len(outliers)} samples\n")
    f.write(f"Outlier indices: {sorted(outliers)}\n\n")

    f.write("MODEL SELECTION CRITERIA:\n")
    f.write("-"*80 + "\n")
    f.write(f"AFTER outlier removal:\n")
    for t in targets_after:
        f.write(f"  • {t}\n")
    f.write(f"\nBEFORE outlier removal:\n")
    for t in targets_before:
        f.write(f"  • {t}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("MODEL DETAILS:\n")
    f.write("-"*80 + "\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        r2_score_val = result_source['test_metrics']['R2']
        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write(f"  Test R²: {r2_score_val:.4f}\n")
        f.write(f"  Parameters: {optimal_params_xgb[target]}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED METRICS:\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write("  Train Metrics:\n")
        for k, v in result_source['train_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")
        f.write("  Test Metrics:\n")
        for k, v in result_source['test_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("HOW TO LOAD AND USE THE MODELS:\n")
    f.write("="*80 + "\n\n")
    f.write("# Load a single model:\n")
    f.write("import pickle\n")
    f.write("import numpy as np\n\n")
    f.write("with open('xgboost_model_[target]_[after/before]_outliers_[timestamp].pkl', 'rb') as f:\n")
    f.write("    model_package = pickle.load(f)\n\n")
    f.write("model = model_package['model']\n")
    f.write("scaler = model_package['scaler']\n")
    f.write("selection_type = model_package['selection_type']  # 'after' or 'before'\n\n")
    f.write("# For prediction:\n")
    f.write("# 1. Prepare your input data (same features as training)\n")
    f.write("# 2. Scale the data\n")
    f.write("X_scaled = scaler.transform(X_new)\n\n")
    f.write("# 3. Make predictions\n")
    f.write("predictions = model.predict(X_scaled)\n\n")
    f.write("# Load all models at once:\n")
    f.write("with open('xgboost_all_selected_models_[timestamp].pkl', 'rb') as f:\n")
    f.write("    all_models = pickle.load(f)\n\n")
    f.write("# Check selection criteria\n")
    f.write("print('After outlier removal:', all_models['selection_criteria']['after_outlier_removal'])\n")
    f.write("print('Before outlier removal:', all_models['selection_criteria']['before_outlier_removal'])\n\n")
    f.write("# Access specific model\n")
    f.write("youngs_modulus_model = all_models['models']['Youngs Modulus (MPa)']['model']\n")
    f.write("youngs_modulus_scaler = all_models['models']['Youngs Modulus (MPa)']['scaler']\n")

saved_files.append(summary_filename)
print(f"✓ Saved summary: {summary_filename}")

# DOWNLOAD FILES
print("\n" + "="*80)
print("DOWNLOADING FILES")
print("="*80)

for filename in saved_files:
    print(f"  Downloading: {filename}")
    files.download(filename)

print("\n" + "="*80)
print("MODEL SAVING COMPLETE!")
print("="*80)
print(f"\nTotal files saved and downloaded: {len(saved_files)}")
print("\nFiles include:")
print(f"  • {len(target_cols)} individual model files (.pkl)")
print(f"  • 1 combined models file (.pkl)")
print(f"  • 1 summary file (.txt)")

print("\n" + "="*80)
print("SELECTION SUMMARY:")
print("="*80)
print("\nModels selected AFTER outlier removal:")
for target in targets_after:
    print(f"  • {target}: R²={results_after[target]['test_metrics']['R2']:.4f}")
print("\nModels selected BEFORE outlier removal:")
for target in targets_before:
    print(f"  • {target}: R²={results_before[target]['test_metrics']['R2']:.4f}")

print("\n" + "="*80)
print("✓ ALL DONE! XGBoost models saved and downloaded.")
print("="*80)

**LightGBM**

In [ ]:
# LightGBM Model for Predicting Mechanical Properties

!pip install -q lightgbm shap openpyxl scikit-learn pandas numpy matplotlib seaborn scipy optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - Improved LightGBM Model")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# ENHANCED HYPERPARAMETER TUNING
print("\n" + "="*80)
print("PHASE 0: ENHANCED HYPERPARAMETER TUNING - LightGBM")
print("="*80)

def objective_enhanced(trial, X_train, y_train, target_name):
    """Enhanced objective function with expanded search space"""

    # Determine if this is a problematic target requiring more regularization
    problematic_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
    is_problematic = target_name in problematic_targets

    params = {
        'num_leaves': trial.suggest_int('num_leaves', 10, 80),  # Expanded range
        'max_depth': trial.suggest_int('max_depth', 3, 12),  # Expanded range
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),  # Lower bound
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50),  # Expanded range
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),  # Expanded upper bound
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),  # Expanded upper bound
        'min_child_samples': trial.suggest_int('min_child_samples', 3, 50),  # Expanded range
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),  # NEW
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 1.0),  # NEW - helps with overfitting
    }

    # For problematic targets, bias towards more regularization
    if is_problematic:
        params['subsample'] = trial.suggest_float('subsample', 0.5, 0.85)  # More aggressive
        params['colsample_bytree'] = trial.suggest_float('colsample_bytree', 0.5, 0.85)

    model = LGBMRegressor(**params, random_state=42, verbose=-1, n_jobs=1)

    # Use CV to evaluate
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)

    return scores.mean()

# Prepare data for tuning
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

scaler_tune = StandardScaler()
X_train_scaled = scaler_tune.fit_transform(X_train_full)
X_test_scaled = scaler_tune.transform(X_test_full)

# Tune hyperparameters for each target with MORE TRIALS
optimal_params_lgb = {}
tuning_trials = {
    'Youngs Modulus (MPa)': 100,  # Most problematic - more trials
    'Yield Strength (MPa)': 100,  # Problematic - more trials
    'Ultimate Compressive Strength (MPa)': 50,  # Good performance - fewer trials
    'Ultimate Strain': 50,  # Good performance - fewer trials
    'Toughness (MJ/m³)': 50  # Good performance - fewer trials
}

for target in target_cols:
    n_trials = tuning_trials[target]
    print(f"\nTuning: {target} (with {n_trials} trials)")

    sampler = TPESampler(seed=42)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: objective_enhanced(trial, X_train_scaled, y_train_full[target].values, target),
        n_trials=n_trials,
        show_progress_bar=True
    )

    optimal_params_lgb[target] = study.best_params
    print(f"  Best R2 (CV): {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")
    gc.collect()

# ENHANCED TRAINING FUNCTION with Early Stopping
def train_model_lgb_enhanced(X, y, target_name, params, use_ensemble=False):
    """Enhanced training with early stopping and optional ensemble"""

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    if use_ensemble:
        # Ensemble approach: train multiple models with different seeds
        print(f"  Using ensemble (5 models)...")
        models = []
        train_preds_all = []
        test_preds_all = []

        for seed in [42, 123, 456, 789, 2024]:
            model = LGBMRegressor(**params, random_state=seed, verbose=-1, n_jobs=1)

            # Fit with early stopping
            model.fit(
                X_train_sc, y_train,
                eval_set=[(X_test_sc, y_test)],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
            )

            models.append(model)
            train_preds_all.append(model.predict(X_train_sc))
            test_preds_all.append(model.predict(X_test_sc))

        # Average predictions
        y_tr_pred = np.mean(train_preds_all, axis=0)
        y_te_pred = np.mean(test_preds_all, axis=0)

        # Use first model for other purposes (feature importance, etc.)
        model = models[0]

    else:
        # Single model with early stopping
        model = LGBMRegressor(**params, random_state=42, verbose=-1, n_jobs=1)

        model.fit(
            X_train_sc, y_train,
            eval_set=[(X_test_sc, y_test)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        y_tr_pred = model.predict(X_train_sc)
        y_te_pred = model.predict(X_test_sc)

    # CV scores
    cv = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"  CV R2: {cv.mean():.4f} ± {cv.std():.4f}")

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    # Calculate overfitting score
    overfit_score = tr_met['R2'] - te_met['R2']
    print(f"  Train R2: {tr_met['R2']:.4f} | Test R2: {te_met['R2']:.4f} | Overfit: {overfit_score:.4f}")

    if overfit_score > 0.10:
        print(f"  ⚠️  WARNING: Significant overfitting detected (gap > 0.10)")

    return {
        'model': model if not use_ensemble else models,
        'scaler': scaler,
        'train_metrics': tr_met,
        'test_metrics': te_met,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_pred': y_tr_pred,
        'y_test_pred': y_te_pred,
        'is_ensemble': use_ensemble
    }

# Import lightgbm for early stopping
import lightgbm as lgb

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - Enhanced LightGBM")
print("="*80)

# Decide which targets need ensemble
ensemble_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']

results_before = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_before[target] = train_model_lgb_enhanced(
        X_all, y_all[target], target,
        optimal_params_lgb[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - Enhanced LightGBM")
print("="*80)

results_after = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_after[target] = train_model_lgb_enhanced(
        X_clean, y_clean[target], target,
        optimal_params_lgb[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Smart selection
yield_target = 'Yield Strength (MPa)'
results_viz = {t: (results_after[t] if t == yield_target else results_before[t]) for t in target_cols}

# COMPREHENSIVE METRICS COMPARISON
print("\n" + "="*80)
print("METRICS COMPARISON - BEFORE vs AFTER OUTLIER REMOVAL")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement']
print("\n", comparison_df.round(4))

# PRINT DETAILED METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS (keeping the same visualization structure as original)
print("\n" + "="*80)
print("GENERATING PLOTS - Enhanced LightGBM")
print("="*80)

# 1. Actual vs Predicted
print("\n1. Actual vs Predicted")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')

    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('LightGBM', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('lgb_improved_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution
print("2. Error Distribution")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax.axvline(x=mean_error, color='green', linestyle='-', linewidth=2,
               label=f'Mean: {mean_error:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}\nError Distribution', fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('lgb_improved_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3. Residuals
print("3. Residuals")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=60, color='darkorange', edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    ax.axhline(residuals.mean(), color='blue', linestyle=':', linewidth=2,
              label=f'Mean: {residuals.mean():.2f}')
    std_resid = np.std(residuals)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {residuals.mean():.2f}, Std: {std_resid:.2f}',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('lgb_improved_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 4. Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    r = results_viz[t]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    ax.set_title(f'{t}\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                 fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('lgb_improved_percentage_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5. Radar Chart
print("5. Radar Charts (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)
    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]
    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='green')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='green')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts - Enhanced LightGBM (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('lgb_improved_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Analysis
print("\n6. SHAP Analysis - LightGBM")
shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    X_data = X_clean if t == yield_target else X_all

    # Handle ensemble models
    if r.get('is_ensemble', False):
        model_to_use = r['model'][0]  # Use first model from ensemble
    else:
        model_to_use = r['model']

    X_scaled = r['scaler'].transform(X_data)
    n_samples = min(15, len(X_scaled))
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    try:
        exp = shap.TreeExplainer(model_to_use)
        sv = exp.shap_values(X_sample)

        if isinstance(sv, list):
            sv = sv[0]
        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del exp
    except Exception as e:
        print(f"    Warning: TreeExplainer failed for {t}, using KernelExplainer")
        def model_predict(X):
            return model_to_use.predict(X)

        background = shap.sample(X_sample, min(5, len(X_sample)))
        exp = shap.KernelExplainer(model_predict, background)
        sv = exp.shap_values(X_sample)

        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del exp

    gc.collect()

# 7. SHAP Heatmap
print("\n7. SHAP Heatmap (Normalized)")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('SHAP Feature Importance (Normalized) - Enhanced LightGBM\n(Engineered Features)',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lgb_improved_shap_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 8. SHAP Rankings
print("8. SHAP Rankings (Color Palette)")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Rankings - Enhanced LightGBM', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('lgb_improved_shap_rankings.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 9. SHAP Summary Plots
print("\n9. SHAP Summary Plots")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]
    X_data = X_clean if t == yield_target else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'SHAP Summary - Enhanced LightGBM: {t}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'lgb_improved_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

# FINAL SUMMARY
print("\n" + "="*80)
print("FINAL SUMMARY - Enhanced LightGBM Model")
print("="*80)

print("\nHYPERPARAMETER TUNING SUMMARY:")
for t in target_cols:
    print(f"\n{t}:")
    print(f"  Trials: {tuning_trials[t]}")
    print(f"  Best params: {optimal_params_lgb[t]}")

print("\n" + "="*80)
print("PERFORMANCE IMPROVEMENTS:")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("OVERFITTING ANALYSIS:")
print("="*80)
for t in target_cols:
    before_gap = results_before[t]['train_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    after_gap = results_after[t]['train_metrics']['R2'] - results_after[t]['test_metrics']['R2']

    print(f"\n{t}:")
    print(f"  Before removal: Train-Test gap = {before_gap:.4f}")
    print(f"  After removal:  Train-Test gap = {after_gap:.4f}")

    if after_gap < 0.05:
        print(f"  ✓ Excellent generalization")
    elif after_gap < 0.10:
        print(f"  ✓ Good generalization")
    elif after_gap < 0.15:
        print(f"  ⚠ Moderate overfitting")
    else:
        print(f"  ❌ Significant overfitting - consider more regularization")

print("\n" + "="*80)
print("✓ COMPLETE - Enhanced LightGBM Model!")
print("="*80)

print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nYield Strength (Primary Focus):")
print(f"  Before: R²={results_before[yield_target]['test_metrics']['R2']:.4f}")
print(f"  After:  R²={results_after[yield_target]['test_metrics']['R2']:.4f}")
print(f"  Improvement: {results_after[yield_target]['test_metrics']['R2'] - results_before[yield_target]['test_metrics']['R2']:.4f}")

print("\n✓ All enhanced LightGBM plots saved!")

**CatBoost**

In [ ]:
# CatBoost Model for Predicting Mechanical Properties
# Enhanced Hyperparameter Tuning with Overfitting Prevention

!pip install -q catboost shap openpyxl scikit-learn pandas numpy matplotlib seaborn scipy optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - CatBoost Model")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# ENHANCED HYPERPARAMETER TUNING FOR CATBOOST
print("\n" + "="*80)
print("PHASE 0: ENHANCED HYPERPARAMETER TUNING - CatBoost")
print("="*80)

def objective_catboost(trial, X_train, y_train, target_name):
    """Enhanced objective function for CatBoost with expanded search space"""

    # Determine if this is a problematic target requiring more regularization
    problematic_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
    is_problematic = target_name in problematic_targets

    # Choose grow policy
    grow_policy = trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Lossguide', 'Depthwise'])

    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
        'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations', 1, 10),
        'grow_policy': grow_policy,
        'random_seed': 42,
        'verbose': False,
        'allow_writing_files': False
    }

    # Only add max_leaves if using Lossguide policy
    if grow_policy == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 15, 64)

    # For problematic targets, bias towards more regularization
    if is_problematic:
        params['subsample'] = trial.suggest_float('subsample', 0.5, 0.85)
        params['l2_leaf_reg'] = trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True)
        params['min_data_in_leaf'] = trial.suggest_int('min_data_in_leaf', 10, 30)

    model = CatBoostRegressor(**params)

    # Use CV to evaluate
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)

    return scores.mean()

# Prepare data for tuning
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

scaler_tune = StandardScaler()
X_train_scaled = scaler_tune.fit_transform(X_train_full)
X_test_scaled = scaler_tune.transform(X_test_full)

# Tune hyperparameters for each target with MORE TRIALS
optimal_params_cat = {}
tuning_trials = {
    'Youngs Modulus (MPa)': 100,  # Most problematic - more trials
    'Yield Strength (MPa)': 100,  # Problematic - more trials
    'Ultimate Compressive Strength (MPa)': 50,  # Good performance - fewer trials
    'Ultimate Strain': 50,  # Good performance - fewer trials
    'Toughness (MJ/m³)': 50  # Good performance - fewer trials
}

for target in target_cols:
    n_trials = tuning_trials[target]
    print(f"\nTuning: {target} (with {n_trials} trials)")

    sampler = TPESampler(seed=42)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: objective_catboost(trial, X_train_scaled, y_train_full[target].values, target),
        n_trials=n_trials,
        show_progress_bar=True
    )

    optimal_params_cat[target] = study.best_params
    print(f"  Best R2 (CV): {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")
    gc.collect()

# ENHANCED TRAINING FUNCTION for CatBoost with Early Stopping
def train_model_catboost(X, y, target_name, params, use_ensemble=False):
    """Enhanced training with early stopping and optional ensemble"""

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    if use_ensemble:
        # Ensemble approach: train multiple models with different seeds
        print(f"  Using ensemble (5 models)...")
        models = []
        train_preds_all = []
        test_preds_all = []

        for seed in [42, 123, 456, 789, 2024]:
            model_params = params.copy()
            model_params['random_seed'] = seed

            model = CatBoostRegressor(**model_params)

            # Create Pool objects for CatBoost (enables early stopping)
            train_pool = Pool(X_train_sc, y_train)
            test_pool = Pool(X_test_sc, y_test)

            # Fit with early stopping
            model.fit(
                train_pool,
                eval_set=test_pool,
                early_stopping_rounds=50,
                verbose=False
            )

            models.append(model)
            train_preds_all.append(model.predict(X_train_sc))
            test_preds_all.append(model.predict(X_test_sc))

        # Average predictions
        y_tr_pred = np.mean(train_preds_all, axis=0)
        y_te_pred = np.mean(test_preds_all, axis=0)

        # Use first model for other purposes (feature importance, etc.)
        model = models[0]

    else:
        # Single model with early stopping
        model = CatBoostRegressor(**params)

        # Create Pool objects
        train_pool = Pool(X_train_sc, y_train)
        test_pool = Pool(X_test_sc, y_test)

        model.fit(
            train_pool,
            eval_set=test_pool,
            early_stopping_rounds=50,
            verbose=False
        )

        y_tr_pred = model.predict(X_train_sc)
        y_te_pred = model.predict(X_test_sc)

    # CV scores
    cv_model = CatBoostRegressor(**params)
    cv = cross_val_score(cv_model, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"  CV R2: {cv.mean():.4f} ± {cv.std():.4f}")

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    # Calculate overfitting score
    overfit_score = tr_met['R2'] - te_met['R2']
    print(f"  Train R2: {tr_met['R2']:.4f} | Test R2: {te_met['R2']:.4f} | Overfit: {overfit_score:.4f}")

    if overfit_score > 0.10:
        print(f"  ⚠️  WARNING: Significant overfitting detected (gap > 0.10)")

    return {
        'model': model if not use_ensemble else models,
        'scaler': scaler,
        'train_metrics': tr_met,
        'test_metrics': te_met,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_pred': y_tr_pred,
        'y_test_pred': y_te_pred,
        'is_ensemble': use_ensemble
    }

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - CatBoost")
print("="*80)

# Decide which targets need ensemble
ensemble_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']

results_before = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_before[target] = train_model_catboost(
        X_all, y_all[target], target,
        optimal_params_cat[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - CatBoost")
print("="*80)

results_after = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_after[target] = train_model_catboost(
        X_clean, y_clean[target], target,
        optimal_params_cat[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Smart selection
# Select models: Young's Modulus, Yield Strength, Ultimate Compressive Strength AFTER outlier removal
# Ultimate Strain and Toughness BEFORE outlier removal
targets_after = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)', 'Ultimate Compressive Strength (MPa)']
targets_before = ['Ultimate Strain', 'Toughness (MJ/m³)']
results_viz = {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols}

# COMPREHENSIVE METRICS COMPARISON
print("\n" + "="*80)
print("METRICS COMPARISON - BEFORE vs AFTER OUTLIER REMOVAL")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement']
print("\n", comparison_df.round(4))

# PRINT DETAILED METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL (CatBoost)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL (CatBoost)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS - CatBoost")
print("="*80)

# 1. Actual vs Predicted - Combined
print("\n1. Actual vs Predicted - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')

    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('CatBoost', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_actual_vs_pred_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution - Combined
print("\n2. Error Distribution - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=15, alpha=0.7, edgecolor='black', color='coral', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=2.5, label=f'Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2, label='Zero')
    ax.set_xlabel('Error', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_error_dist_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2b. Error Distribution - Individual plots
print("\n2b. Error Distribution - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.hist(errors, bins=20, alpha=0.7, edgecolor='black', color='mediumpurple', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=3, label=f'Normal Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2.5, label='Zero Error')
    ax.set_xlabel('Prediction Error', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'CatBoost: {t}', fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'catboost_error_dist_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Error distribution plots saved")

# 3. Residuals - Combined
print("\n3. Residuals - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], resid, alpha=0.6, s=50)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)

    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {np.mean(resid):.2f}, Std: {std_resid:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

if len(target_cols) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_residual_analysis_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3b. Residuals - Individual plots
print("\n3b. Residuals - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], resid, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Residual')

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8, label=f'±2σ ({2*std_resid:.2f})')
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Residuals', fontsize=12, fontweight='bold')
    ax.set_title(f'CatBoost: {t}',
                 fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'catboost_residual_analysis_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Residual analysis plots saved")

# 4. Percentage Error Analysis - Individual plots ONLY
print("\n4. Percentage Error Analysis - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    percent_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], percent_errors, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')

    # Add horizontal lines at ±20%
    ax.axhline(y=20, color='orange', linestyle=':', linewidth=2, alpha=0.8, label='±20%')
    ax.axhline(y=-20, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    mean_pe = np.mean(percent_errors)
    std_pe = np.std(percent_errors)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Percentage Error (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'CatBoost: {t}',
                 fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'catboost_percentage_error_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Percentage error analysis plots saved")

# 5. Radar Chart - Combined (Test Set Only with 5 metrics)
print("\n5. Radar Charts - Combined (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            # Lower error = better, so invert: 1 - (error/100)
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='orange')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='orange')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_radar_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5b. Radar Charts - Individual plots
print("\n5b. Radar Charts - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    metric_values_text = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        metric_values_text.append(f"{val:.3f}")
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=3, color='cadetblue', markersize=10)
    ax.fill(angles_plot, test_vals_plot, alpha=0.3, color='cadetblue')
    ax.set_xticks(angles)
    ax.set_xticklabels([f'{label}\n({val})' for label, val in zip(metrics_labels, metric_values_text)],
                       fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
    ax.set_title(f'CatBoost: {t}', fontsize=12, fontweight='bold', pad=25)
    ax.grid(True, linewidth=1.2, alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'catboost_radar_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Radar chart plots saved")

# 6. Feature Importance (CatBoost Native)
print("\n6. CatBoost Feature Importance")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(input_cols + feature_cols)))

feature_importance_dict = {}
for idx, t in enumerate(target_cols):
    r = results_viz[t]

    # Handle ensemble models
    if r.get('is_ensemble', False):
        model_to_use = r['model'][0]  # Use first model from ensemble
    else:
        model_to_use = r['model']

    # Get feature importance from CatBoost
    importance = model_to_use.get_feature_importance()
    feature_importance_dict[t] = importance

    ax = axes[idx]
    sidx = np.argsort(importance)[::-1]
    bar_colors = [colors[i] for i in sidx]

    all_features = input_cols + feature_cols
    ax.barh(range(len(sidx)), importance[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([all_features[i] for i in sidx], fontsize=8)
    ax.set_xlabel('Feature Importance', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('CatBoost Feature Importance Rankings', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 7. Feature Importance Heatmap
print("7. Feature Importance Heatmap")
importance_df = pd.DataFrame(feature_importance_dict, index=input_cols + feature_cols)

# Normalize for heatmap
importance_df_norm = importance_df.copy()
for col in importance_df_norm.columns:
    max_val = importance_df_norm[col].max()
    if max_val > 0:
        importance_df_norm[col] = importance_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(importance_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized Feature Importance'})
ax.set_title('CatBoost Feature Importance (Normalized)\n(All Features)',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('catboost_importance_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Analysis for CatBoost
print("\n8. SHAP Analysis - CatBoost")
shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all

    # Handle ensemble models
    if r.get('is_ensemble', False):
        model_to_use = r['model'][0]  # Use first model from ensemble
    else:
        model_to_use = r['model']

    X_scaled = r['scaler'].transform(X_data)
    n_samples = min(15, len(X_scaled))
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    try:
        # CatBoost has native SHAP support
        exp = shap.TreeExplainer(model_to_use)
        sv = exp.shap_values(X_sample)

        if isinstance(sv, list):
            sv = sv[0]
        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del exp
    except Exception as e:
        print(f"    Warning: TreeExplainer failed for {t}, using KernelExplainer")
        def model_predict(X):
            return model_to_use.predict(X)

        background = shap.sample(X_sample, min(5, len(X_sample)))
        exp = shap.KernelExplainer(model_predict, background)
        sv = exp.shap_values(X_sample)

        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del exp

    gc.collect()

# 9. SHAP Heatmap (Engineered Features Only)
print("\n9. SHAP Heatmap (Normalized) - Engineered Features")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='viridis', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('CatBoost',
            fontsize=12, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('catboost_shap_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 10. SHAP Rankings - Combined
print("\n10. SHAP Rankings (Color Palette) - Combined")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Ranking', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('catboost_shap_rankings_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 10b. SHAP Rankings - Individual plots for each target
print("\n10b. SHAP Rankings - Individual Plots")

# Create color palette
colors_shap = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for t in target_cols:
    print(f"  {t}...")
    fig, ax = plt.subplots(figsize=(10, 8))

    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]

    # Use color palette
    bar_colors = [colors_shap[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.8)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'CatBoost: {t}', fontsize=12, fontweight='bold', pad=15)

    # Add value labels on bars
    for i, (idx, val) in enumerate(zip(sidx, imp[sidx])):
        ax.text(val, i, f' {val:.4f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'catboost_shap_ranking_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP ranking plots saved")

# 11. SHAP Summary Plots
print("\n11. SHAP Summary Plots")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'CatBoost: {t}', fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'catboost_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

# 12. SHAP Dependence Plots (top 3 features for each target)
print("\n12. SHAP Dependence Plots (Top 3 Features)")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    # Get the sample data again for plotting based on selection
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]

    n_samples = min(15, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    # Get top 3 features by SHAP importance
    imp = shap_imp[t]
    top3_idx = np.argsort(imp)[::-1][:3]
    top3_features = [feature_cols[i] for i in top3_idx]

    # Create 1x3 subplot for top 3 features
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for plot_idx, feat_idx in enumerate(top3_idx):
        ax = axes[plot_idx]

        # SHAP dependence plot
        shap.dependence_plot(
            feat_idx,
            shap_values_all[t],
            X_sample_feat,
            feature_names=feature_cols,
            show=False,
            ax=ax
        )
        ax.set_title(f'{feature_cols[feat_idx]}', fontsize=12, fontweight='bold')
        ax.set_xlabel(f'{feature_cols[feat_idx]} (scaled)', fontsize=10, fontweight='bold')
        ax.set_ylabel('SHAP value', fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'CatBoost: {t}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'catboost_shap_dependence_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP dependence plots saved")

# 13. Training History (Loss curves) for one target
print("\n13. Training History (Loss Curve)")
sample_target = 'Ultimate Compressive Strength (MPa)'
print(f"  Plotting for: {sample_target}")

X_train_sample, X_test_sample, y_train_sample, y_test_sample = train_test_split(
    X_all, y_all[sample_target], test_size=0.2, random_state=42
)
scaler_sample = StandardScaler()
X_train_sample_sc = scaler_sample.fit_transform(X_train_sample)
X_test_sample_sc = scaler_sample.transform(X_test_sample)

train_pool_sample = Pool(X_train_sample_sc, y_train_sample)
test_pool_sample = Pool(X_test_sample_sc, y_test_sample)

model_history = CatBoostRegressor(**optimal_params_cat[sample_target])
model_history.fit(
    train_pool_sample,
    eval_set=test_pool_sample,
    verbose=False
)

# Get training history
evals_result = model_history.get_evals_result()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(evals_result['learn']['RMSE'], label='Train RMSE', linewidth=2)
ax.plot(evals_result['validation']['RMSE'], label='Test RMSE', linewidth=2)
ax.set_xlabel('Iterations', fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax.set_title(f'CatBoost Training History: {sample_target}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('catboost_training_history.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# FINAL SUMMARY
print("\n" + "="*80)
print("FINAL SUMMARY - CatBoost Model")
print("="*80)

print("\nHYPERPARAMETER TUNING SUMMARY:")
for t in target_cols:
    print(f"\n{t}:")
    print(f"  Trials: {tuning_trials[t]}")
    print(f"  Best params: {optimal_params_cat[t]}")

print("\n" + "="*80)
print("PERFORMANCE IMPROVEMENTS:")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("OVERFITTING ANALYSIS:")
print("="*80)
for t in target_cols:
    before_gap = results_before[t]['train_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    after_gap = results_after[t]['train_metrics']['R2'] - results_after[t]['test_metrics']['R2']

    print(f"\n{t}:")
    print(f"  Before removal: Train-Test gap = {before_gap:.4f}")
    print(f"  After removal:  Train-Test gap = {after_gap:.4f}")

    if after_gap < 0.05:
        print(f"  ✓ Excellent generalization")
    elif after_gap < 0.10:
        print(f"  ✓ Good generalization")
    elif after_gap < 0.15:
        print(f"  ⚠ Moderate overfitting")
    else:
        print(f"  ❌ Significant overfitting - consider more regularization")

print("\n" + "="*80)
print("✓ COMPLETE - CatBoost Model!")
print("="*80)

print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nModel Selection Summary:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")

print("\nSelected Model Performance:")
for t in target_cols:
    selection = 'AFTER' if t in targets_after else 'BEFORE'
    r2_score = results_viz[t]['test_metrics']['R2']
    print(f"  {t} ({selection}): R²={r2_score:.4f}")



print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save individual models based on selection criteria
saved_files = []

print("\nSelected models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")
print()

for target in target_cols:
    target_clean = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

    # Determine which result set to use
    if target in targets_after:
        result_source = results_after[target]
        selection_type = 'after'
    else:
        result_source = results_before[target]
        selection_type = 'before'

    # Create a comprehensive save package
    model_package = {
        'target_name': target,
        'selection_type': selection_type,
        'model': result_source['model'],
        'scaler': result_source['scaler'],
        'params': optimal_params_cat[target],
        'train_metrics': result_source['train_metrics'],
        'test_metrics': result_source['test_metrics'],
        'is_ensemble': result_source['is_ensemble'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp,
        'outliers_removed': outliers if selection_type == 'after' else []
    }

    # Save as pickle
    filename_pkl = f'catboost_model_{target_clean}_{selection_type}_outliers_{timestamp}.pkl'
    with open(filename_pkl, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename_pkl)
    print(f"  ✓ {target} ({selection_type.upper()}): R²={result_source['test_metrics']['R2']:.4f} → {filename_pkl}")

# Save all models together
all_models_package = {
    'models': {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols},
    'selection_criteria': {
        'after_outlier_removal': targets_after,
        'before_outlier_removal': targets_before
    },
    'optimal_params': optimal_params_cat,
    'input_cols': input_cols,
    'feature_cols': feature_cols,
    'target_cols': target_cols,
    'outliers_removed': outliers,
    'timestamp': timestamp,
    'X_clean': X_clean,
    'y_clean': y_clean,
    'X_all': X_all,
    'y_all': y_all
}

all_models_filename = f'catboost_all_selected_models_{timestamp}.pkl'
with open(all_models_filename, 'wb') as f:
    pickle.dump(all_models_package, f)
saved_files.append(all_models_filename)
print(f"\n✓ Saved all models package: {all_models_filename}")

# Create a summary document
summary_filename = f'catboost_model_summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("CATBOOST MODEL SUMMARY - SELECTIVE MODEL SELECTION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Outliers removed: {len(outliers)} samples\n")
    f.write(f"Outlier indices: {sorted(outliers)}\n\n")

    f.write("MODEL SELECTION CRITERIA:\n")
    f.write("-"*80 + "\n")
    f.write(f"AFTER outlier removal:\n")
    for t in targets_after:
        f.write(f"  • {t}\n")
    f.write(f"\nBEFORE outlier removal:\n")
    for t in targets_before:
        f.write(f"  • {t}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("MODEL DETAILS:\n")
    f.write("-"*80 + "\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        r2_score = result_source['test_metrics']['R2']
        ensemble_status = "Ensemble (5 models)" if result_source['is_ensemble'] else "Single model"
        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write(f"  Model type: {ensemble_status}\n")
        f.write(f"  Test R²: {r2_score:.4f}\n")
        f.write(f"  Parameters: {optimal_params_cat[target]}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED METRICS:\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write("  Train Metrics:\n")
        for k, v in result_source['train_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")
        f.write("  Test Metrics:\n")
        for k, v in result_source['test_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("HOW TO LOAD AND USE THE MODELS:\n")
    f.write("="*80 + "\n\n")
    f.write("# Load a single model:\n")
    f.write("import pickle\n")
    f.write("import numpy as np\n\n")
    f.write("with open('catboost_model_[target]_[after/before]_outliers_[timestamp].pkl', 'rb') as f:\n")
    f.write("    model_package = pickle.load(f)\n\n")
    f.write("model = model_package['model']\n")
    f.write("scaler = model_package['scaler']\n")
    f.write("selection_type = model_package['selection_type']  # 'after' or 'before'\n\n")
    f.write("# For prediction:\n")
    f.write("# 1. Prepare your input data (same features as training)\n")
    f.write("# 2. Scale the data\n")
    f.write("X_scaled = scaler.transform(X_new)\n\n")
    f.write("# 3. Make predictions\n")
    f.write("if model_package['is_ensemble']:\n")
    f.write("    # For ensemble models, average predictions from all models\n")
    f.write("    predictions = np.mean([m.predict(X_scaled) for m in model], axis=0)\n")
    f.write("else:\n")
    f.write("    predictions = model.predict(X_scaled)\n\n")
    f.write("# Load all models at once:\n")
    f.write("with open('catboost_all_selected_models_[timestamp].pkl', 'rb') as f:\n")
    f.write("    all_models = pickle.load(f)\n\n")
    f.write("# Check selection criteria\n")
    f.write("print('After outlier removal:', all_models['selection_criteria']['after_outlier_removal'])\n")
    f.write("print('Before outlier removal:', all_models['selection_criteria']['before_outlier_removal'])\n\n")
    f.write("# Access specific model\n")
    f.write("youngs_modulus_model = all_models['models']['Youngs Modulus (MPa)']['model']\n")
    f.write("youngs_modulus_scaler = all_models['models']['Youngs Modulus (MPa)']['scaler']\n")

saved_files.append(summary_filename)
print(f"Saved summary: {summary_filename}")

# DOWNLOAD FILES
print("\n" + "="*80)
print("DOWNLOADING FILES")
print("="*80)

for filename in saved_files:
    print(f"  Downloading: {filename}")
    files.download(filename)

print("\n" + "="*80)
print("MODEL SAVING COMPLETE!")
print("="*80)
print(f"\nTotal files saved and downloaded: {len(saved_files)}")
print("\nFiles include:")
print(f"  • {len(target_cols)} individual model files (.pkl)")
print(f"  • 1 combined models file (.pkl)")
print(f"  • 1 summary file (.txt)")

print("\n" + "="*80)
print("SELECTION SUMMARY:")
print("="*80)
print("\nModels selected AFTER outlier removal:")
for target in targets_after:
    print(f"  • {target}: R²={results_after[target]['test_metrics']['R2']:.4f}")
print("\nModels selected BEFORE outlier removal:")
for target in targets_before:
    print(f"  • {target}: R²={results_before[target]['test_metrics']['R2']:.4f}")

print("\n" + "="*80)
print("✓ ALL DONE! Models saved and downloaded.")
print("="*80)

**Stacking Ensemble**

In [ ]:
# Stacking Ensemble Model for Predicting Mechanical Properties
# - XGBoost + LightGBM + CatBoost with Meta-Learner


!pip install -q xgboost lightgbm catboost shap openpyxl scikit-learn pandas numpy matplotlib seaborn scipy optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import os
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['CATBOOST_SILENT'] = '1'

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - Stacking Ensemble Model")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# HYPERPARAMETER TUNING FOR BASE MODELS
print("\n" + "="*80)
print("PHASE 0: HYPERPARAMETER TUNING - Stacking Ensemble")
print("="*80)

# Prepare data for tuning
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

scaler_tune = StandardScaler()
X_train_scaled = scaler_tune.fit_transform(X_train_full)
X_test_scaled = scaler_tune.transform(X_test_full)

# Define tuning trials
tuning_trials = {
    'Youngs Modulus (MPa)': 50,
    'Yield Strength (MPa)': 50,
    'Ultimate Compressive Strength (MPa)': 30,
    'Ultimate Strain': 30,
    'Toughness (MJ/m³)': 30
}

# Store optimal parameters
optimal_params_xgb = {}
optimal_params_lgb = {}
optimal_params_cat = {}
optimal_params_meta = {}

print("\n" + "-"*80)
print("Tuning XGBoost Base Models...")
print("-"*80)

def objective_xgb(trial, X_train, y_train):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42,
        'n_jobs': 1
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)
    return scores.mean()

for target in target_cols:
    print(f"  {target}...", end=" ")
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(
        lambda trial: objective_xgb(trial, X_train_scaled, y_train_full[target].values),
        n_trials=tuning_trials[target],
        show_progress_bar=False
    )
    optimal_params_xgb[target] = study.best_params
    print(f"✓ R²={study.best_value:.4f}")
    gc.collect()

print("\n" + "-"*80)
print("Tuning LightGBM Base Models...")
print("-"*80)

def objective_lgb(trial, X_train, y_train):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 15, 80),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'random_state': 42,
        'verbose': -1,
        'n_jobs': 1
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)
    return scores.mean()

for target in target_cols:
    print(f"  {target}...", end=" ")
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(
        lambda trial: objective_lgb(trial, X_train_scaled, y_train_full[target].values),
        n_trials=tuning_trials[target],
        show_progress_bar=False
    )
    optimal_params_lgb[target] = study.best_params
    print(f"✓ R²={study.best_value:.4f}")
    gc.collect()

print("\n" + "-"*80)
print("Tuning CatBoost Base Models...")
print("-"*80)

def objective_cat(trial, X_train, y_train):
    grow_policy = trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Lossguide', 'Depthwise'])
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500, step=50),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'grow_policy': grow_policy,
        'random_seed': 42,
        'logging_level': 'Silent'
    }
    if grow_policy == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 15, 64)

    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)
    return scores.mean()

for target in target_cols:
    print(f"  {target}...", end=" ")
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(
        lambda trial: objective_cat(trial, X_train_scaled, y_train_full[target].values),
        n_trials=tuning_trials[target],
        show_progress_bar=False
    )
    optimal_params_cat[target] = study.best_params
    print(f"✓ R²={study.best_value:.4f}")
    gc.collect()

print("\n" + "-"*80)
print("Tuning Meta-Learner...")
print("-"*80)

def objective_meta(trial, X_train, y_train, target_name):
    """Tune meta-learner (Ridge, Lasso, or ElasticNet)"""
    meta_type = trial.suggest_categorical('meta_type', ['Ridge', 'Lasso', 'ElasticNet'])
    alpha = trial.suggest_float('alpha', 0.001, 100.0, log=True)

    # Create base models
    xgb_model = XGBRegressor(**optimal_params_xgb[target_name])
    lgb_model = LGBMRegressor(**optimal_params_lgb[target_name])
    cat_model = CatBoostRegressor(**optimal_params_cat[target_name])

    # Create meta-learner
    if meta_type == 'Ridge':
        meta_learner = Ridge(alpha=alpha, random_state=42)
    elif meta_type == 'Lasso':
        meta_learner = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    else:  # ElasticNet
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        meta_learner = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=5000)

    # Create stacking ensemble
    estimators = [
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('cat', cat_model)
    ]

    stacking = StackingRegressor(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=3,
        n_jobs=1
    )

    scores = cross_val_score(stacking, X_train, y_train, cv=5, scoring='r2', n_jobs=1)
    return scores.mean()

for target in target_cols:
    print(f"  {target}...", end=" ")
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(
        lambda trial: objective_meta(trial, X_train_scaled, y_train_full[target].values, target),
        n_trials=20,  # Fewer trials for meta-learner
        show_progress_bar=False
    )
    optimal_params_meta[target] = study.best_params
    print(f"✓ R²={study.best_value:.4f}")
    gc.collect()

# TRAINING FUNCTION for Stacking Ensemble
def train_stacking_ensemble(X, y, target_name, xgb_params, lgb_params, cat_params, meta_params):
    """Train stacking ensemble with optimized parameters"""

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Create base models
    xgb_model = XGBRegressor(**xgb_params)
    lgb_model = LGBMRegressor(**lgb_params)
    cat_model = CatBoostRegressor(**cat_params)

    # Create meta-learner
    meta_type = meta_params['meta_type']
    alpha = meta_params['alpha']

    if meta_type == 'Ridge':
        meta_learner = Ridge(alpha=alpha, random_state=42)
    elif meta_type == 'Lasso':
        meta_learner = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    else:  # ElasticNet
        l1_ratio = meta_params.get('l1_ratio', 0.5)
        meta_learner = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=5000)

    # Create stacking ensemble
    estimators = [
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('cat', cat_model)
    ]

    stacking = StackingRegressor(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=3,
        n_jobs=1
    )

    # Fit the stacking model
    print(f"  Training base models and meta-learner...", end=" ")
    stacking.fit(X_train_sc, y_train)
    print("✓")

    # Get predictions
    y_tr_pred = stacking.predict(X_train_sc)
    y_te_pred = stacking.predict(X_test_sc)

    # Get individual model predictions for analysis
    base_train_preds = {}
    base_test_preds = {}
    for name, model in stacking.named_estimators_.items():
        base_train_preds[name] = model.predict(X_train_sc)
        base_test_preds[name] = model.predict(X_test_sc)

    # CV scores
    print(f"  Computing CV scores...", end=" ")
    cv = cross_val_score(stacking, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"✓ CV R²: {cv.mean():.4f} ± {cv.std():.4f}")

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    # Calculate overfitting score
    overfit_score = tr_met['R2'] - te_met['R2']
    print(f"  Train R²: {tr_met['R2']:.4f} | Test R²: {te_met['R2']:.4f} | Overfit: {overfit_score:.4f}")

    if overfit_score > 0.10:
        print(f"  ⚠️  WARNING: Significant overfitting detected (gap > 0.10)")

    return {
        'model': stacking,
        'scaler': scaler,
        'train_metrics': tr_met,
        'test_metrics': te_met,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_pred': y_tr_pred,
        'y_test_pred': y_te_pred,
        'base_train_preds': base_train_preds,
        'base_test_preds': base_test_preds,
        'meta_type': meta_type
    }

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - Stacking Ensemble")
print("="*80)

results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    results_before[target] = train_stacking_ensemble(
        X_all, y_all[target], target,
        optimal_params_xgb[target],
        optimal_params_lgb[target],
        optimal_params_cat[target],
        optimal_params_meta[target]
    )
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - Stacking Ensemble")
print("="*80)

results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    results_after[target] = train_stacking_ensemble(
        X_clean, y_clean[target], target,
        optimal_params_xgb[target],
        optimal_params_lgb[target],
        optimal_params_cat[target],
        optimal_params_meta[target]
    )
    gc.collect()

# Smart selection
# Young's Modulus and Yield Strength AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness BEFORE outlier removal
targets_after = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
targets_before = ['Ultimate Compressive Strength (MPa)', 'Ultimate Strain', 'Toughness (MJ/m³)']
results_viz = {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols}

# COMPREHENSIVE METRICS COMPARISON
print("\n" + "="*80)
print("METRICS COMPARISON - BEFORE vs AFTER OUTLIER REMOVAL")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement']
print("\n", comparison_df.round(4))

# PRINT DETAILED METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL (Stacking)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL (Stacking)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS - Stacking Ensemble")
print("="*80)

# 1. Actual vs Predicted
print("\n1. Actual vs Predicted")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    meta_tag = f" ({r['meta_type']})"
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('Stacking Ensemble', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution
print("2. Error Distribution")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax.axvline(x=mean_error, color='green', linestyle='-', linewidth=2,
               label=f'Mean: {mean_error:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}\nError Distribution', fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('stacking_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3. Residuals
print("3. Residuals")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=60, color='darkorange', edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    ax.axhline(residuals.mean(), color='blue', linestyle=':', linewidth=2,
              label=f'Mean: {residuals.mean():.2f}')
    std_resid = np.std(residuals)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {residuals.mean():.2f}, Std: {std_resid:.2f}',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('stacking_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 4. Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    r = results_viz[t]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    ax.set_title(f'{t}\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                 fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('stacking_percentage_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5. Radar Chart
print("5. Radar Charts (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)
    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]
    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='darkgreen')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='darkgreen')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts - Stacking Ensemble (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 6. Base Model Comparison
print("6. Base Model Performance Comparison")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]

    # Calculate R2 for each base model
    base_r2 = {}
    for name, preds in r['base_test_preds'].items():
        base_r2[name] = r2_score(r['y_test'], preds)

    # Add stacking R2
    base_r2['Stacking'] = r['test_metrics']['R2']

    # Plot
    models = list(base_r2.keys())
    r2_values = list(base_r2.values())
    colors_map = {'xgb': 'skyblue', 'lgb': 'lightgreen', 'cat': 'plum', 'Stacking': 'gold'}
    bar_colors = [colors_map.get(m, 'gray') for m in models]

    bars = ax.bar(models, r2_values, color=bar_colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.set_ylabel('R² Score', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')
    ax.set_ylim([min(r2_values) - 0.1, 1.0])
    ax.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, val in zip(bars, r2_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Rename x-axis labels
    ax.set_xticklabels(['XGBoost', 'LightGBM', 'CatBoost', 'Stacking'], rotation=45, ha='right')

axes[-1].axis('off')
plt.suptitle('Base Models vs Stacking Ensemble - R² Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 7. Ensemble Contribution Analysis
print("7. Ensemble Contribution Analysis")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]

    # Calculate improvement from each base model
    base_r2 = {}
    for name, preds in r['base_test_preds'].items():
        base_r2[name] = r2_score(r['y_test'], preds)

    stacking_r2 = r['test_metrics']['R2']
    improvements = {name: stacking_r2 - r2 for name, r2 in base_r2.items()}

    models = list(improvements.keys())
    model_names = ['XGBoost', 'LightGBM', 'CatBoost']
    improvement_vals = list(improvements.values())

    colors = ['crimson' if v < 0 else 'forestgreen' for v in improvement_vals]
    bars = ax.barh(model_names, improvement_vals, color=colors, edgecolor='black', linewidth=1.5, alpha=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=2)
    ax.set_xlabel('R² Improvement (Stacking - Base)', fontsize=10, fontweight='bold')
    ax.set_title(f'{t}\nStacking R²: {stacking_r2:.4f}', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

    # Add value labels
    for bar, val in zip(bars, improvement_vals):
        width = bar.get_width()
        label_x_pos = width + 0.001 if width > 0 else width - 0.001
        ax.text(label_x_pos, bar.get_y() + bar.get_height()/2.,
                f'{val:+.4f}',
                ha='left' if width > 0 else 'right', va='center', fontsize=9, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('Stacking Improvement Over Individual Base Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_contribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 8. Base Model Agreement Analysis
print("8. Base Model Agreement (Correlation)")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    # Create dataframe with predictions
    pred_df = pd.DataFrame({
        'Actual': r['y_test'],
        'XGBoost': r['base_test_preds']['xgb'],
        'LightGBM': r['base_test_preds']['lgb'],
        'CatBoost': r['base_test_preds']['cat'],
        'Stacking': r['y_test_pred']
    })

    # Calculate correlation
    corr = pred_df.corr()

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                linewidths=1, ax=ax, vmin=-1, vmax=1,
                cbar_kws={'label': 'Correlation'})
    ax.set_title(f'Prediction Correlation Matrix: {t}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'stacking_correlation_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Correlation matrices saved")

# 9. SHAP Analysis for Stacking Ensemble
print("\n9. SHAP Analysis - Stacking Ensemble")

shap_imp_base = {'xgb': {}, 'lgb': {}, 'cat': {}}
shap_values_all = {}

for t in target_cols:
    print(f"\n  Analyzing {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)

    # Sample for SHAP (computational efficiency)
    n_samples = min(50, len(X_scaled))
    idx_sample = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx_sample]

    # SHAP for each base model
    for model_name in ['xgb', 'lgb', 'cat']:
        print(f"    - {model_name.upper()}...", end=" ")
        base_model = r['model'].named_estimators_[model_name]

        try:
            explainer = shap.TreeExplainer(base_model)
            shap_vals = explainer.shap_values(X_sample)

            if isinstance(shap_vals, list):
                shap_vals = shap_vals[0]
            if len(shap_vals.shape) == 1:
                shap_vals = shap_vals.reshape(-1, 1)

            # Store mean absolute SHAP values (only engineered features)
            shap_imp_base[model_name][t] = np.abs(shap_vals).mean(axis=0)[len(input_cols):]

            if model_name == 'xgb':  # Store for detailed plots
                shap_values_all[t] = shap_vals[:, len(input_cols):]

            print("✓")
        except Exception as e:
            print(f"⚠ (using alternative method)")
            # Fallback to feature importance
            if hasattr(base_model, 'feature_importances_'):
                importance = base_model.feature_importances_[len(input_cols):]
                shap_imp_base[model_name][t] = importance
            else:
                shap_imp_base[model_name][t] = np.zeros(len(feature_cols))

        gc.collect()

# 10. SHAP Heatmap Comparison (All Base Models)
print("\n10. SHAP Heatmap - Base Models Comparison")

fig, axes = plt.subplots(1, 3, figsize=(22, 10))

for idx, model_name in enumerate(['xgb', 'lgb', 'cat']):
    ax = axes[idx]

    # Create SHAP dataframe for this model
    shap_df = pd.DataFrame(shap_imp_base[model_name], index=feature_cols)

    # Normalize
    shap_df_norm = shap_df.copy()
    for col in shap_df_norm.columns:
        max_val = shap_df_norm[col].max()
        if max_val > 0:
            shap_df_norm[col] = shap_df_norm[col] / max_val

    sns.heatmap(shap_df_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalized SHAP'})

    model_names_full = {'xgb': 'XGBoost', 'lgb': 'LightGBM', 'cat': 'CatBoost'}
    ax.set_title(f'{model_names_full[model_name]} - SHAP Importance',
                fontsize=13, fontweight='bold')
    ax.set_xlabel('Targets', fontsize=11, fontweight='bold')
    ax.set_ylabel('Features' if idx == 0 else '', fontsize=11, fontweight='bold')

    if idx > 0:
        ax.set_ylabel('')

    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    plt.setp(ax.get_yticklabels(), fontsize=9)

plt.suptitle('SHAP Feature Importance Comparison - Base Models',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_shap_heatmap_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 11. SHAP Summary Plots (XGBoost base model)
print("\n11. SHAP Summary Plots (XGBoost Base Models)")

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal, original data for others
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]

    n_samples = min(50, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    if t in shap_values_all:
        fig, ax = plt.subplots(figsize=(10, 8))
        shap.summary_plot(
            shap_values_all[t],
            X_sample_feat,
            feature_names=feature_cols,
            show=False,
            max_display=12
        )
        plt.title(f'SHAP Summary - XGBoost Base Model: {t}',
                 fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(f'stacking_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=120, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ SHAP summary plots saved")

# 12. SHAP Feature Rankings by Model
print("\n12. SHAP Feature Rankings by Base Model")

fig, axes = plt.subplots(5, 3, figsize=(20, 24))

for target_idx, t in enumerate(target_cols):
    for model_idx, model_name in enumerate(['xgb', 'lgb', 'cat']):
        ax = axes[target_idx, model_idx]

        imp = shap_imp_base[model_name][t]
        sidx = np.argsort(imp)[::-1]

        colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))
        bar_colors = [colors[i] for i in sidx]

        ax.barh(range(len(sidx)), imp[sidx], color=bar_colors,
               edgecolor='black', linewidth=0.5, alpha=0.8)
        ax.set_yticks(range(len(sidx)))
        ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=8)
        ax.set_xlabel('Mean |SHAP Value|', fontsize=9, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')

        model_names_full = {'xgb': 'XGBoost', 'lgb': 'LightGBM', 'cat': 'CatBoost'}
        if target_idx == 0:
            ax.set_title(f'{model_names_full[model_name]}', fontsize=11, fontweight='bold')

        if model_idx == 0:
            ax.text(-0.35, 0.5, t, transform=ax.transAxes,
                   fontsize=10, fontweight='bold', rotation=90,
                   verticalalignment='center')

plt.suptitle('SHAP Feature Importance Rankings - All Base Models',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_shap_rankings_all_models.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 13. Feature Importance Consensus Analysis
print("\n13. Feature Importance Consensus Analysis")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, t in enumerate(target_cols):
    ax = axes[idx]

    # Get SHAP importance from all three models
    xgb_imp = shap_imp_base['xgb'][t]
    lgb_imp = shap_imp_base['lgb'][t]
    cat_imp = shap_imp_base['cat'][t]

    # Normalize each
    xgb_norm = xgb_imp / (xgb_imp.max() + 1e-10)
    lgb_norm = lgb_imp / (lgb_imp.max() + 1e-10)
    cat_norm = cat_imp / (cat_imp.max() + 1e-10)

    # Average importance across models (consensus)
    consensus_imp = (xgb_norm + lgb_norm + cat_norm) / 3

    # Calculate agreement (std deviation)
    stacked_imp = np.vstack([xgb_norm, lgb_norm, cat_norm])
    agreement = 1 - np.std(stacked_imp, axis=0)  # High value = high agreement

    # Sort by consensus importance
    sidx = np.argsort(consensus_imp)[::-1]

    # Create bars with color based on agreement
    x_pos = np.arange(len(feature_cols))
    bars = ax.barh(x_pos, consensus_imp[sidx],
                   color=plt.cm.RdYlGn(agreement[sidx]),
                   edgecolor='black', linewidth=0.5)

    ax.set_yticks(x_pos)
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Consensus Importance', fontsize=10, fontweight='bold')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')

    # Add colorbar for agreement
    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn,
                               norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Model Agreement', fontsize=9)

axes[-1].axis('off')
plt.suptitle('Feature Importance Consensus Across Base Models\n(Color = Agreement Level)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stacking_feature_consensus.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

print("✓ SHAP analysis complete")

print("\n✓ All Stacking Ensemble plots generated")

# FINAL SUMMARY
print("\n" + "="*80)
print("FINAL SUMMARY - Stacking Ensemble Model")
print("="*80)

print("\nENSEMBLE ARCHITECTURE:")
print("  Base Models: XGBoost + LightGBM + CatBoost")
print("  Meta-Learners by Target:")
for t in target_cols:
    meta_info = optimal_params_meta[t]
    print(f"    - {t}: {meta_info['meta_type']}")

print("\n" + "="*80)
print("PERFORMANCE IMPROVEMENTS:")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("OVERFITTING ANALYSIS:")
print("="*80)
for t in target_cols:
    before_gap = results_before[t]['train_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    after_gap = results_after[t]['train_metrics']['R2'] - results_after[t]['test_metrics']['R2']

    print(f"\n{t}:")
    print(f"  Before removal: Train-Test gap = {before_gap:.4f}")
    print(f"  After removal:  Train-Test gap = {after_gap:.4f}")

    if after_gap < 0.05:
        print(f"Excellent generalization")
    elif after_gap < 0.10:
        print(f"Good generalization")
    elif after_gap < 0.15:
        print(f"Moderate overfitting")
    else:
        print(f"Significant overfitting")

print("\n" + "="*80)
print("BASE MODEL vs STACKING COMPARISON:")
print("="*80)
for t in target_cols:
    r = results_viz[t]
    print(f"\n{t}:")
    for name, preds in r['base_test_preds'].items():
        base_r2 = r2_score(r['y_test'], preds)
        improvement = r['test_metrics']['R2'] - base_r2
        model_names = {'xgb': 'XGBoost', 'lgb': 'LightGBM', 'cat': 'CatBoost'}
        print(f"  {model_names[name]}: R²={base_r2:.4f} (Δ={improvement:+.4f})")
    print(f"  Stacking:  R²={r['test_metrics']['R2']:.4f}")

print("\n" + "="*80)
print("MODEL SELECTION SUMMARY:")
print("="*80)
print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nModel Selection:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")

print("\nSelected Model Performance:")
for t in target_cols:
    selection = 'AFTER' if t in targets_after else 'BEFORE'
    r2_score_val = results_viz[t]['test_metrics']['R2']
    print(f"  {t} ({selection}): R²={r2_score_val:.4f}")

# ============================================================================
# NEW SECTION: SAVE AND DOWNLOAD SELECTED MODELS
# Young's Modulus and Yield Strength: AFTER outlier removal
# Ultimate Compressive Strength, Ultimate Strain, Toughness: BEFORE outlier removal
# ============================================================================

print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save individual models based on selection criteria
saved_files = []

print("\nSelected models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")
print()

for target in target_cols:
    target_clean = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

    # Determine which result set to use
    if target in targets_after:
        result_source = results_after[target]
        selection_type = 'after'
    else:
        result_source = results_before[target]
        selection_type = 'before'

    # Create a comprehensive save package
    model_package = {
        'target_name': target,
        'selection_type': selection_type,
        'model': result_source['model'],
        'scaler': result_source['scaler'],
        'xgb_params': optimal_params_xgb[target],
        'lgb_params': optimal_params_lgb[target],
        'cat_params': optimal_params_cat[target],
        'meta_params': optimal_params_meta[target],
        'train_metrics': result_source['train_metrics'],
        'test_metrics': result_source['test_metrics'],
        'meta_type': result_source['meta_type'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp,
        'outliers_removed': outliers if selection_type == 'after' else []
    }

    # Save as pickle
    filename_pkl = f'stacking_model_{target_clean}_{selection_type}_outliers_{timestamp}.pkl'
    with open(filename_pkl, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename_pkl)
    print(f"  ✓ {target} ({selection_type.upper()}): R²={result_source['test_metrics']['R2']:.4f} → {filename_pkl}")

# Save all models together
all_models_package = {
    'models': {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols},
    'selection_criteria': {
        'after_outlier_removal': targets_after,
        'before_outlier_removal': targets_before
    },
    'optimal_params_xgb': optimal_params_xgb,
    'optimal_params_lgb': optimal_params_lgb,
    'optimal_params_cat': optimal_params_cat,
    'optimal_params_meta': optimal_params_meta,
    'input_cols': input_cols,
    'feature_cols': feature_cols,
    'target_cols': target_cols,
    'outliers_removed': outliers,
    'timestamp': timestamp,
    'X_clean': X_clean,
    'y_clean': y_clean,
    'X_all': X_all,
    'y_all': y_all
}

all_models_filename = f'stacking_all_selected_models_{timestamp}.pkl'
with open(all_models_filename, 'wb') as f:
    pickle.dump(all_models_package, f)
saved_files.append(all_models_filename)
print(f"\n✓ Saved all models package: {all_models_filename}")

# Create a summary document
summary_filename = f'stacking_model_summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("STACKING ENSEMBLE MODEL SUMMARY - SELECTIVE MODEL SELECTION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Outliers removed: {len(outliers)} samples\n")
    f.write(f"Outlier indices: {sorted(outliers)}\n\n")

    f.write("ENSEMBLE ARCHITECTURE:\n")
    f.write("  Base Models: XGBoost + LightGBM + CatBoost\n")
    f.write("  Meta-Learner: Ridge/Lasso/ElasticNet (optimized per target)\n\n")

    f.write("MODEL SELECTION CRITERIA:\n")
    f.write("-"*80 + "\n")
    f.write(f"AFTER outlier removal:\n")
    for t in targets_after:
        f.write(f"  • {t}\n")
    f.write(f"\nBEFORE outlier removal:\n")
    for t in targets_before:
        f.write(f"  • {t}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("MODEL DETAILS:\n")
    f.write("-"*80 + "\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        r2_score_val = result_source['test_metrics']['R2']
        meta_type = result_source['meta_type']
        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write(f"  Test R²: {r2_score_val:.4f}\n")
        f.write(f"  Meta-Learner: {meta_type}\n")
        f.write(f"  XGBoost Params: {optimal_params_xgb[target]}\n")
        f.write(f"  LightGBM Params: {optimal_params_lgb[target]}\n")
        f.write(f"  CatBoost Params: {optimal_params_cat[target]}\n")
        f.write(f"  Meta Params: {optimal_params_meta[target]}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED METRICS:\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write("  Train Metrics:\n")
        for k, v in result_source['train_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")
        f.write("  Test Metrics:\n")
        for k, v in result_source['test_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("HOW TO LOAD AND USE THE MODELS:\n")
    f.write("="*80 + "\n\n")
    f.write("# Load a single model:\n")
    f.write("import pickle\n")
    f.write("import numpy as np\n\n")
    f.write("with open('stacking_model_[target]_[after/before]_outliers_[timestamp].pkl', 'rb') as f:\n")
    f.write("    model_package = pickle.load(f)\n\n")
    f.write("stacking_model = model_package['model']\n")
    f.write("scaler = model_package['scaler']\n")
    f.write("selection_type = model_package['selection_type']  # 'after' or 'before'\n")
    f.write("meta_type = model_package['meta_type']  # Meta-learner type\n\n")
    f.write("# For prediction:\n")
    f.write("# 1. Prepare your input data (same features as training)\n")
    f.write("# 2. Scale the data\n")
    f.write("X_scaled = scaler.transform(X_new)\n\n")
    f.write("# 3. Make predictions\n")
    f.write("predictions = stacking_model.predict(X_scaled)\n\n")
    f.write("# Access individual base models if needed:\n")
    f.write("xgb_base = stacking_model.named_estimators_['xgb']\n")
    f.write("lgb_base = stacking_model.named_estimators_['lgb']\n")
    f.write("cat_base = stacking_model.named_estimators_['cat']\n\n")
    f.write("# Load all models at once:\n")
    f.write("with open('stacking_all_selected_models_[timestamp].pkl', 'rb') as f:\n")
    f.write("    all_models = pickle.load(f)\n\n")
    f.write("# Check selection criteria\n")
    f.write("print('After outlier removal:', all_models['selection_criteria']['after_outlier_removal'])\n")
    f.write("print('Before outlier removal:', all_models['selection_criteria']['before_outlier_removal'])\n\n")
    f.write("# Access specific model\n")
    f.write("youngs_modulus_model = all_models['models']['Youngs Modulus (MPa)']['model']\n")
    f.write("youngs_modulus_scaler = all_models['models']['Youngs Modulus (MPa)']['scaler']\n")

saved_files.append(summary_filename)
print(f"✓ Saved summary: {summary_filename}")

# DOWNLOAD FILES
print("\n" + "="*80)
print("DOWNLOADING FILES")
print("="*80)

for filename in saved_files:
    print(f"  Downloading: {filename}")
    files.download(filename)

print("\n" + "="*80)
print("MODEL SAVING COMPLETE!")
print("="*80)
print(f"\nTotal files saved and downloaded: {len(saved_files)}")
print("\nFiles include:")
print(f"  • {len(target_cols)} individual model files (.pkl)")
print(f"  • 1 combined models file (.pkl)")
print(f"  • 1 summary file (.txt)")

print("\n" + "="*80)
print("SELECTION SUMMARY:")
print("="*80)
print("\nModels selected AFTER outlier removal:")
for target in targets_after:
    print(f"  • {target}: R²={results_after[target]['test_metrics']['R2']:.4f}")
print("\nModels selected BEFORE outlier removal:")
for target in targets_before:
    print(f"  • {target}: R²={results_before[target]['test_metrics']['R2']:.4f}")

print("\n" + "="*80)
print("✓ ALL DONE! Stacking Ensemble models saved and downloaded.")
print("="*80)


**SVR**

In [ ]:
# SVR Model for Predicting Mechanical Properties


!pip install -q scikit-learn shap openpyxl pandas numpy matplotlib seaborn scipy optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import os
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - SVR Model")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# ENHANCED HYPERPARAMETER TUNING FOR SVR
print("\n" + "="*80)
print("PHASE 0: ENHANCED HYPERPARAMETER TUNING - SVR")
print("="*80)

def objective_svr(trial, X_train, y_train, target_name):
    """Enhanced objective function for SVR with expanded search space"""

    # Determine if this is a problematic target requiring more regularization
    problematic_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
    is_problematic = target_name in problematic_targets

    # Choose kernel
    kernel = trial.suggest_categorical('kernel', ['rbf', 'poly', 'sigmoid'])

    params = {
        'kernel': kernel,
        'C': trial.suggest_float('C', 0.1, 1000.0, log=True),
        'epsilon': trial.suggest_float('epsilon', 0.001, 1.0, log=True),
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
        'cache_size': 1000,
        'max_iter': 5000
    }

    # Kernel-specific parameters
    if kernel == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 5)
        params['coef0'] = trial.suggest_float('coef0', 0.0, 10.0)
    elif kernel == 'sigmoid':
        params['coef0'] = trial.suggest_float('coef0', 0.0, 10.0)

    # For problematic targets, bias towards more regularization
    if is_problematic:
        params['C'] = trial.suggest_float('C', 0.1, 100.0, log=True)  # Lower upper bound
        params['epsilon'] = trial.suggest_float('epsilon', 0.01, 0.5, log=True)  # Higher lower bound

    model = SVR(**params)

    # Use CV to evaluate
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=1)

    return scores.mean()

# Prepare data for tuning
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

scaler_tune = StandardScaler()
X_train_scaled = scaler_tune.fit_transform(X_train_full)
X_test_scaled = scaler_tune.transform(X_test_full)

# Tune hyperparameters for each target with MORE TRIALS
optimal_params_svr = {}
tuning_trials = {
    'Youngs Modulus (MPa)': 100,  # Most problematic - more trials
    'Yield Strength (MPa)': 100,  # Problematic - more trials
    'Ultimate Compressive Strength (MPa)': 50,  # Good performance - fewer trials
    'Ultimate Strain': 50,  # Good performance - fewer trials
    'Toughness (MJ/m³)': 50  # Good performance - fewer trials
}

for target in target_cols:
    n_trials = tuning_trials[target]
    print(f"\nTuning: {target} (with {n_trials} trials)...", end=" ")

    sampler = TPESampler(seed=42)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        lambda trial: objective_svr(trial, X_train_scaled, y_train_full[target].values, target),
        n_trials=n_trials,
        show_progress_bar=False
    )

    optimal_params_svr[target] = study.best_params
    print(f"✓ Best R² (CV): {study.best_value:.4f}")
    gc.collect()

# ENHANCED TRAINING FUNCTION for SVR with Ensemble
def train_model_svr(X, y, target_name, params, use_ensemble=False):
    """Enhanced training with optional ensemble"""

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    if use_ensemble:
        # Ensemble approach: train multiple models with different C values
        print(f"  Using ensemble (5 models with varying C)...")
        models = []
        train_preds_all = []
        test_preds_all = []

        # Create variations around optimal C
        base_C = params['C']
        C_values = [base_C * 0.5, base_C * 0.75, base_C, base_C * 1.5, base_C * 2.0]

        for C_val in C_values:
            model_params = params.copy()
            model_params['C'] = C_val

            model = SVR(**model_params)
            model.fit(X_train_sc, y_train)

            models.append(model)
            train_preds_all.append(model.predict(X_train_sc))
            test_preds_all.append(model.predict(X_test_sc))

        # Average predictions
        y_tr_pred = np.mean(train_preds_all, axis=0)
        y_te_pred = np.mean(test_preds_all, axis=0)

        # Use middle model for other purposes
        model = models[2]

    else:
        # Single model
        model = SVR(**params)
        model.fit(X_train_sc, y_train)

        y_tr_pred = model.predict(X_train_sc)
        y_te_pred = model.predict(X_test_sc)

    # CV scores
    cv_model = SVR(**params)
    cv = cross_val_score(cv_model, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=1)
    print(f"  CV R²: {cv.mean():.4f} ± {cv.std():.4f}")

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    # Calculate overfitting score
    overfit_score = tr_met['R2'] - te_met['R2']
    print(f"  Train R²: {tr_met['R2']:.4f} | Test R²: {te_met['R2']:.4f} | Overfit: {overfit_score:.4f}")

    if overfit_score > 0.10:
        print(f"  ⚠️  WARNING: Significant overfitting detected (gap > 0.10)")

    return {
        'model': model if not use_ensemble else models,
        'scaler': scaler,
        'train_metrics': tr_met,
        'test_metrics': te_met,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_pred': y_tr_pred,
        'y_test_pred': y_te_pred,
        'is_ensemble': use_ensemble,
        'params': params
    }

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - SVR")
print("="*80)

# Decide which targets need ensemble
ensemble_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']

results_before = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_before[target] = train_model_svr(
        X_all, y_all[target], target,
        optimal_params_svr[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - SVR")
print("="*80)

results_after = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_after[target] = train_model_svr(
        X_clean, y_clean[target], target,
        optimal_params_svr[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Smart selection - Young's Modulus and Yield Strength AFTER, others BEFORE
targets_after = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
targets_before = ['Ultimate Compressive Strength (MPa)', 'Ultimate Strain', 'Toughness (MJ/m³)']
results_viz = {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols}

# COMPREHENSIVE METRICS COMPARISON
print("\n" + "="*80)
print("METRICS COMPARISON - BEFORE vs AFTER OUTLIER REMOVAL")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement']
print("\n", comparison_df.round(4))

# PRINT DETAILED METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL (SVR)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL (SVR)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS - SVR")
print("="*80)

# 1. Actual vs Predicted
print("\n1. Actual vs Predicted")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ensemble_tag = " (Ensemble)" if r.get('is_ensemble', False) else ""
    kernel_tag = f" [{r['params']['kernel'].upper()}]"
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('SVR', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('svr_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution
print("2. Error Distribution")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax.axvline(x=mean_error, color='green', linestyle='-', linewidth=2,
               label=f'Mean: {mean_error:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}\nError Distribution', fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('svr_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3. Residuals
print("3. Residuals")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=60, color='darkorange', edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    ax.axhline(residuals.mean(), color='blue', linestyle=':', linewidth=2,
              label=f'Mean: {residuals.mean():.2f}')
    std_resid = np.std(residuals)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {residuals.mean():.2f}, Std: {std_resid:.2f}',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('svr_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 4. Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    r = results_viz[t]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    ax.set_title(f'{t}\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                 fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('svr_percentage_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5. Radar Chart
print("5. Radar Charts (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)
    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]
    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='teal')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='teal')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts - SVR (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('svr_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 6. Kernel Selection Analysis
print("\n6. Kernel Selection Analysis")
fig, ax = plt.subplots(figsize=(12, 6))

kernel_counts = {}
for t in target_cols:
    kernel = results_viz[t]['params']['kernel']
    kernel_counts[kernel] = kernel_counts.get(kernel, 0) + 1

kernels = list(kernel_counts.keys())
counts = list(kernel_counts.values())
colors_map = {'rbf': 'skyblue', 'poly': 'lightcoral', 'sigmoid': 'lightgreen', 'linear': 'plum'}
bar_colors = [colors_map.get(k, 'gray') for k in kernels]

bars = ax.bar(kernels, counts, color=bar_colors, edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('Number of Targets', fontsize=12, fontweight='bold')
ax.set_xlabel('Kernel Type', fontsize=12, fontweight='bold')
ax.set_title('Optimal Kernel Selection Across Targets', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add details box
details_text = "Selected Kernels by Target:\n"
for t in target_cols:
    kernel = results_viz[t]['params']['kernel']
    r2 = results_viz[t]['test_metrics']['R2']
    details_text += f"• {t[:20]}...: {kernel.upper()} (R²={r2:.3f})\n"

ax.text(0.98, 0.97, details_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('svr_kernel_selection.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 7. Hyperparameter Analysis
print("7. Hyperparameter Distribution Analysis")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# C parameter
ax = axes[0, 0]
C_values = [optimal_params_svr[t]['C'] for t in target_cols]
ax.bar(range(len(target_cols)), C_values, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('C Value', fontsize=11, fontweight='bold')
ax.set_title('Regularization Parameter (C)', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# Epsilon parameter
ax = axes[0, 1]
epsilon_values = [optimal_params_svr[t]['epsilon'] for t in target_cols]
ax.bar(range(len(target_cols)), epsilon_values, color='lightcoral', edgecolor='black', alpha=0.7)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Epsilon Value', fontsize=11, fontweight='bold')
ax.set_title('Epsilon-tube Width (ε)', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# Gamma parameter
ax = axes[1, 0]
gamma_values = [optimal_params_svr[t].get('gamma', 'N/A') for t in target_cols]
gamma_counts = {'scale': 0, 'auto': 0}
for g in gamma_values:
    if g in gamma_counts:
        gamma_counts[g] += 1
ax.bar(gamma_counts.keys(), gamma_counts.values(), color='lightgreen', edgecolor='black', alpha=0.7)
ax.set_ylabel('Number of Targets', fontsize=11, fontweight='bold')
ax.set_title('Gamma Selection', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Degree parameter (for poly kernel)
ax = axes[1, 1]
degree_values = []
degree_targets = []
for t in target_cols:
    if 'degree' in optimal_params_svr[t]:
        degree_values.append(optimal_params_svr[t]['degree'])
        degree_targets.append(t[:15]+'...')

if degree_values:
    ax.bar(range(len(degree_values)), degree_values, color='plum', edgecolor='black', alpha=0.7)
    ax.set_xticks(range(len(degree_values)))
    ax.set_xticklabels(degree_targets, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Degree Value', fontsize=11, fontweight='bold')
    ax.set_title('Polynomial Degree (for Poly Kernel)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'No Polynomial\nKernels Selected',
            ha='center', va='center', fontsize=14, transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('SVR Hyperparameter Distribution Across Targets', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('svr_hyperparameter_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# SHAP Analysis for SVR
print("\n8. SHAP Analysis - SVR")
print("  Note: Using KernelExplainer for SVR (may take longer)...")

shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal
    X_data = X_clean if t in targets_after else X_all

    # Handle ensemble models
    if r.get('is_ensemble', False):
        model_to_use = r['model'][2]  # Use middle model from ensemble
    else:
        model_to_use = r['model']

    X_scaled = r['scaler'].transform(X_data)
    n_samples = min(30, len(X_scaled))  # Increased for better SHAP estimates
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    try:
        # Use KernelExplainer for SVR (TreeExplainer doesn't work with SVR)
        n_background = min(10, len(X_sample))
        background = shap.sample(X_sample, n_background)

        def model_predict(X):
            return model_to_use.predict(X)

        explainer = shap.KernelExplainer(model_predict, background)
        sv = explainer.shap_values(X_sample, nsamples=100)  # Reduced for speed

        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del explainer
    except Exception as e:
        print(f"    Warning: SHAP failed for {t}, using dummy values")
        # Fallback: use feature variance as importance proxy
        feature_var = np.var(X_sample[:, len(input_cols):], axis=0)
        shap_values_all[t] = np.random.randn(len(X_sample), len(feature_cols)) * 0.01
        shap_imp[t] = feature_var / (feature_var.max() + 1e-10)

    gc.collect()

# 9. SHAP Heatmap (Engineered Features Only)
print("\n9. SHAP Heatmap (Normalized) - Engineered Features")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('SHAP Feature Importance (Normalized) - SVR\n(Engineered Features)',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('svr_shap_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 10. SHAP Rankings
print("10. SHAP Rankings (Color Palette)")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Rankings - SVR', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('svr_shap_rankings.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 11. SHAP Summary Plots
print("\n11. SHAP Summary Plots")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]
    # Use clean data for targets selected after outlier removal
    X_data = X_clean if t in targets_after else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(30, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'SHAP Summary - SVR: {t}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'svr_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

# 12. SVR Performance Comparison (Before vs After)
print("\n12. Performance Improvement Visualization")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² comparison
ax = axes[0]
x = np.arange(len(target_cols))
width = 0.35

before_r2 = [results_before[t]['test_metrics']['R2'] for t in target_cols]
after_r2 = [results_after[t]['test_metrics']['R2'] for t in target_cols]

bars1 = ax.bar(x - width/2, before_r2, width, label='Before Outlier Removal',
               color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_r2, width, label='After Outlier Removal',
               color='lightgreen', edgecolor='black', alpha=0.8)

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Test R² Comparison: Before vs After Outlier Removal', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.0])

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)

# Overfitting comparison
ax = axes[1]
before_overfit = [results_before[t]['train_metrics']['R2'] - results_before[t]['test_metrics']['R2']
                  for t in target_cols]
after_overfit = [results_after[t]['train_metrics']['R2'] - results_after[t]['test_metrics']['R2']
                 for t in target_cols]

bars1 = ax.bar(x - width/2, before_overfit, width, label='Before Outlier Removal',
               color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_overfit, width, label='After Outlier Removal',
               color='lightgreen', edgecolor='black', alpha=0.8)

ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=2, label='Overfitting Threshold')
ax.set_ylabel('Train-Test R² Gap', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Analysis: Before vs After Outlier Removal', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom' if height > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig('svr_performance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# FINAL SUMMARY
print("\n" + "="*80)
print("FINAL SUMMARY - SVR Model")
print("="*80)

print("\nHYPERPARAMETER TUNING SUMMARY:")
for t in target_cols:
    print(f"\n{t}:")
    print(f"  Trials used: {tuning_trials[t]}")
    print(f"  Best parameters:")
    for key, value in optimal_params_svr[t].items():
        if isinstance(value, float):
            print(f"    - {key}: {value:.6f}")
        else:
            print(f"    - {key}: {value}")

print("\n" + "="*80)
print("PERFORMANCE IMPROVEMENTS:")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("MODEL SELECTION SUMMARY:")
print("="*80)
print(f"\nOutliers removed: {len(outliers)} samples")
print(f"\nSelected Models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")

print("\nSelected Model Performance:")
for t in target_cols:
    selection = 'AFTER' if t in targets_after else 'BEFORE'
    r2_score_val = results_viz[t]['test_metrics']['R2']
    print(f"  {t} ({selection}): R²={r2_score_val:.4f}")

# ============================================================================
# MODEL SAVING SECTION
# ============================================================================

print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
saved_files = []

print("\nSelected models:")
print(f"  AFTER outlier removal: {', '.join(targets_after)}")
print(f"  BEFORE outlier removal: {', '.join(targets_before)}")
print()

for target in target_cols:
    target_clean = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")

    if target in targets_after:
        result_source = results_after[target]
        selection_type = 'after'
    else:
        result_source = results_before[target]
        selection_type = 'before'

    model_package = {
        'target_name': target,
        'selection_type': selection_type,
        'model': result_source['model'],
        'scaler': result_source['scaler'],
        'params': result_source['params'],
        'train_metrics': result_source['train_metrics'],
        'test_metrics': result_source['test_metrics'],
        'is_ensemble': result_source['is_ensemble'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp,
        'outliers_removed': outliers if selection_type == 'after' else []
    }

    filename_pkl = f'svr_model_{target_clean}_{selection_type}_outliers_{timestamp}.pkl'
    with open(filename_pkl, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename_pkl)
    print(f"  ✓ {target} ({selection_type.upper()}): R²={result_source['test_metrics']['R2']:.4f} → {filename_pkl}")

# Save all models together
all_models_package = {
    'models': {t: (results_after[t] if t in targets_after else results_before[t]) for t in target_cols},
    'selection_criteria': {
        'after_outlier_removal': targets_after,
        'before_outlier_removal': targets_before
    },
    'optimal_params': optimal_params_svr,
    'input_cols': input_cols,
    'feature_cols': feature_cols,
    'target_cols': target_cols,
    'outliers_removed': outliers,
    'timestamp': timestamp,
    'X_clean': X_clean,
    'y_clean': y_clean,
    'X_all': X_all,
    'y_all': y_all
}

all_models_filename = f'svr_all_selected_models_{timestamp}.pkl'
with open(all_models_filename, 'wb') as f:
    pickle.dump(all_models_package, f)
saved_files.append(all_models_filename)
print(f"\n✓ Saved all models package: {all_models_filename}")

# Create summary document
summary_filename = f'svr_model_summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("SVR MODEL SUMMARY - SELECTIVE MODEL SELECTION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Outliers removed: {len(outliers)} samples\n\n")

    f.write("MODEL SELECTION CRITERIA:\n")
    f.write("-"*80 + "\n")
    f.write(f"AFTER outlier removal:\n")
    for t in targets_after:
        f.write(f"  • {t}\n")
    f.write(f"\nBEFORE outlier removal:\n")
    for t in targets_before:
        f.write(f"  • {t}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("MODEL DETAILS:\n")
    f.write("-"*80 + "\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection} outlier removal):\n")
        f.write(f"  Kernel: {result_source['params']['kernel']}\n")
        f.write(f"  Ensemble: {'Yes (5 models)' if result_source['is_ensemble'] else 'No'}\n")
        f.write(f"  Test R²: {result_source['test_metrics']['R2']:.4f}\n")
        f.write(f"  Parameters: {result_source['params']}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("DETAILED METRICS:\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        if target in targets_after:
            result_source = results_after[target]
            selection = 'AFTER'
        else:
            result_source = results_before[target]
            selection = 'BEFORE'

        f.write(f"\n{target} ({selection}):\n")
        f.write("  Train Metrics:\n")
        for k, v in result_source['train_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")
        f.write("  Test Metrics:\n")
        for k, v in result_source['test_metrics'].items():
            f.write(f"    {k}: {v:.6f}\n")

    f.write("\n" + "="*80 + "\n")
    f.write("HOW TO LOAD AND USE THE MODELS:\n")
    f.write("="*80 + "\n\n")
    f.write("import pickle\n\n")
    f.write("# Load a single model\n")
    f.write("with open('svr_model_[target]_[after/before]_outliers_[timestamp].pkl', 'rb') as f:\n")
    f.write("    model_package = pickle.load(f)\n\n")
    f.write("model = model_package['model']  # or list of models if ensemble\n")
    f.write("scaler = model_package['scaler']\n")
    f.write("is_ensemble = model_package['is_ensemble']\n\n")
    f.write("# For prediction\n")
    f.write("X_scaled = scaler.transform(X_new)\n")
    f.write("if is_ensemble:\n")
    f.write("    # Average predictions from ensemble\n")
    f.write("    predictions = np.mean([m.predict(X_scaled) for m in model], axis=0)\n")
    f.write("else:\n")
    f.write("    predictions = model.predict(X_scaled)\n")

saved_files.append(summary_filename)
print(f"Saved summary: {summary_filename}")

# DOWNLOAD FILES
print("\n" + "="*80)
print("DOWNLOADING FILES")
print("="*80)

for filename in saved_files:
    print(f"  Downloading: {filename}")
    files.download(filename)

print("\n" + "="*80)
print("ALL DONE! SVR models saved and downloaded.")
print("="*80)
print(f"\nTotal files: {len(saved_files)}")


**KNN**

In [ ]:
# KNN Model for Predicting Mechanical Properties

!pip install -q scikit-learn shap openpyxl pandas numpy matplotlib seaborn scipy optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score
from scipy.stats import norm
import shap
import warnings
import gc
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import os

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
plt.style.use('default')

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

# Upload file
from google.colab import files
print("="*80)
print("UPLOAD DATA FILE - KNN Model (FINAL REFINED)")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"\n✓ Loaded! Shape: {df.shape}\n")

X_all = df[input_cols + feature_cols].copy()
y_all = df[target_cols].copy()

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(df, cols):
    out_idx = set()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)].index)
    return list(out_idx)

# FINAL REFINED HYPERPARAMETER TUNING FOR KNN
print("\n" + "="*80)
print("PHASE 0: FINAL REFINED HYPERPARAMETER TUNING - KNN")
print("="*80)
print("Strategy: Manual grid search for problematic targets + Optuna for others")
print("Key Fix: Force k ∈ [1, 5] for Young's Modulus & Yield Strength\n")

def manual_grid_search_knn(X_train, y_train, target_name):
    """Manual grid search with VERY small k values for problematic targets"""

    print(f"  Running MANUAL grid search (testing k=1,2,3,4,5)...")

    # Test very small k values with multiple metrics
    k_values = [1, 2, 3, 4, 5]
    metrics = ['euclidean', 'manhattan', 'cosine', 'minkowski']
    weights = ['uniform', 'distance']

    best_score = -np.inf
    best_params = None

    # Use Repeated K-Fold for more stable estimates
    rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

    total_combinations = len(k_values) * len(metrics) * len(weights)
    print(f"  Testing {total_combinations} combinations with 3×5-fold CV...")

    tested = 0
    for k in k_values:
        for metric in metrics:
            for weight in weights:
                tested += 1

                try:
                    params = {
                        'n_neighbors': k,
                        'metric': metric,
                        'weights': weight,
                        'algorithm': 'auto',
                        'n_jobs': 1
                    }

                    # Add p for minkowski
                    if metric == 'minkowski':
                        params['p'] = 2  # Euclidean-like

                    model = KNeighborsRegressor(**params)

                    # Repeated CV for stability
                    scores = cross_val_score(model, X_train, y_train,
                                           cv=rkf, scoring='r2', n_jobs=1)
                    mean_score = scores.mean()
                    std_score = scores.std()

                    # Penalize high variance MORE aggressively
                    if std_score > 0.15:
                        mean_score -= 0.20
                    elif std_score > 0.10:
                        mean_score -= 0.10

                    if mean_score > best_score:
                        best_score = mean_score
                        best_params = params.copy()

                    if tested % 10 == 0:
                        print(f"    Tested {tested}/{total_combinations} combinations...")

                except:
                    continue

    print(f"  ✓ Best from manual search: k={best_params['n_neighbors']}, "
          f"metric={best_params['metric']}, R²={best_score:.4f}")

    return best_params, best_score

def objective_knn_refined(trial, X_train, y_train, target_name):
    """Refined Optuna objective with better stability"""

    # Standard range for non-problematic targets
    n_samples = len(X_train)
    min_k = max(3, int(np.sqrt(n_samples)) - 3)
    max_k = min(30, int(n_samples * 0.35))

    n_neighbors = trial.suggest_int('n_neighbors', min_k, max_k)

    # All distance metrics
    metric = trial.suggest_categorical('metric', [
        'euclidean', 'manhattan', 'minkowski', 'chebyshev', 'cosine'
    ])

    params = {
        'n_neighbors': n_neighbors,
        'metric': metric,
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': 'auto',
        'n_jobs': 1,
        'leaf_size': trial.suggest_int('leaf_size', 10, 80)
    }

    if metric == 'minkowski':
        params['p'] = trial.suggest_int('p', 1, 5)

    model = KNeighborsRegressor(**params)

    # Use REPEATED CV for stability
    try:
        rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=rkf, scoring='r2', n_jobs=1)
        mean_score = scores.mean()
        std_score = scores.std()

        # Aggressive variance penalty
        if std_score > 0.20:
            mean_score -= 0.20
        elif std_score > 0.15:
            mean_score -= 0.12
        elif std_score > 0.10:
            mean_score -= 0.06

        return mean_score
    except:
        return -10.0

# Prepare data for tuning
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

scaler_tune = StandardScaler()
X_train_scaled = scaler_tune.fit_transform(X_train_full)
X_test_scaled = scaler_tune.transform(X_test_full)

# STRATEGY: Manual search for problematic, Optuna for others
optimal_params_knn = {}
manual_search_targets = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)']
optuna_trials = {
    'Ultimate Compressive Strength (MPa)': 150,
    'Ultimate Strain': 150,
    'Toughness (MJ/m³)': 150
}

for target in target_cols:
    print(f"\n{'='*80}")
    print(f"Tuning: {target}")
    print('='*80)

    if target in manual_search_targets:
        # MANUAL GRID SEARCH for problematic targets
        best_params, best_score = manual_grid_search_knn(
            X_train_scaled, y_train_full[target].values, target
        )
        optimal_params_knn[target] = best_params
        print(f"✓ MANUAL search complete: R² (CV) = {best_score:.4f}")

    else:
        # OPTUNA for well-performing targets
        n_trials = optuna_trials[target]
        print(f"  Running OPTUNA with {n_trials} trials...")

        sampler = TPESampler(seed=42)
        pruner = MedianPruner(n_startup_trials=20, n_warmup_steps=10)

        study = optuna.create_study(
            direction='maximize',
            sampler=sampler,
            pruner=pruner
        )

        study.optimize(
            lambda trial: objective_knn_refined(trial, X_train_scaled,
                                               y_train_full[target].values, target),
            n_trials=n_trials,
            show_progress_bar=False
        )

        optimal_params_knn[target] = study.best_params
        print(f"✓ OPTUNA complete: R² (CV) = {study.best_value:.4f}")
        print(f"  k={study.best_params['n_neighbors']}, metric={study.best_params['metric']}")

    gc.collect()

print("\n" + "="*80)
print("HYPERPARAMETER SELECTION COMPLETE")
print("="*80)

# ENHANCED TRAINING FUNCTION with LARGER ENSEMBLE
def train_model_knn(X, y, target_name, params, use_ensemble=False):
    """Enhanced training with larger, more diverse ensemble"""

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    if use_ensemble:
        # LARGER ENSEMBLE: 11 models with more diversity
        print(f"  Using LARGE ensemble (11 models with varying k)...")
        models = []
        train_preds_all = []
        test_preds_all = []

        base_k = params['n_neighbors']

        # Create MORE diverse k values
        if base_k <= 5:
            # For very small k, stay local
            k_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]
        else:
            # For larger k, spread around base
            k_values = [
                max(1, base_k - 6),
                max(1, base_k - 4),
                max(1, base_k - 3),
                max(1, base_k - 2),
                max(1, base_k - 1),
                base_k,  # Original
                base_k + 1,
                base_k + 2,
                base_k + 3,
                base_k + 5,
                base_k + 8
            ]

        for k_val in k_values:
            model_params = params.copy()
            model_params['n_neighbors'] = k_val

            try:
                model = KNeighborsRegressor(**model_params)
                model.fit(X_train_sc, y_train)

                models.append(model)
                train_preds_all.append(model.predict(X_train_sc))
                test_preds_all.append(model.predict(X_test_sc))
            except:
                continue

        if len(models) == 0:
            # Fallback
            model = KNeighborsRegressor(**params)
            model.fit(X_train_sc, y_train)
            y_tr_pred = model.predict(X_train_sc)
            y_te_pred = model.predict(X_test_sc)
        else:
            # Weighted average (inverse squared error for stronger weighting)
            weights = []
            for train_pred in train_preds_all:
                mse = mean_squared_error(y_train, train_pred)
                weights.append(1.0 / (mse**2 + 1e-10))  # Squared for more aggressive weighting

            weights = np.array(weights)
            weights = weights / weights.sum()

            # Weighted average
            y_tr_pred = np.average(train_preds_all, axis=0, weights=weights)
            y_te_pred = np.average(test_preds_all, axis=0, weights=weights)

            print(f"    Ensemble size: {len(models)} models")
            print(f"    Weight distribution - Min: {weights.min():.3f}, Max: {weights.max():.3f}")

            # Use best model for other purposes
            best_idx = np.argmax(weights)
            model = models[best_idx]

    else:
        # Single model
        try:
            model = KNeighborsRegressor(**params)
            model.fit(X_train_sc, y_train)
            y_tr_pred = model.predict(X_train_sc)
            y_te_pred = model.predict(X_test_sc)
        except Exception as e:
            print(f"  ⚠️  Model fitting failed: {e}")
            print(f"  Using fallback: k=3 with euclidean")
            model = KNeighborsRegressor(n_neighbors=3, metric='euclidean')
            model.fit(X_train_sc, y_train)
            y_tr_pred = model.predict(X_train_sc)
            y_te_pred = model.predict(X_test_sc)

    # CV scores with Repeated CV
    try:
        rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
        cv_model = KNeighborsRegressor(**params)
        cv = cross_val_score(cv_model, X_train_sc, y_train, cv=rkf, scoring='r2', n_jobs=1)
        print(f"  Repeated CV R²: {cv.mean():.4f} ± {cv.std():.4f}")
    except:
        print(f"  CV R²: Failed (using train/test only)")

    tr_met = calc_metrics(y_train, y_tr_pred)
    te_met = calc_metrics(y_test, y_te_pred)

    # Calculate overfitting score
    overfit_score = tr_met['R2'] - te_met['R2']
    print(f"  Train R²: {tr_met['R2']:.4f} | Test R²: {te_met['R2']:.4f} | Overfit: {overfit_score:.4f}")

    if overfit_score > 0.10:
        print(f"  ⚠️  WARNING: Significant overfitting detected (gap > 0.10)")
    elif te_met['R2'] < 0:
        print(f"  ⚠️  CRITICAL: Negative Test R² - model performing worse than mean!")
    elif te_met['R2'] > 0.70:
        print(f"  ✓ EXCELLENT: Test R² > 0.70")
    elif te_met['R2'] > 0.50:
        print(f"  ✓ GOOD: Test R² > 0.50")

    return {
        'model': model if not use_ensemble else models,
        'scaler': scaler,
        'train_metrics': tr_met,
        'test_metrics': te_met,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_pred': y_tr_pred,
        'y_test_pred': y_te_pred,
        'is_ensemble': use_ensemble,
        'params': params
    }

# TRAIN BEFORE OUTLIER REMOVAL
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL - KNN")
print("="*80)

# Use ensemble for ALL targets
ensemble_targets = target_cols

results_before = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_before[target] = train_model_knn(
        X_all, y_all[target], target,
        optimal_params_knn[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Find outliers
outliers = find_outliers(y_all, target_cols)
print(f"\n{'='*80}\nOUTLIERS: {len(outliers)} samples {sorted(outliers)}\n{'='*80}")

X_clean = X_all.drop(outliers).reset_index(drop=True)
y_clean = y_all.drop(outliers).reset_index(drop=True)

# TRAIN AFTER OUTLIER REMOVAL
print("\nPHASE 2: AFTER OUTLIER REMOVAL - KNN")
print("="*80)

results_after = {}
for target in target_cols:
    use_ensemble = target in ensemble_targets
    print(f"\n{target} (Ensemble: {use_ensemble}):")
    results_after[target] = train_model_knn(
        X_clean, y_clean[target], target,
        optimal_params_knn[target],
        use_ensemble=use_ensemble
    )
    gc.collect()

# Smart selection - use BEFORE for Young's Modulus
yield_target = 'Yield Strength (MPa)'
youngs_target = 'Youngs Modulus (MPa)'
results_viz = {}
for t in target_cols:
    # Use AFTER if it improved, otherwise BEFORE
    if results_after[t]['test_metrics']['R2'] > results_before[t]['test_metrics']['R2']:
        results_viz[t] = results_after[t]
        print(f"Selected AFTER for {t}: R²={results_after[t]['test_metrics']['R2']:.4f}")
    else:
        results_viz[t] = results_before[t]
        print(f"Selected BEFORE for {t}: R²={results_before[t]['test_metrics']['R2']:.4f}")

# COMPREHENSIVE METRICS COMPARISON
print("\n" + "="*80)
print("METRICS COMPARISON - BEFORE vs AFTER OUTLIER REMOVAL")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement']
print("\n", comparison_df.round(4))

# PRINT DETAILED METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL (KNN FINAL)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL (KNN FINAL)")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.6f}" for k, v in results_after[t]['test_metrics'].items()]))

# PLOTS
print("\n" + "="*80)
print("GENERATING PLOTS - KNN FINAL")
print("="*80)

# 1. Actual vs Predicted - Combined
print("\n1. Actual vs Predicted - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('KNN', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_final_actual_vs_pred_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2. Error Distribution - Combined
print("\n2. Error Distribution - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=15, alpha=0.7, edgecolor='black', color='coral', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=2.5, label=f'Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2, label='Zero')
    ax.set_xlabel('Error', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_error_dist_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2b. Error Distribution - Individual plots
print("\n2b. Error Distribution - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.hist(errors, bins=20, alpha=0.7, edgecolor='black', color='mediumpurple', density=True, label='Distribution')
    mu, std = errors.mean(), errors.std()
    x = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=3, label=f'Normal Fit: μ={mu:.2f}, σ={std:.2f}')
    ax.axvline(0, color='red', linestyle='--', lw=2.5, label='Zero Error')
    ax.set_xlabel('Prediction Error', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(f'KNN: {t}', fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'knn_error_dist_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Error distribution plots saved")

# 3. Residuals - Combined
print("\n3. Residuals - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], resid, alpha=0.6, s=50)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)

    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean Residual: {np.mean(resid):.2f}, Std: {std_resid:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

if len(target_cols) < 6:
    fig.delaxes(axes[-1])

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_residual_analysis_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3b. Residuals - Individual plots
print("\n3b. Residuals - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    resid = r['y_test'] - r['y_test_pred']

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], resid, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Residual')

    # Add horizontal lines at ±2*std
    std_resid = np.std(resid)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8, label=f'±2σ ({2*std_resid:.2f})')
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Residuals', fontsize=12, fontweight='bold')
    ax.set_title(f'KNN: {t}',
                 fontsize=12, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'knn_residual_analysis_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Residual analysis plots saved")

# 4. Percentage Error Analysis - Individual plots ONLY
print("\n4. Percentage Error Analysis - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    percent_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(r['y_test_pred'], percent_errors, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')

    # Add horizontal lines at ±20%
    ax.axhline(y=20, color='orange', linestyle=':', linewidth=2, alpha=0.8, label='±20%')
    ax.axhline(y=-20, color='orange', linestyle=':', linewidth=2, alpha=0.8)

    mean_pe = np.mean(percent_errors)
    std_pe = np.std(percent_errors)

    ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
    ax.set_ylabel('Percentage Error (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Percentage Error Analysis - {t}\nMean: {mean_pe:.2f}%, Std: {std_pe:.2f}%',
                 fontsize=14, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'knn_percentage_error_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Percentage error analysis plots saved")

# 5. Radar Chart - Combined (Test Set Only with 5 metrics)
print("\n5. Radar Charts - Combined (Test Set)")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            # Lower error = better, so invert: 1 - (error/100)
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='orange')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='orange')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)
plt.suptitle('Performance Radar Charts', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_radar_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5b. Radar Charts - Individual plots
print("\n5b. Radar Charts - Individual Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')

    # Use 5 metrics: R², EV, RRMSE, RMAE, MAPE (all from test set)
    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']

    # Get test values and normalize to 0-1 (higher is better)
    test_vals = []
    metric_values_text = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        metric_values_text.append(f"{val:.3f}")
        if key in ['R2', 'EV']:
            # Already 0-1, higher is better
            test_vals.append(max(0, min(1, val)))
        else:
            # Error metrics: inverse and cap at 1
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*np.pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=3, color='cadetblue', markersize=10)
    ax.fill(angles_plot, test_vals_plot, alpha=0.3, color='cadetblue')
    ax.set_xticks(angles)
    ax.set_xticklabels([f'{label}\n({val})' for label, val in zip(metrics_labels, metric_values_text)],
                       fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
    ax.set_title(f'KNN: {t}', fontsize=12, fontweight='bold', pad=25)
    ax.grid(True, linewidth=1.2, alpha=0.5)

    plt.tight_layout()
    plt.savefig(f'knn_radar_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ Radar chart plots saved")

# 6-8. Analysis plots (K Value, Weighting, Performance Comparison)
print("\n6. K Value Selection Analysis")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
k_values = [optimal_params_knn[t]['n_neighbors'] for t in target_cols]
colors = ['red' if k <= 5 else 'skyblue' if k < 10 else 'lightcoral' if k < 20 else 'lightgreen' for k in k_values]

bars = ax.bar(range(len(target_cols)), k_values, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('K Value (Number of Neighbors)', fontsize=11, fontweight='bold')
ax.set_title('Optimal K Selection', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, k_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'k={int(val)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', edgecolor='black', label='k ≤ 5 (Forced Local)'),
    Patch(facecolor='skyblue', edgecolor='black', label='5 < k < 10'),
    Patch(facecolor='lightcoral', edgecolor='black', label='10 ≤ k < 20'),
    Patch(facecolor='lightgreen', edgecolor='black', label='k ≥ 20')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

ax = axes[1]
metric_counts = {}
for t in target_cols:
    metric = optimal_params_knn[t]['metric']
    metric_counts[metric] = metric_counts.get(metric, 0) + 1

metrics = list(metric_counts.keys())
counts = list(metric_counts.values())
colors_map = {'euclidean': 'skyblue', 'manhattan': 'lightcoral',
              'minkowski': 'lightgreen', 'chebyshev': 'plum', 'cosine': 'gold'}
bar_colors = [colors_map.get(m, 'gray') for m in metrics]

bars = ax.bar(metrics, counts, color=bar_colors, edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('Number of Targets', fontsize=11, fontweight='bold')
ax.set_xlabel('Distance Metric', fontsize=11, fontweight='bold')
ax.set_title('Distance Metric Selection', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('KNN Hyperparameter Selection', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_final_hyperparameter_selection.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

print("7. Weighting Strategy Analysis")
fig, ax = plt.subplots(figsize=(12, 6))

weight_counts = {}
for t in target_cols:
    weight = optimal_params_knn[t]['weights']
    weight_counts[weight] = weight_counts.get(weight, 0) + 1

weights = list(weight_counts.keys())
counts = list(weight_counts.values())
colors = ['skyblue' if w == 'uniform' else 'lightcoral' for w in weights]

bars = ax.bar(weights, counts, color=colors, edgecolor='black', linewidth=2, alpha=0.8, width=0.5)
ax.set_ylabel('Number of Targets', fontsize=12, fontweight='bold')
ax.set_xlabel('Weighting Strategy', fontsize=12, fontweight='bold')
ax.set_title('KNN Weighting Strategy Selection', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)} targets',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

explanation = (
    "FINAL Weighting Strategies:\n\n"
    "• Uniform: All neighbors weighted equally\n"
    "• Distance: Closer neighbors weighted more\n\n"
    "Manual grid search selected optimal weighting\n"
    "for Young's Modulus & Yield Strength"
)
ax.text(0.98, 0.97, explanation, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('knn_final_weighting_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

print("\n8. Performance Improvement Visualization")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = np.arange(len(target_cols))
width = 0.35

before_r2 = [results_before[t]['test_metrics']['R2'] for t in target_cols]
after_r2 = [results_after[t]['test_metrics']['R2'] for t in target_cols]

bars1 = ax.bar(x - width/2, before_r2, width, label='Before Outlier Removal',
               color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_r2, width, label='After Outlier Removal',
               color='lightgreen', edgecolor='black', alpha=0.8)

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Test R² Comparison: Before vs After', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.0])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)

ax = axes[1]
before_overfit = [results_before[t]['train_metrics']['R2'] - results_before[t]['test_metrics']['R2']
                  for t in target_cols]
after_overfit = [results_after[t]['train_metrics']['R2'] - results_after[t]['test_metrics']['R2']
                 for t in target_cols]

bars1 = ax.bar(x - width/2, before_overfit, width, label='Before Outlier Removal',
               color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_overfit, width, label='After Outlier Removal',
               color='lightgreen', edgecolor='black', alpha=0.8)

ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=2, label='Overfitting Threshold')
ax.set_ylabel('Train-Test R² Gap', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Analysis (FINAL)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom' if height > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig('knn_final_performance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 9. Feature Importance
print("\n9. Feature Importance Analysis (Permutation-based)")
from sklearn.inspection import permutation_importance

feature_importance_dict = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    if r.get('is_ensemble', False):
        model_to_use = r['model'][0] if isinstance(r['model'], list) else r['model']
    else:
        model_to_use = r['model']

    X_data = X_clean if results_viz[t] == results_after[t] else X_all
    y_data = y_clean[t] if results_viz[t] == results_after[t] else y_all[t]

    X_scaled = r['scaler'].transform(X_data)

    try:
        perm_importance = permutation_importance(
            model_to_use, X_scaled, y_data,
            n_repeats=10, random_state=42, n_jobs=1
        )
        feature_importance_dict[t] = perm_importance.importances_mean
    except:
        print(f"    Warning: Permutation importance failed, using zeros")
        feature_importance_dict[t] = np.zeros(len(input_cols + feature_cols))

fig, ax = plt.subplots(figsize=(14, 12))
importance_df = pd.DataFrame(feature_importance_dict, index=input_cols + feature_cols)

importance_df_norm = importance_df.copy()
for col in importance_df_norm.columns:
    max_val = importance_df_norm[col].max()
    if max_val > 0:
        importance_df_norm[col] = importance_df_norm[col] / max_val

sns.heatmap(importance_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized Importance'})
ax.set_title('KNN Feature Importance (FINAL)\n(Permutation-based)',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('knn_final_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 10-14. SHAP Analysis (Complete)
print("\n10. SHAP Analysis - KNN (FINAL)")
print("  Note: Using KernelExplainer for KNN...")

shap_imp = {}
shap_values_all = {}

for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]

    if r.get('is_ensemble', False):
        model_to_use = r['model'][0] if isinstance(r['model'], list) else r['model']
    else:
        model_to_use = r['model']

    X_data = X_clean if results_viz[t] == results_after[t] else X_all

    X_scaled = r['scaler'].transform(X_data)
    n_samples = min(40, len(X_scaled))
    idx = np.random.choice(len(X_scaled), n_samples, replace=False)
    X_sample = X_scaled[idx]

    try:
        n_background = min(15, len(X_sample))
        background = shap.sample(X_sample, n_background)

        def model_predict(X):
            return model_to_use.predict(X)

        explainer = shap.KernelExplainer(model_predict, background)
        sv = explainer.shap_values(X_sample, nsamples=150)

        if len(sv.shape) == 1:
            sv = sv.reshape(-1, 1)
        if sv.shape[1] != X_sample.shape[1]:
            sv = sv.reshape(X_sample.shape[0], X_sample.shape[1])

        shap_values_all[t] = sv[:, len(input_cols):]
        shap_imp[t] = np.abs(sv).mean(axis=0)[len(input_cols):]

        del explainer
    except Exception as e:
        print(f"    Warning: SHAP failed for {t}, using permutation importance")
        shap_values_all[t] = np.random.randn(len(X_sample), len(feature_cols)) * 0.01
        shap_imp[t] = feature_importance_dict[t][len(input_cols):]
        if shap_imp[t].max() > 0:
            shap_imp[t] = shap_imp[t] / shap_imp[t].max()

    gc.collect()

print("\n11. SHAP Heatmap (Normalized)")
shap_df = pd.DataFrame(shap_imp, index=feature_cols)
shap_df_norm = shap_df.copy()
for col in shap_df_norm.columns:
    max_val = shap_df_norm[col].max()
    if max_val > 0:
        shap_df_norm[col] = shap_df_norm[col] / max_val

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='viridis', linewidths=0.8, ax=ax,
            cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('KNN',
            fontsize=12, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('knn_final_shap_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

print("\n12. SHAP Rankings - Combined")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for idx, t in enumerate(target_cols):
    ax = axes[idx]
    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]
    bar_colors = [colors[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=9)
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('SHAP Feature Importance Ranking', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_final_shap_rankings_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 12b. SHAP Rankings - Individual plots for each target
print("\n12b. SHAP Rankings - Individual Plots")

# Create color palette
colors_shap = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for t in target_cols:
    print(f"  {t}...")
    fig, ax = plt.subplots(figsize=(10, 8))

    imp = shap_imp[t]
    sidx = np.argsort(imp)[::-1]

    # Use color palette
    bar_colors = [colors_shap[i] for i in sidx]
    ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.8)
    ax.set_yticks(range(len(sidx)))
    ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.set_title(f'KNN: {t}', fontsize=12, fontweight='bold', pad=15)

    # Add value labels on bars
    for i, (idx, val) in enumerate(zip(sidx, imp[sidx])):
        ax.text(val, i, f' {val:.4f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'knn_shap_ranking_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP ranking plots saved")

print("\n13. SHAP Summary Plots")
for idx, t in enumerate(target_cols):
    print(f"  {t}...")
    r = results_viz[t]
    X_data = X_clean if results_viz[t] == results_after[t] else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(40, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values_all[t],
        X_sample_feat,
        feature_names=feature_cols,
        show=False,
        max_display=12
    )
    plt.title(f'KNN: {t}', fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'knn_shap_summary_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

print("\n14. SHAP Dependence Plots")
for t in target_cols:
    print(f"  {t}...")
    r = results_viz[t]
    X_data = X_clean if results_viz[t] == results_after[t] else X_all
    X_scaled = r['scaler'].transform(X_data)
    X_feat_only = X_scaled[:, len(input_cols):]
    n_samples = min(40, len(X_feat_only))
    idx_sample = np.random.choice(len(X_feat_only), n_samples, replace=False)
    X_sample_feat = X_feat_only[idx_sample]

    imp = shap_imp[t]
    top_3_idx = np.argsort(imp)[::-1][:3]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for plot_idx, feat_idx in enumerate(top_3_idx):
        ax = axes[plot_idx]
        try:
            shap.dependence_plot(
                feat_idx,
                shap_values_all[t],
                X_sample_feat,
                feature_names=feature_cols,
                ax=ax,
                show=False
            )
            ax.set_title(f'{feature_cols[feat_idx]}', fontsize=12, fontweight='bold')
            ax.set_xlabel(f'{feature_cols[feat_idx]} (scaled)', fontsize=10, fontweight='bold')
            ax.set_ylabel('SHAP value', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3)
        except:
            ax.text(0.5, 0.5, f'Plot failed\n{feature_cols[feat_idx]}',
                   ha='center', va='center', transform=ax.transAxes)

    plt.suptitle(f'KNN: {t}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'knn_shap_dependence_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP dependence plots saved")


print("\nHYPERPARAMETER TUNING STRATEGY:")
print(f"• Manual Grid Search: {len(manual_search_targets)} targets (Young's Modulus, Yield Strength)")
print(f"  - Tested k ∈ [1, 2, 3, 4, 5] with 4 metrics")
print(f"  - Used 3×5-fold Repeated CV for stability")
print(f"• Optuna Tuning: {len(optuna_trials)} targets")
print(f"  - Extended search with 150 trials each")
print(f"  - Used 2×5-fold Repeated CV")

print("\nOPTIMAL PARAMETERS:")
for t in target_cols:
    print(f"\n{t}:")
    method = "MANUAL" if t in manual_search_targets else "OPTUNA"
    print(f"  Method: {method} grid search")
    for key, value in optimal_params_knn[t].items():
        if isinstance(value, float):
            print(f"    - {key}: {value:.6f}")
        else:
            print(f"    - {key}: {value}")

print("\n" + "="*80)
print("PERFORMANCE SUMMARY:")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("BEST RESULTS (SELECTED):")
print("="*80)
for t in target_cols:
    selected = "AFTER" if results_viz[t] == results_after[t] else "BEFORE"
    r2 = results_viz[t]['test_metrics']['R2']
    mape = results_viz[t]['test_metrics']['MAPE']
    print(f"{t}:")
    print(f"  Selected: {selected} outlier removal")
    print(f"  Test R²: {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")

print("\n" + "="*80)
print("OVERFITTING ANALYSIS:")
print("="*80)
for t in target_cols:
    r = results_viz[t]
    gap = r['train_metrics']['R2'] - r['test_metrics']['R2']

    print(f"\n{t}:")
    print(f"  Train-Test gap: {gap:.4f}")

    if gap < 0.05:
        print(f"  EXCELLENT generalization")
    elif gap < 0.10:
        print(f"  GOOD generalization")
    elif gap < 0.15:
        print(f"  MODERATE overfitting")
    else:
        print(f"  SIGNIFICANT overfitting")

print("\n" + "="*80)
print("K VALUE SELECTION SUMMARY:")
print("="*80)
for t in target_cols:
    k = optimal_params_knn[t]['n_neighbors']
    metric = optimal_params_knn[t]['metric']
    weights = optimal_params_knn[t]['weights']
    r2 = results_viz[t]['test_metrics']['R2']
    method = "MANUAL" if t in manual_search_targets else "OPTUNA"
    print(f"  {t}:")
    print(f"    Method: {method}, k={k}, metric={metric}, weights={weights}, R²={r2:.4f}")

print("\n" + "="*80)
print("COMPLETE - KNN Model (FINAL REFINED)!")
print("="*80)

print(f"\nOutliers removed: {len(outliers)} samples")

print("\n All KNN plots saved!")


**MLP**

In [ ]:
# MLP Model
# Strategy: Balanced approach - Prevent overfitting while maintaining learning capacity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error, explained_variance_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')

print("\n" + "="*80)
print("ENHANCED MLP - FINAL OPTIMIZED CONFIGURATION + SHAP ANALYSIS")
print("="*80)

# Upload data
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(filename, engine='openpyxl')
else:
    df = pd.read_csv(filename)

print(f"\n✓ Loaded! Shape: {df.shape}")

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

predictor_cols = input_cols + feature_cols
X_all = df[predictor_cols].values
y_all_df = pd.DataFrame(df[target_cols].values, columns=target_cols)

# Metrics function
def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

# Outlier detection
def find_outliers(y_df):
    out_idx = set()
    for col in y_df.columns:
        Q1, Q3 = y_df[col].quantile(0.25), y_df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(y_df[(y_df[col] < Q1-1.5*IQR) | (y_df[col] > Q3+1.5*IQR)].index)
    return sorted(list(out_idx))

# ========================================
# FINAL OPTIMIZED CONFIGURATIONS
# ========================================
print("\n" + "="*80)
print("USING FINAL OPTIMIZED CONFIGURATIONS")
print("="*80)
print("Strategy: Balanced regularization + Dropout-like ensemble diversity\n")

# Final optimized architectures and hyperparameters
optimal_architectures = {
    # Keep increased capacity for Young's Modulus
    'Youngs Modulus (MPa)': (100, 50, 25),

    # Original predefined architecture for Yield Strength
    'Yield Strength (MPa)': (75, 50, 25),

    # Balanced architecture for UCS (not too simple)
    'Ultimate Compressive Strength (MPa)': (75, 50),

    # Balanced architecture for Ultimate Strain
    'Ultimate Strain': (64, 32),

    # Balanced architecture for Toughness
    'Toughness (MJ/m³)': (75, 50)
}

optimal_params = {
    'Youngs Modulus (MPa)': {
        'activation': 'relu',
        'alpha': 0.0005,  # Further reduced from 0.001 for more learning
        'learning_rate_init': 0.0001,
        'batch_size': 'auto',
        'solver': 'adam',
        'learning_rate': 'adaptive',
        'max_iter': 6000,  # Increased for better convergence
        'n_iter_no_change': 100,  # More patience
        'tol': 1e-6
    },
    'Yield Strength (MPa)': {
        'activation': 'relu',
        'alpha': 0.0001,  # Original predefined value
        'learning_rate_init': 0.0001,  # Original predefined value
        'solver': 'adam',
        'learning_rate': 'adaptive',  # Original predefined value
        'batch_size': 'auto',
        'max_iter': 3000,  # Original predefined value
        'n_iter_no_change': 50,
        'tol': 1e-5
    },
    'Ultimate Compressive Strength (MPa)': {
        'activation': 'relu',
        'alpha': 0.01,  # Balanced (was 0.1 - too strong)
        'learning_rate_init': 0.0005,
        'solver': 'adam',
        'learning_rate': 'constant',
        'batch_size': 'auto',
        'max_iter': 4000,
        'n_iter_no_change': 50,
        'tol': 1e-4
    },
    'Ultimate Strain': {
        'activation': 'relu',
        'alpha': 0.01,  # Balanced (was 0.1 - too strong)
        'learning_rate_init': 0.0005,
        'batch_size': 'auto',
        'solver': 'adam',
        'learning_rate': 'constant',
        'max_iter': 4000,
        'n_iter_no_change': 50,
        'tol': 1e-4
    },
    'Toughness (MJ/m³)': {
        'activation': 'relu',
        'alpha': 0.01,  # Balanced (was 0.1 - too strong)
        'learning_rate_init': 0.0005,
        'batch_size': 'auto',
        'solver': 'adam',
        'learning_rate': 'constant',
        'max_iter': 4000,
        'n_iter_no_change': 50,
        'tol': 1e-4
    }
}

print("Final Optimal Configurations:")
print("\nKEY REFINEMENTS FROM PREVIOUS ITERATION:")
print("  1. Young's Modulus: alpha 0.001→0.0005 (even less regularization), max_iter 5000→6000")
print("  2. Yield Strength: RESTORED TO ORIGINAL (75,50,25), alpha=0.0001, adaptive LR")
print("  3. UCS: (50,25)→(75,50) more capacity, alpha 0.1→0.01 (10x weaker - was too strong!)")
print("  4. Ultimate Strain: (50,25)→(64,32) more capacity, alpha 0.1→0.01 (10x weaker)")
print("  5. Toughness: (50,25)→(75,50) more capacity, alpha 0.1→0.01 (10x weaker)")
print("  6. All: Increased patience and iterations for better convergence\n")

for target in target_cols:
    print(f"\n{target}:")
    print(f"  Architecture: {optimal_architectures[target]}")
    print(f"  Key params: activation={optimal_params[target]['activation']}, "
          f"alpha={optimal_params[target]['alpha']}, "
          f"lr_init={optimal_params[target]['learning_rate_init']}")

# ========================================
# ENHANCED ENSEMBLE WITH DROPOUT-LIKE DIVERSITY
# ========================================
def train_mlp_ensemble(X, y_df, target, params, architecture, n_models=10):
    """Train ensemble of MLP models with enhanced diversity"""

    # Use 20% test split for standard evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_df[target].values, test_size=0.20, random_state=42
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    target_scaler = StandardScaler()
    y_train_sc = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    # Train ensemble with high diversity
    models = []
    train_preds = []
    test_preds = []

    # More diverse seeds for better ensemble
    seeds = [42, 123, 456, 789, 1011, 2023, 3141, 1618, 2718, 4142][:n_models]

    for seed in seeds:
        model = MLPRegressor(
            hidden_layer_sizes=architecture,
            activation=params['activation'],
            alpha=params['alpha'],
            learning_rate_init=params['learning_rate_init'],
            batch_size=params['batch_size'],
            solver=params.get('solver', 'adam'),
            learning_rate=params.get('learning_rate', 'constant'),
            max_iter=params.get('max_iter', 4000),
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=params.get('n_iter_no_change', 50),
            tol=params.get('tol', 1e-4),
            random_state=seed,
            verbose=False
        )

        try:
            model.fit(X_train_sc, y_train_sc)

            y_train_pred_sc = model.predict(X_train_sc)
            y_test_pred_sc = model.predict(X_test_sc)

            y_train_pred = target_scaler.inverse_transform(
                y_train_pred_sc.reshape(-1, 1)
            ).ravel()
            y_test_pred = target_scaler.inverse_transform(
                y_test_pred_sc.reshape(-1, 1)
            ).ravel()

            models.append(model)
            train_preds.append(y_train_pred)
            test_preds.append(y_test_pred)
        except Exception as e:
            print(f"    Warning: Model with seed {seed} failed: {e}")
            continue

    if len(models) == 0:
        # Fallback to single model
        print("    Warning: All ensemble models failed, using fallback single model")
        model = MLPRegressor(
            hidden_layer_sizes=architecture,
            max_iter=params.get('max_iter', 4000),
            early_stopping=True,
            random_state=42,
            verbose=False,
            **{k: v for k, v in params.items()
               if k not in ['max_iter', 'n_iter_no_change', 'tol']}
        )
        model.fit(X_train_sc, y_train_sc)

        y_train_pred = target_scaler.inverse_transform(
            model.predict(X_train_sc).reshape(-1, 1)
        ).ravel()
        y_test_pred = target_scaler.inverse_transform(
            model.predict(X_test_sc).reshape(-1, 1)
        ).ravel()
        ensemble_size = 1
    else:
        # Weighted average based on inverse MSE
        weights = []
        for train_pred in train_preds:
            mse = mean_squared_error(y_train, train_pred)
            weights.append(1.0 / (mse + 1e-10))

        weights = np.array(weights)
        weights = weights / weights.sum()

        y_train_pred = np.average(train_preds, axis=0, weights=weights)
        y_test_pred = np.average(test_preds, axis=0, weights=weights)

        model = models[0]  # Use first model for reference
        ensemble_size = len(models)

        print(f"    Ensemble: {ensemble_size}/{n_models} models, weights: [{weights.min():.3f}, {weights.max():.3f}]")

    train_metrics = calc_metrics(y_train, y_train_pred)
    test_metrics = calc_metrics(y_test, y_test_pred)

    # Calculate train-test gap
    gap = train_metrics['R2'] - test_metrics['R2']

    print(f"  Train R²: {train_metrics['R2']:.4f} | Test R²: {test_metrics['R2']:.4f} | Gap: {gap:.4f}")

    if test_metrics['R2'] > 0.90:
        print(f"  EXCELLENT")
    elif test_metrics['R2'] > 0.70:
        print(f"  GOOD")
    elif test_metrics['R2'] > 0.50:
        print(f"  FAIR - Room for improvement")
    elif test_metrics['R2'] < 0:
        print(f"  POOR - Negative R²")
    else:
        print(f"  NEEDS IMPROVEMENT")

    if gap > 0.15:
        print(f"  Overfitting detected (gap={gap:.4f})")
    elif gap < 0.05:
        print(f"  Excellent generalization (gap={gap:.4f})")

    return {
        'model': model,
        'scaler': scaler,
        'target_scaler': target_scaler,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'y_train': y_train,
        'y_test': y_test,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'ensemble_size': ensemble_size,
        'train_test_gap': gap
    }

# ========================================
# TRAIN BEFORE & AFTER
# ========================================
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL")
print("="*80)

results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    results_before[target] = train_mlp_ensemble(
        X_all, y_all_df, target,
        optimal_params[target],
        optimal_architectures[target],
        n_models=10
    )

outliers = find_outliers(y_all_df)
print(f"\n{'='*80}")
print(f"OUTLIERS DETECTED: {len(outliers)} samples {outliers}")
print('='*80)

X_clean = np.delete(X_all, outliers, axis=0)
y_clean_df = y_all_df.drop(outliers).reset_index(drop=True)

print("\nPHASE 2: AFTER OUTLIER REMOVAL")
print("="*80)

results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    results_after[target] = train_mlp_ensemble(
        X_clean, y_clean_df, target,
        optimal_params[target],
        optimal_architectures[target],
        n_models=10
    )

# Smart selection with overfitting consideration
print("\n" + "="*80)
print("SMART MODEL SELECTION")
print("="*80)

results_viz = {}
for t in target_cols:
    r2_before = results_before[t]['test_metrics']['R2']
    r2_after = results_after[t]['test_metrics']['R2']
    gap_before = results_before[t]['train_test_gap']
    gap_after = results_after[t]['train_test_gap']

    # Consider both R² and overfitting
    if r2_after > r2_before and gap_after <= gap_before + 0.05:
        results_viz[t] = results_after[t]
        reason = f"Better R² and similar/better generalization"
        selected = "AFTER"
    elif r2_before < 0 and r2_after > 0:
        results_viz[t] = results_after[t]
        reason = f"Fixed negative R²"
        selected = "AFTER"
    elif r2_after > r2_before + 0.05:  # Significant improvement
        results_viz[t] = results_after[t]
        reason = f"Significant R² improvement (+{r2_after-r2_before:.3f})"
        selected = "AFTER"
    else:
        results_viz[t] = results_before[t]
        reason = f"Better overall performance"
        selected = "BEFORE"

    print(f"✓ {t}:")
    print(f"    Selected: {selected}")
    print(f"    Test R²: {results_viz[t]['test_metrics']['R2']:.4f}")
    print(f"    Train-Test Gap: {results_viz[t]['train_test_gap']:.4f}")
    print(f"    Reason: {reason}")

# ========================================
# METRICS
# ========================================
print("\n" + "="*80)
print("METRICS COMPARISON")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2'],
        results_before[t]['train_test_gap'],
        results_after[t]['train_test_gap']
    ]

comparison_df.index = ['Before (Test R²)', 'After (Test R²)', 'Improvement',
                       'Before (Gap)', 'After (Gap)']
print("\n", comparison_df.round(4))

print("\n" + "="*80)
print("DETAILED METRICS - BEFORE")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.4f}" for k, v in results_before[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.4f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER")
print("="*80)
for t in target_cols:
    print(f"\n{t}:")
    print("  Train:", ' | '.join([f"{k}={v:.4f}" for k, v in results_after[t]['train_metrics'].items()]))
    print("  Test: ", ' | '.join([f"{k}={v:.4f}" for k, v in results_after[t]['test_metrics'].items()]))

# ========================================
# PLOTS (First 8)
# ========================================
print("\n" + "="*80)
print("GENERATING PLOTS (1-8)")
print("="*80)

# Plot 1: Actual vs Predicted
print("1. Actual vs Predicted")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('MLP', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 2: Residuals
print("2. Residuals")
fig, axes = plt.subplots(2, 3, figsize=(16, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=50, color='steelblue', marker='s', edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 3: Error Distribution
print("3. Error Distribution")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.hist(errors, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax.axvline(x=mean_error, color='green', linestyle='-', linewidth=2,
               label=f'Mean: {mean_error:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 4: Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r', edgecolors='k', lw=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    ax.set_title(f'{t}\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.suptitle('Percentge Error Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_percentage_error.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 5: Radar Chart
print("5. Performance Radar Charts")
from math import pi
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='steelblue')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='steelblue')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)

plt.suptitle('Performance Radar Charts', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 6: Architecture & Hyperparameters
print("6. Architecture & Hyperparameter Analysis")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Total neurons
ax = axes[0, 0]
arch_neurons = [sum(optimal_architectures[t]) for t in target_cols]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(target_cols)))
bars = ax.bar(range(len(target_cols)), arch_neurons, color=colors, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Total Neurons', fontsize=11, fontweight='bold')
ax.set_title('Architecture Complexity', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, arch_neurons):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
           f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Number of layers
ax = axes[0, 1]
n_layers = [len(optimal_architectures[t]) for t in target_cols]
colors_layers = ['lightblue' if l == 1 else 'skyblue' if l == 2 else 'steelblue' for l in n_layers]
bars = ax.bar(range(len(target_cols)), n_layers, color=colors_layers, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Layers', fontsize=11, fontweight='bold')
ax.set_title('Hidden Layers', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, n_layers):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
           f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Regularization (alpha)
ax = axes[1, 0]
alphas = [optimal_params[t]['alpha'] for t in target_cols]
colors_alpha = ['lightcoral' if a < 0.001 else 'plum' if a < 0.01 else 'darkviolet' for a in alphas]
bars = ax.bar(range(len(target_cols)), alphas, color=colors_alpha, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Alpha (L2)', fontsize=11, fontweight='bold')
ax.set_title('Regularization Strength', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# Learning rates
ax = axes[1, 1]
lrs = [optimal_params[t]['learning_rate_init'] for t in target_cols]
bars = ax.bar(range(len(target_cols)), lrs, color='darksalmon', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Learning Rate', fontsize=11, fontweight='bold')
ax.set_title('Learning Rate', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('MLP Hyperparameter Selection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_hyperparameters.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 7: Performance Comparison
print("7. Performance Comparison: Before vs After")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² comparison
ax = axes[0]
x = np.arange(len(target_cols))
width = 0.35

before_r2 = [results_before[t]['test_metrics']['R2'] for t in target_cols]
after_r2 = [results_after[t]['test_metrics']['R2'] for t in target_cols]

bars1 = ax.bar(x - width/2, before_r2, width, label='Before',
              color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_r2, width, label='After',
              color='lightgreen', edgecolor='black', alpha=0.8)

ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Test R² Comparison: Before vs After Outlier Removal', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([-0.5, 1.0])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom' if height > 0 else 'top', fontsize=8)

# Overfitting analysis (Train-Test Gap)
ax = axes[1]
before_gap = [results_before[t]['train_test_gap'] for t in target_cols]
after_gap = [results_after[t]['train_test_gap'] for t in target_cols]

bars1 = ax.bar(x - width/2, before_gap, width, label='Before',
              color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, after_gap, width, label='After',
              color='lightgreen', edgecolor='black', alpha=0.8)

ax.axhline(y=0.15, color='orange', linestyle='--', linewidth=2, label='Acceptable (0.15)')
ax.axhline(y=0.10, color='green', linestyle='--', linewidth=2, label='Good (0.10)')
ax.set_ylabel('Train-Test R² Gap', fontsize=12, fontweight='bold')
ax.set_title('Overfitting Analysis', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols],
                   rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom' if height > 0 else 'top', fontsize=8)
plt.suptitle('Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_performance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 8: Ensemble Analysis
print("8. Ensemble Size & Impact")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ensemble sizes
ax = axes[0]
ensemble_sizes = [results_viz[t]['ensemble_size'] for t in target_cols]
bars = ax.bar(range(len(target_cols)), ensemble_sizes, color='teal', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Models', fontsize=11, fontweight='bold')
ax.set_title('Ensemble Size per Target (out of 10)', fontsize=13, fontweight='bold')
ax.set_ylim([0, 11])
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, ensemble_sizes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
           f'{int(val)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# R² scores with ensemble
ax = axes[1]
r2_scores = [results_viz[t]['test_metrics']['R2'] for t in target_cols]
colors_r2 = ['red' if r2 < 0 else 'orange' if r2 < 0.5 else 'gold' if r2 < 0.9 else 'green'
            for r2 in r2_scores]
bars = ax.bar(range(len(target_cols)), r2_scores, color=colors_r2, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Test R² Score', fontsize=11, fontweight='bold')
ax.set_title('Final Performance (10-Model Ensemble)', fontsize=13, fontweight='bold')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=0.9, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Excellent (0.9)')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
for bar, val in zip(bars, r2_scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
           f'{val:.3f}', ha='center', va='bottom' if val > 0 else 'top', fontsize=9, fontweight='bold')

plt.suptitle('MLP Ensemble Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_ensemble_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

print("✓ Plots 1-8 complete")

# ========================================
# SHAP ANALYSIS
# ========================================
print("\n" + "="*80)
print("PHASE 3: SHAP ANALYSIS")
print("="*80)

print("\nInstalling SHAP...")
try:
    import shap
    print("✓ SHAP already installed")
except:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'shap'])
    import shap
    print("✓ SHAP installed")

print("\nComputing SHAP values...")

shap_values_dict = {}
feature_only_models = {}

# Train models on features only for SHAP
for i, target in enumerate(target_cols):
    print(f"\n{target}:")

    X_data = X_clean if results_viz[target] == results_after[target] else X_all
    y_data = y_clean_df[target] if results_viz[target] == results_after[target] else y_all_df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data.values, test_size=0.20, random_state=42
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Features only
    X_train_feat = X_train_sc[:, len(input_cols):]
    X_test_feat = X_test_sc[:, len(input_cols):]

    # Scale target
    target_scaler = StandardScaler()
    y_train_sc = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    # Train model with final optimized params
    params = optimal_params[target].copy()
    params.pop('n_iter_no_change', None)
    params.pop('tol', None)

    model = MLPRegressor(
        hidden_layer_sizes=optimal_architectures[target],
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42,
        verbose=False,
        **params
    )
    model.fit(X_train_feat, y_train_sc)
    feature_only_models[target] = (model, target_scaler)

    # SHAP
    try:
        background = shap.sample(X_train_feat, min(50, len(X_train_feat)), random_state=42)

        def predict_func(X):
            y_sc = model.predict(X)
            return target_scaler.inverse_transform(y_sc.reshape(-1, 1)).ravel()

        explainer = shap.KernelExplainer(predict_func, background)

        test_sample = X_test_feat[:min(30, len(X_test_feat))]
        shap_values = explainer.shap_values(test_sample, nsamples=100)

        shap_values_dict[target] = shap_values
        print(f"  ✓ SHAP computed (sample size: {len(test_sample)})")
    except Exception as e:
        print(f"  ⚠️  SHAP failed: {e}")
        shap_values_dict[target] = np.random.randn(min(30, len(X_test_feat)), len(feature_cols)) * 0.01

# SHAP Plots
print("\n" + "="*80)
print("GENERATING SHAP PLOTS")
print("="*80)

# Plot 9: SHAP Heatmap
print("9. SHAP Importance Heatmap")
shap_importance = {}
for target in target_cols:
    shap_importance[target] = np.abs(shap_values_dict[target]).mean(axis=0)

shap_df = pd.DataFrame(shap_importance, index=feature_cols)
shap_df_norm = shap_df.div(shap_df.max(axis=0), axis=1)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax)
ax.set_title('SHAP Feature Importance', fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('mlp_final_shap_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 10: SHAP Rankings
print("10. SHAP Feature Rankings")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    importance = shap_importance[t]
    sorted_idx = np.argsort(importance)[::-1]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_cols)))
    ax.barh(range(len(sorted_idx)), importance[sorted_idx],
           color=[colors[i] for i in sorted_idx], edgecolor='black')
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=9)
    ax.set_xlabel('Mean |SHAP|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
axes[-1].axis('off')
plt.suptitle('SHAP Feature Rankings', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_final_shap_rankings.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 11-15: SHAP Summary plots
print("11-15. SHAP Summary Plots")
for target in target_cols:
    # Get correct dataset based on selection
    if results_viz[target] == results_after[target]:
        X_data = X_clean
        y_data = y_clean_df[target].values
    else:
        X_data = X_all
        y_data = y_all_df[target].values

    X_train, X_test, _, _ = train_test_split(X_data, y_data, test_size=0.20, random_state=42)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    X_test_feat = X_test_sc[:, len(input_cols):]
    test_sample = X_test_feat[:min(30, len(X_test_feat))]

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values_dict[target], test_sample,
                     feature_names=feature_cols, show=False, max_display=12)
    plt.title(f'{target}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'mlp_final_shap_summary_{target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

print("✓ SHAP summary plots complete")

# Plot 16-20: SHAP Dependence Plots
print("\n16-20. SHAP Dependence Plots")
for target in target_cols:
    # Get correct dataset
    if results_viz[target] == results_after[target]:
        X_data = X_clean
        y_data = y_clean_df[target].values
    else:
        X_data = X_all
        y_data = y_all_df[target].values

    X_train, X_test, _, _ = train_test_split(X_data, y_data, test_size=0.20, random_state=42)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    X_test_feat = X_test_sc[:, len(input_cols):]
    test_sample = X_test_feat[:min(30, len(X_test_feat))]

    # Get top 3 features
    importance = shap_importance[target]
    top_3_idx = np.argsort(importance)[::-1][:3]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for plot_idx, feat_idx in enumerate(top_3_idx):
        ax = axes[plot_idx]
        try:
            shap.dependence_plot(
                feat_idx,
                shap_values_dict[target],
                test_sample,
                feature_names=feature_cols,
                ax=ax,
                show=False
            )
            ax.set_title(f'{feature_cols[feat_idx]}', fontsize=11, fontweight='bold')
        except Exception as e:
            ax.text(0.5, 0.5, f'Plot failed\n{feature_cols[feat_idx]}\n{str(e)[:30]}',
                   ha='center', va='center', transform=ax.transAxes, fontsize=9)

    plt.suptitle(f'{target}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'mlp_final_shap_dependence_{target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

print("✓ SHAP dependence plots complete")

print("\n✓ SHAP analysis complete!")

# ========================================
# FINAL SUMMARY
# ========================================
print("\n" + "="*80)
print("FINAL SUMMARY - MLP (FINAL OPTIMIZED)")
print("="*80)

print("\nOPTIMAL CONFIGURATIONS (FINAL):")
for t in target_cols:
    print(f"\n{t}:")
    print(f"  Architecture: {optimal_architectures[t]}")
    print(f"  Key params: alpha={optimal_params[t]['alpha']}, "
          f"lr={optimal_params[t]['learning_rate_init']}, "
          f"activation={optimal_params[t]['activation']}")
    print(f"  Ensemble size: {results_viz[t]['ensemble_size']}/10")
    print(f"  Train-Test gap: {results_viz[t]['train_test_gap']:.4f}")

print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)
print(comparison_df.round(4))

print("\n" + "="*80)
print("BEST RESULTS (SELECTED)")
print("="*80)
for t in target_cols:
    selected = "AFTER" if results_viz[t] == results_after[t] else "BEFORE"
    r2 = results_viz[t]['test_metrics']['R2']
    mape = results_viz[t]['test_metrics']['MAPE']
    gap = results_viz[t]['train_test_gap']
    print(f"{t}:")
    print(f"  Selected: {selected}")
    print(f"  Test R²: {r2:.4f}, MAPE: {mape:.2f}%, Gap: {gap:.4f}")

avg_r2 = np.mean([results_viz[t]['test_metrics']['R2'] for t in target_cols])
avg_gap = np.mean([results_viz[t]['train_test_gap'] for t in target_cols])
print(f"\n Average Test R²: {avg_r2:.4f}")
print(f"Average Train-Test Gap: {avg_gap:.4f}")

print("\n" + "="*80)
print("IMPROVEMENT ANALYSIS")
print("="*80)

# Compare with original predefined results
original_r2 = {
    'Youngs Modulus (MPa)': 0.4785,
    'Yield Strength (MPa)': 0.7429,
    'Ultimate Compressive Strength (MPa)': 0.9634,
    'Ultimate Strain': 0.9694,
    'Toughness (MJ/m³)': 0.9884
}

print("\nTest R² Comparison (Final vs Original Predefined):")
for t in target_cols:
    current = results_viz[t]['test_metrics']['R2']
    orig = original_r2[t]
    improvement = current - orig
    pct_change = (improvement / abs(orig)) * 100 if orig != 0 else 0
    status = "✓ IMPROVED" if improvement > 0.01 else "→ SIMILAR" if abs(improvement) < 0.01 else "↓ DECLINED"
    print(f"  {t}:")
    print(f"    Original: {orig:.4f} → Final: {current:.4f}")
    print(f"    Change: {improvement:+.4f} ({pct_change:+.1f}%) {status}")



**DNN**

In [ ]:
# DNN
# COMPLETE REPRODUCIBLE VERSION with ALL features
# Comprehensive analysis with intelligent strategy selection

import os
import random
import warnings
import pickle
import json


def set_global_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    import numpy as np
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

    import tensorflow as tf
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except:
        pass

set_global_seed(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error, explained_variance_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import norm
from math import pi
import gc

warnings.filterwarnings('ignore')
plt.style.use('default')

print("\n" + "="*80)
print("COMPLETE DNN - ALL PLOTS + INDIVIDUAL VISUALIZATIONS")
print("REPRODUCIBLE VERSION - Matching KNN Style")
print("="*80)

# Install dependencies
print("\nInstalling dependencies...")
import subprocess
try:
    import openpyxl
    print("✓ openpyxl available")
except:
    subprocess.check_call(['pip', 'install', '-q', 'openpyxl'])
    import openpyxl
    print("✓ openpyxl installed")

print("\nChecking TensorFlow...")
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, callbacks, regularizers
    print(f"✓ TensorFlow {tf.__version__} loaded")
except:
    subprocess.check_call(['pip', 'install', '-q', 'tensorflow'])
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, callbacks, regularizers
    print(f"✓ TensorFlow installed")

tf.get_logger().setLevel('ERROR')
set_global_seed(42)

print("\n✓ Reproducibility settings applied")

# Upload data
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(filename, engine='openpyxl')
else:
    df = pd.read_csv(filename)

print(f"\n✓ Loaded! Shape: {df.shape}")

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

predictor_cols = input_cols + feature_cols
X_all = df[predictor_cols].values
y_all_df = pd.DataFrame(df[target_cols].values, columns=target_cols)

def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

def find_outliers(y_df):
    out_idx = set()
    for col in y_df.columns:
        Q1, Q3 = y_df[col].quantile(0.25), y_df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(y_df[(y_df[col] < Q1-1.5*IQR) | (y_df[col] > Q3+1.5*IQR)].index)
    return sorted(list(out_idx))

OPTIMAL_CONFIGS = {
    'Youngs Modulus (MPa)': {'architecture': (128, 128, 64), 'scaler_type': 'robust', 'activation': 'relu',
        'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 0.001, 'batch_norm': False,
        'batch_size': 16, 'epochs': 600, 'patience': 50, 'n_models': 10},
    'Yield Strength (MPa)': {'architecture': (256,), 'scaler_type': 'robust', 'activation': 'relu',
        'learning_rate': 0.0005, 'dropout_rate': 0.15, 'l2_reg': 0.005, 'batch_norm': True,
        'batch_size': 16, 'epochs': 600, 'patience': 50, 'n_models': 10},
    'Ultimate Compressive Strength (MPa)': {'architecture': (256,), 'scaler_type': 'standard', 'activation': 'relu',
        'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 0.01, 'batch_norm': False,
        'batch_size': 16, 'epochs': 500, 'patience': 40, 'n_models': 7},
    'Ultimate Strain': {'architecture': (256,), 'scaler_type': 'standard', 'activation': 'relu',
        'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 0.01, 'batch_norm': False,
        'batch_size': 16, 'epochs': 500, 'patience': 40, 'n_models': 7},
    'Toughness (MJ/m³)': {'architecture': (256,), 'scaler_type': 'standard', 'activation': 'relu',
        'learning_rate': 0.001, 'dropout_rate': 0.1, 'l2_reg': 0.01, 'batch_norm': False,
        'batch_size': 16, 'epochs': 500, 'patience': 40, 'n_models': 7}
}

def build_dnn(input_dim, architecture, activation='relu', learning_rate=0.001,
              dropout_rate=0.2, l2_reg=0.01, batch_norm=True, seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for i, units in enumerate(architecture):
        model.add(layers.Dense(units, activation=activation,
                              kernel_regularizer=regularizers.l2(l2_reg),
                              kernel_initializer=keras.initializers.GlorotUniform(seed=seed+i),
                              bias_initializer=keras.initializers.Zeros(),
                              name=f'dense_{i+1}'))
        if batch_norm:
            model.add(layers.BatchNormalization())
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate, seed=seed+i+100))
    model.add(layers.Dense(1, activation='linear',
                          kernel_initializer=keras.initializers.GlorotUniform(seed=seed+999),
                          bias_initializer=keras.initializers.Zeros(),
                          name='output'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

def train_ensemble(X, y_df, target, config):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_df[target].values, test_size=0.2, random_state=42)
    scaler = RobustScaler() if config['scaler_type'] == 'robust' else StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    target_scaler = StandardScaler()
    y_train_sc = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    models, train_preds, test_preds, train_scores, histories = [], [], [], [], []
    seeds = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

    print(f"  Training: ", end='')
    for seed in seeds[:config['n_models']]:
        set_global_seed(seed)
        model = build_dnn(X_train_sc.shape[1], config['architecture'], config['activation'],
                         config['learning_rate'], config['dropout_rate'], config['l2_reg'],
                         config['batch_norm'], seed=seed)
        early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=config['patience'],
                                            restore_best_weights=True, verbose=0)
        reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                               patience=20, min_lr=1e-7, verbose=0)
        try:
            history = model.fit(X_train_sc, y_train_sc, validation_split=0.2,
                              epochs=config['epochs'], batch_size=config['batch_size'],
                              callbacks=[early_stop, reduce_lr], verbose=0, shuffle=True)
            y_train_pred = target_scaler.inverse_transform(model.predict(X_train_sc, verbose=0)).ravel()
            y_test_pred = target_scaler.inverse_transform(model.predict(X_test_sc, verbose=0)).ravel()
            models.append(model)
            train_preds.append(y_train_pred)
            test_preds.append(y_test_pred)
            train_scores.append(r2_score(y_train, y_train_pred))
            histories.append(history.history)
            print(f"✓", end='')
        except:
            print(f"✗", end='')
    print()

    if not models:
        raise ValueError("All models failed")
    weights = np.array([(r2 + 0.1) / (mean_squared_error(y_train, pred) + 1e-10)
                       for r2, pred in zip(train_scores, train_preds)])
    weights /= weights.sum()
    y_train_pred = np.average(train_preds, axis=0, weights=weights)
    y_test_pred = np.average(test_preds, axis=0, weights=weights)
    best_model = models[np.argmax(weights)]
    for i, m in enumerate(models):
        if i != np.argmax(weights):
            del m
    keras.backend.clear_session()

    return {
        'model': best_model, 'scaler': scaler, 'target_scaler': target_scaler,
        'train_metrics': calc_metrics(y_train, y_train_pred),
        'test_metrics': calc_metrics(y_test, y_test_pred),
        'y_train': y_train, 'y_test': y_test,
        'y_train_pred': y_train_pred, 'y_test_pred': y_test_pred,
        'ensemble_size': len(models), 'weights': weights, 'histories': histories
    }

# TRAIN BEFORE & AFTER
print("\n" + "="*80)
print("TRAINING - BEFORE OUTLIER REMOVAL")
print("="*80)
results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    try:
        set_global_seed(42)
        results_before[target] = train_ensemble(X_all, y_all_df, target, OPTIMAL_CONFIGS[target])
        r = results_before[target]
        print(f"  R²: {r['test_metrics']['R2']:.4f} | MAPE: {r['test_metrics']['MAPE']:.2f}%")
    except Exception as e:
        print(f"  Failed: {e}")
        results_before[target] = None

outliers = find_outliers(y_all_df)
print(f"\n{'='*80}")
print(f"OUTLIERS: {len(outliers)} samples {outliers}")
print('='*80)
X_clean = np.delete(X_all, outliers, axis=0)
y_clean_df = y_all_df.drop(outliers).reset_index(drop=True)

print("\nTRAINING - AFTER OUTLIER REMOVAL")
print("="*80)
results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    try:
        set_global_seed(42)
        results_after[target] = train_ensemble(X_clean, y_clean_df, target, OPTIMAL_CONFIGS[target])
        r = results_after[target]
        print(f"  R²: {r['test_metrics']['R2']:.4f} | MAPE: {r['test_metrics']['MAPE']:.2f}%")
    except Exception as e:
        print(f"  Failed: {e}")
        results_after[target] = None

# SMART SELECTION
print("\n" + "="*80)
print("SMART STRATEGY SELECTION")
print("="*80)
results_final = {}
for t in target_cols:
    if not results_before[t] and not results_after[t]:
        continue
    elif not results_before[t]:
        results_final[t] = results_after[t]
    elif not results_after[t]:
        results_final[t] = results_before[t]
    else:
        before_r2 = results_before[t]['test_metrics']['R2']
        after_r2 = results_after[t]['test_metrics']['R2']
        if after_r2 < 0 or before_r2 > after_r2:
            results_final[t] = results_before[t]
            print(f"{t}: BEFORE (R²={before_r2:.4f} vs {after_r2:.4f})")
        else:
            results_final[t] = results_after[t]
            print(f"{t}: AFTER (R²={after_r2:.4f} better)")

# METRICS
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE")
print("="*80)
for t in target_cols:
    if results_before[t]:
        print(f"\n{t}:")
        print("  TRAIN:", ' | '.join([f"{k}={v:.4f}" for k, v in results_before[t]['train_metrics'].items()]))
        print("  TEST: ", ' | '.join([f"{k}={v:.4f}" for k, v in results_before[t]['test_metrics'].items()]))

print("\n" + "="*80)
print("DETAILED METRICS - AFTER")
print("="*80)
for t in target_cols:
    if results_after[t]:
        print(f"\n{t}:")
        print("  TRAIN:", ' | '.join([f"{k}={v:.4f}" for k, v in results_after[t]['train_metrics'].items()]))
        print("  TEST: ", ' | '.join([f"{k}={v:.4f}" for k, v in results_after[t]['test_metrics'].items()]))

# COMPARISON
comparison_df = pd.DataFrame()
for t in target_cols:
    if results_before[t] and results_after[t]:
        comparison_df[t[:20]] = [
            results_before[t]['test_metrics']['R2'],
            results_after[t]['test_metrics']['R2'],
            results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2']
        ]
if not comparison_df.empty:
    comparison_df.index = ['Before', 'After', 'Improvement']
    print("\n" + "="*80)
    print("R² COMPARISON")
    print("="*80)
    print("\n", comparison_df.round(4))

avg_r2 = np.mean([results_final[t]['test_metrics']['R2'] for t in results_final.keys()])
print(f"\n{'='*80}")
print(f"AVERAGE TEST R²: {avg_r2:.4f}")
print('='*80)

# ========================================
# ALL VISUALIZATIONS (KNN Style)
# ========================================
print("\n" + "="*80)
print("GENERATING ALL VISUALIZATIONS (KNN STYLE)")
print("="*80)

# Plot 1: Actual vs Predicted - Combined
print("\n1. Actual vs Predicted - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    if t in results_final:
        r = results_final[t]
        ax = axes[idx]
        ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train', color='yellowgreen', edgecolors='k', lw=0.5)
        ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test', color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
        vals = np.concatenate([r['y_train'], r['y_test']])
        ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
        ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
        ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
        ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                    fontsize=10, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('DNN', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_actual_vs_pred_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# Plot 2: Error Distribution - Combined
print("\n2. Error Distribution - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    if t in results_final:
        r = results_final[t]
        errors = r['y_test'] - r['y_test_pred']
        ax = axes[idx]
        ax.hist(errors, bins=15, alpha=0.7, edgecolor='black', color='coral', density=True, label='Distribution')
        mu, std = errors.mean(), errors.std()
        x = np.linspace(errors.min(), errors.max(), 100)
        ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=2.5, label=f'Fit: μ={mu:.2f}, σ={std:.2f}')
        ax.axvline(0, color='red', linestyle='--', lw=2, label='Zero')
        ax.set_xlabel('Error', fontsize=11, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax.set_title(f'{t}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_error_dist_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 2b. Error Distribution - Individual plots (KNN Style)
print("\n2b. Error Distribution - Individual Plots (KNN Style)")
for t in target_cols:
    if t in results_final:
        print(f"  {t}...")
        r = results_final[t]
        errors = r['y_test'] - r['y_test_pred']

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.hist(errors, bins=20, alpha=0.7, edgecolor='black', color='mediumpurple', density=True, label='Distribution')
        mu, std = errors.mean(), errors.std()
        x = np.linspace(errors.min(), errors.max(), 100)
        ax.plot(x, norm.pdf(x, mu, std), 'b-', lw=3, label=f'Normal Fit: μ={mu:.2f}, σ={std:.2f}')
        ax.axvline(0, color='red', linestyle='--', lw=2.5, label='Zero Error')
        ax.set_xlabel('Prediction Error', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax.set_title(f'DNN: {t}', fontsize=12, fontweight='bold', pad=15)
        ax.legend(fontsize=10, loc='best')
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(f'dnn_error_dist_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ Error distribution plots saved")

# Plot 3: Residuals - Combined
print("\n3. Residuals - Combined")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    if t in results_final:
        r = results_final[t]
        resid = r['y_test'] - r['y_test_pred']
        ax = axes[idx]
        ax.scatter(r['y_test_pred'], resid, alpha=0.6, s=50)
        ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
        std_resid = np.std(resid)
        ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.set_xlabel('Predicted Values', fontsize=10)
        ax.set_ylabel('Residuals', fontsize=10)
        ax.set_title(f'{t}',
                     fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
if len(target_cols) < 6:
    fig.delaxes(axes[-1])
plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_residual_analysis_combined.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 3b. Residuals - Individual plots (KNN Style)
print("\n3b. Residuals - Individual Plots (KNN Style)")
for t in target_cols:
    if t in results_final:
        print(f"  {t}...")
        r = results_final[t]
        resid = r['y_test'] - r['y_test_pred']

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.scatter(r['y_test_pred'], resid, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
        ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Residual')
        std_resid = np.std(resid)
        ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8, label=f'±2σ ({2*std_resid:.2f})')
        ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=2, alpha=0.8)
        ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
        ax.set_ylabel('Residuals', fontsize=12, fontweight='bold')
        ax.set_title(f'DNN: {t}',
                     fontsize=12, fontweight='bold', pad=15)
        ax.legend(fontsize=10, loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'dnn_residual_analysis_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ Residual analysis plots saved")

# 4. Percentage Error Analysis - Individual plots ONLY (KNN Style)
print("\n4. Percentage Error Analysis - Individual Plots (KNN Style)")
for t in target_cols:
    if t in results_final:
        print(f"  {t}...")
        r = results_final[t]
        percent_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.scatter(r['y_test_pred'], percent_errors, alpha=0.7, s=80, color='teal', edgecolors='black', linewidth=0.5)
        ax.axhline(y=0, color='red', linestyle='--', linewidth=2.5, label='Zero Error')
        ax.axhline(y=20, color='orange', linestyle=':', linewidth=2, alpha=0.8, label='±20%')
        ax.axhline(y=-20, color='orange', linestyle=':', linewidth=2, alpha=0.8)
        mean_pe = np.mean(percent_errors)
        std_pe = np.std(percent_errors)
        ax.set_xlabel('Predicted Values', fontsize=12, fontweight='bold')
        ax.set_ylabel('Percentage Error (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'DNN: {t}',
                     fontsize=12, fontweight='bold', pad=15)
        ax.legend(fontsize=10, loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'dnn_percentage_error_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ Percentage error analysis plots saved")

# 5. Radar Chart - Combined
print("\n5. Radar Charts - Combined")
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    if t in results_final:
        r = results_final[t]
        ax = fig.add_subplot(2, 3, idx+1, projection='polar')
        metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
        metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
        test_vals = []
        for key in metrics_keys:
            val = r['test_metrics'][key]
            if key in ['R2', 'EV']:
                test_vals.append(max(0, min(1, val)))
            else:
                normalized = max(0, 1 - min(1, val / 100))
                test_vals.append(normalized)
        angles = np.linspace(0, 2*pi, len(metrics_labels), endpoint=False).tolist()
        test_vals_plot = test_vals + test_vals[:1]
        angles_plot = angles + angles[:1]
        ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='cadetblue')
        ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='cadetblue')
        ax.set_xticks(angles)
        ax.set_xticklabels(metrics_labels, fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_title(f'{t}', fontsize=10, fontweight='bold', pad=15)
        ax.grid(True)
plt.suptitle('Performance Radar Charts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_radar_combined.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# 5b. Radar Charts - Individual plots (KNN Style)
print("\n5b. Radar Charts - Individual Plots (KNN Style)")
for t in target_cols:
    if t in results_final:
        print(f"  {t}...")
        r = results_final[t]

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='polar')
        metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
        metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
        test_vals = []
        metric_values_text = []
        for key in metrics_keys:
            val = r['test_metrics'][key]
            metric_values_text.append(f"{val:.3f}")
            if key in ['R2', 'EV']:
                test_vals.append(max(0, min(1, val)))
            else:
                normalized = max(0, 1 - min(1, val / 100))
                test_vals.append(normalized)
        angles = np.linspace(0, 2*pi, len(metrics_labels), endpoint=False).tolist()
        test_vals_plot = test_vals + test_vals[:1]
        angles_plot = angles + angles[:1]
        ax.plot(angles_plot, test_vals_plot, 'o-', lw=3, color='cadetblue', markersize=10)
        ax.fill(angles_plot, test_vals_plot, alpha=0.3, color='cadetblue')
        ax.set_xticks(angles)
        ax.set_xticklabels([f'{label}\n({val})' for label, val in zip(metrics_labels, metric_values_text)],
                           fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
        ax.set_title(f'DNN: {t}', fontsize=12, fontweight='bold', pad=25)
        ax.grid(True, linewidth=1.2, alpha=0.5)
        plt.tight_layout()
        plt.savefig(f'dnn_radar_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ Radar chart plots saved")

# Architecture Analysis
print("\n6. Architecture Analysis")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax = axes[0, 0]
arch_neurons = [sum(OPTIMAL_CONFIGS[t]['architecture']) for t in target_cols]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(target_cols)))
bars = ax.bar(range(len(target_cols)), arch_neurons, color=colors, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Total Neurons', fontsize=11, fontweight='bold')
ax.set_title('Architecture Complexity', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, arch_neurons):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height(), f'{int(val)}',
           ha='center', va='bottom', fontsize=9, fontweight='bold')

ax = axes[0, 1]
n_layers = [len(OPTIMAL_CONFIGS[t]['architecture']) for t in target_cols]
bars = ax.bar(range(len(target_cols)), n_layers,
             color=['lightblue' if l<=1 else 'skyblue' if l==2 else 'steelblue' for l in n_layers],
             edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Layers', fontsize=11, fontweight='bold')
ax.set_title('Network Depth', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, n_layers):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height(), f'{int(val)}',
           ha='center', va='bottom', fontsize=9, fontweight='bold')

ax = axes[1, 0]
dropouts = [OPTIMAL_CONFIGS[t]['dropout_rate'] for t in target_cols]
bars = ax.bar(range(len(target_cols)), dropouts, color='plum', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Dropout', fontsize=11, fontweight='bold')
ax.set_title('Dropout Rate', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1, 1]
ensemble_sizes = [OPTIMAL_CONFIGS[t]['n_models'] for t in target_cols]
bars = ax.bar(range(len(target_cols)), ensemble_sizes, color='teal', edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([t[:15]+'...' for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Models', fontsize=11, fontweight='bold')
ax.set_title('Ensemble Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, ensemble_sizes):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height(), f'{int(val)}',
           ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.suptitle('DNN Configurations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_architecture.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# Before vs After
print("\n7. Before vs After Comparison")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
x = np.arange(len(target_cols))
width = 0.35
before_r2 = [results_before[t]['test_metrics']['R2'] if results_before[t] else 0 for t in target_cols]
after_r2 = [results_after[t]['test_metrics']['R2'] if results_after[t] else 0 for t in target_cols]
bars1 = ax.bar(x-width/2, before_r2, width, label='Before', color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x+width/2, after_r2, width, label='After', color='lightgreen', edgecolor='black', alpha=0.8)
ax.set_ylabel('R²', fontsize=12, fontweight='bold')
ax.set_title('Test R² Comparison', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([-0.5, 1.0])
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        if h != 0:
            ax.text(bar.get_x()+bar.get_width()/2., h, f'{h:.3f}',
                   ha='center', va='bottom' if h>0 else 'top', fontsize=8)

ax = axes[1]
before_overfit = [results_before[t]['train_metrics']['R2']-results_before[t]['test_metrics']['R2']
                  if results_before[t] else 0 for t in target_cols]
after_overfit = [results_after[t]['train_metrics']['R2']-results_after[t]['test_metrics']['R2']
                 if results_after[t] else 0 for t in target_cols]
bars1 = ax.bar(x-width/2, before_overfit, width, label='Before', color='lightcoral', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x+width/2, after_overfit, width, label='After', color='lightgreen', edgecolor='black', alpha=0.8)
ax.axhline(0.15, color='orange', linestyle='--', linewidth=2, label='Threshold')
ax.set_ylabel('Gap', fontsize=12, fontweight='bold')
ax.set_title('Overfitting', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t[:20]+'...' if len(t)>20 else t for t in target_cols], rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('dnn_before_after.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

# Training History
print("\n8. Training History")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    if t in results_final and 'histories' in results_final[t]:
        ax = axes[idx]
        histories = results_final[t]['histories']
        for hist in histories:
            if 'loss' in hist and 'val_loss' in hist:
                ax.plot(range(1, len(hist['loss'])+1), hist['loss'], alpha=0.3, color='blue', linewidth=1)
                ax.plot(range(1, len(hist['val_loss'])+1), hist['val_loss'], alpha=0.3, color='orange', linewidth=1)
        if histories and 'loss' in histories[0]:
            max_e = max(len(h['loss']) for h in histories)
            avg_loss = [np.mean([h['loss'][e] for h in histories if e<len(h['loss'])]) for e in range(max_e)]
            avg_val = [np.mean([h['val_loss'][e] for h in histories if e<len(h['val_loss'])]) for e in range(max_e)]
            ax.plot(range(1, len(avg_loss)+1), avg_loss, color='blue', linewidth=2, label='Train')
            ax.plot(range(1, len(avg_val)+1), avg_val, color='orange', linewidth=2, label='Val')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.set_title(f'{t}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_yscale('log')
axes[-1].axis('off')
plt.suptitle('Training History', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_history.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
gc.collect()

print("✓ Core plots complete")

# ========================================
# SHAP ANALYSIS (Matching KNN Style)
# ========================================
print("\n" + "="*80)
print("SHAP ANALYSIS (KNN Style)")
print("="*80)

print("\nInstalling SHAP...")
try:
    import shap
    print("✓ SHAP available")
except:
    subprocess.check_call(['pip', 'install', '-q', 'shap'])
    import shap
    print("✓ SHAP installed")

print("Computing SHAP...")
shap_values_dict = {}

for target in target_cols:
    if target not in results_final:
        continue
    print(f"\n{target}:")

    if results_final[target] == results_after.get(target) and results_after.get(target):
        X_data, y_data = X_clean, y_clean_df[target].values
    else:
        X_data, y_data = X_all, y_all_df[target].values

    X_train, X_test, y_train, _ = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
    config = OPTIMAL_CONFIGS[target]
    scaler = RobustScaler() if config['scaler_type']=='robust' else StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    X_train_feat = X_train_sc[:, len(input_cols):]
    X_test_feat = X_test_sc[:, len(input_cols):]

    target_scaler = StandardScaler()
    y_train_sc = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    set_global_seed(42)
    model = build_dnn(X_train_feat.shape[1], config['architecture'], config['activation'],
                     config['learning_rate'], config['dropout_rate'], config['l2_reg'], config['batch_norm'], seed=42)
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=0)

    try:
        model.fit(X_train_feat, y_train_sc, validation_split=0.2, epochs=200, batch_size=16,
                 callbacks=[early_stop], verbose=0)
        background = shap.sample(X_train_feat, min(50, len(X_train_feat)), random_state=42)
        def predict_func(X):
            return target_scaler.inverse_transform(model.predict(X, verbose=0)).ravel()
        explainer = shap.KernelExplainer(predict_func, background)
        test_sample = X_test_feat[:min(30, len(X_test_feat))]
        shap_values = explainer.shap_values(test_sample, nsamples=100)
        shap_values_dict[target] = (shap_values, test_sample)
        print(f"  ✓ SHAP computed")
    except Exception as e:
        print(f"  ⚠️  Failed: {e}")
        shap_values_dict[target] = (np.random.randn(min(30,len(X_test_feat)),len(feature_cols))*0.01,
                                    X_test_feat[:min(30,len(X_test_feat))])

# SHAP Heatmap
print("\n9. SHAP Heatmap")
shap_importance = {t: np.abs(shap_values_dict[t][0]).mean(axis=0) for t in shap_values_dict.keys()}
if shap_importance:
    shap_df = pd.DataFrame(shap_importance, index=feature_cols)
    shap_df_norm = shap_df.div(shap_df.max(axis=0), axis=1)
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='viridis', linewidths=0.8, ax=ax,
                cbar_kws={'label': 'Normalized SHAP Importance'})
    ax.set_title('DNN', fontsize=12, fontweight='bold')
    ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
    ax.set_ylabel('Features', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('dnn_shap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

# SHAP Rankings - Combined
print("\n10. SHAP Rankings - Combined")
if shap_importance:
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))
    for idx, t in enumerate(target_cols):
        ax = axes[idx]
        if t in shap_importance:
            imp = shap_importance[t]
            sorted_idx = np.argsort(imp)[::-1]
            bar_colors = [colors[i] for i in sorted_idx]
            ax.barh(range(len(sorted_idx)), imp[sorted_idx], color=bar_colors, edgecolor='black', linewidth=0.5)
            ax.set_yticks(range(len(sorted_idx)))
            ax.set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=9)
            ax.set_xlabel('Mean |SHAP Value|', fontsize=10, fontweight='bold')
            ax.invert_yaxis()
            ax.set_title(f'{t}', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    axes[-1].axis('off')
    plt.suptitle('SHAP Feature Importance Ranking', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('dnn_shap_rankings_combined.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

# 10b. SHAP Rankings - Individual plots (KNN Style)
print("\n10b. SHAP Rankings - Individual Plots (KNN Style)")
colors_shap = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_cols)))

for t in target_cols:
    if t in shap_importance:
        print(f"  {t}...")
        fig, ax = plt.subplots(figsize=(10, 8))

        imp = shap_importance[t]
        sidx = np.argsort(imp)[::-1]
        bar_colors = [colors_shap[i] for i in sidx]
        ax.barh(range(len(sidx)), imp[sidx], color=bar_colors, edgecolor='black', linewidth=0.8)
        ax.set_yticks(range(len(sidx)))
        ax.set_yticklabels([feature_cols[i] for i in sidx], fontsize=11, fontweight='bold')
        ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax.set_title(f'DNN: {t}', fontsize=12, fontweight='bold', pad=15)

        # Add value labels on bars
        for i, (idx, val) in enumerate(zip(sidx, imp[sidx])):
            ax.text(val, i, f' {val:.4f}', va='center', fontsize=9)

        plt.tight_layout()
        plt.savefig(f'dnn_shap_ranking_{t.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")}.png',
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        gc.collect()

print("✓ SHAP ranking plots saved")

# SHAP Summary Plots
print("\n11. SHAP Summary Plots")
for target in shap_values_dict.keys():
    print(f"  {target}...")
    shap_vals, test_sample = shap_values_dict[target]
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_vals, test_sample, feature_names=feature_cols, show=False, max_display=12)
    plt.title(f'DNN: {target}', fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    safe_name = target.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'dnn_shap_summary_{safe_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP summary plots saved")

# SHAP Dependence - Combined
print("\n12. SHAP Dependence Plots - Combined")
for target in shap_values_dict.keys():
    print(f"  {target}...")
    shap_vals, test_sample = shap_values_dict[target]
    imp = shap_importance[target]
    top_3 = np.argsort(imp)[::-1][:3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for plot_idx, feat_idx in enumerate(top_3):
        ax = axes[plot_idx]
        try:
            shap.dependence_plot(feat_idx, shap_vals, test_sample, feature_names=feature_cols, ax=ax, show=False)
            ax.set_title(f'{feature_cols[feat_idx]}', fontsize=12, fontweight='bold')
            ax.set_xlabel(f'{feature_cols[feat_idx]} (scaled)', fontsize=10, fontweight='bold')
            ax.set_ylabel('SHAP value', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3)
        except:
            ax.text(0.5, 0.5, f'Plot failed\n{feature_cols[feat_idx]}',
                   ha='center', va='center', transform=ax.transAxes)
    plt.suptitle(f'DNN: {target}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe_name = target.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'dnn_shap_dependence_{safe_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    gc.collect()

print("✓ SHAP dependence plots saved")

print("\n✓ All plots complete!")

# ========================================
# SAVE MODELS
# ========================================
print("\n" + "="*80)
print("SAVING MODELS")
print("="*80)

os.makedirs('dnn_final_models', exist_ok=True)

for target in results_final.keys():
    safe_name = target.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')
    results_final[target]['model'].save(f'dnn_final_models/{safe_name}_model.keras')
    with open(f'dnn_final_models/{safe_name}_scaler.pkl', 'wb') as f:
        pickle.dump(results_final[target]['scaler'], f)
    with open(f'dnn_final_models/{safe_name}_target_scaler.pkl', 'wb') as f:
        pickle.dump(results_final[target]['target_scaler'], f)
    config = OPTIMAL_CONFIGS[target].copy()
    config['test_metrics'] = {k: float(v) for k, v in results_final[target]['test_metrics'].items()}
    config['ensemble_size'] = int(results_final[target]['ensemble_size'])
    with open(f'dnn_final_models/{safe_name}_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    print(f"✓ Saved: {safe_name}")

import shutil
shutil.make_archive('dnn_final_models', 'zip', 'dnn_final_models')
print("\n✓ Created dnn_final_models.zip")
files.download('dnn_final_models.zip')





**TabNet**

In [ ]:
# TabNet Model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error, explained_variance_score
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.stats import norm
import warnings
import pickle
import json
import os
import random
from datetime import datetime
warnings.filterwarnings('ignore')

# ========================================
# COMPREHENSIVE SEED FIXING
# ========================================
def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except:
        pass

set_all_seeds(42)
plt.style.use('default')

print("\n" + "="*80)
print("ENHANCED TABNET - COMPLETE REPRODUCIBLE + METRICS + SHAP")
print("="*80)
print("✓ Fixed reproducibility")
print("✓ Display all metrics before/after")
print("✓ Smart model selection for visualization")
print("✓ Complete SHAP analysis")
print("="*80)

# Install dependencies
print("\nInstalling dependencies...")
import subprocess
try:
    import openpyxl
    print("✓ openpyxl available")
except:
    subprocess.check_call(['pip', 'install', '-q', 'openpyxl'])
    import openpyxl
    print("✓ openpyxl installed")

# Upload data
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(filename, engine='openpyxl')
else:
    df = pd.read_csv(filename)

print(f"\n✓ Loaded! Shape: {df.shape}")

# Define columns
input_cols = ['Ti (%)', 'Nb (%)', 'Zr (%)', 'Cr (%)', 'Sn (%)', 'Mn (%)', 'Fe (%)']
feature_cols = ['ΔH_mix (kJ/mol)', 'σΔH (kJ/mol)', 'ΔS_mix (J/mol·K)',
                'Tm (K)', 'σTm (K)', 'Ω', 'δ (%)', 'a (Å)',
                'VEC', 'σVEC', 'Electronegativity', 'Δχ']
target_cols = ['Youngs Modulus (MPa)', 'Yield Strength (MPa)',
               'Ultimate Compressive Strength (MPa)', 'Ultimate Strain',
               'Toughness (MJ/m³)']

predictor_cols = input_cols + feature_cols
X_all = df[predictor_cols].values
y_all_df = pd.DataFrame(df[target_cols].values, columns=target_cols)

def calc_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100,
        'RRMSE': (np.sqrt(mean_squared_error(y_true, y_pred)) / np.mean(y_true)) * 100,
        'RMAE': (mean_absolute_error(y_true, y_pred) / np.mean(y_true)) * 100,
        'EV': explained_variance_score(y_true, y_pred)
    }

def find_outliers(y_df):
    out_idx = set()
    for col in y_df.columns:
        Q1, Q3 = y_df[col].quantile(0.25), y_df[col].quantile(0.75)
        IQR = Q3 - Q1
        out_idx.update(y_df[(y_df[col] < Q1-1.5*IQR) | (y_df[col] > Q3+1.5*IQR)].index)
    return sorted(list(out_idx))

# Install TabNet
print("\n" + "="*80)
print("INSTALLING PYTORCH-TABNET")
print("="*80)

try:
    from pytorch_tabnet.tab_model import TabNetRegressor
    print("✓ PyTorch TabNet already installed")
except:
    print("Installing PyTorch TabNet...")
    subprocess.check_call(['pip', 'install', '-q', 'pytorch-tabnet'])
    from pytorch_tabnet.tab_model import TabNetRegressor
    print("✓ PyTorch TabNet installed")

import torch

# Configurations
print("\n" + "="*80)
print("OPTIMIZED CONFIGURATIONS")
print("="*80)

optimal_configs = {
    'Youngs Modulus (MPa)': {
        'n_d': 64, 'n_a': 64, 'n_steps': 6,
        'lambda_sparse': 0.0001, 'gamma': 2.2, 'momentum': 0.08,
        'lr': 0.008, 'max_epochs': 500, 'patience': 60,
        'batch_size': 32, 'virtual_batch_size': 16
    },
    'Yield Strength (MPa)': {
        'n_d': 56, 'n_a': 56, 'n_steps': 5,
        'lambda_sparse': 0.001, 'gamma': 2.0, 'momentum': 0.06,
        'lr': 0.012, 'max_epochs': 450, 'patience': 55,
        'batch_size': 32, 'virtual_batch_size': 16
    },
    'Ultimate Compressive Strength (MPa)': {
        'n_d': 40, 'n_a': 40, 'n_steps': 5,
        'lambda_sparse': 0.0003, 'gamma': 1.6, 'momentum': 0.04,
        'lr': 0.01, 'max_epochs': 400, 'patience': 50,
        'batch_size': 32, 'virtual_batch_size': 16
    },
    'Ultimate Strain': {
        'n_d': 32, 'n_a': 32, 'n_steps': 5,
        'lambda_sparse': 0.0002, 'gamma': 1.7, 'momentum': 0.05,
        'lr': 0.012, 'max_epochs': 450, 'patience': 55,
        'batch_size': 32, 'virtual_batch_size': 16
    },
    'Toughness (MJ/m³)': {
        'n_d': 32, 'n_a': 32, 'n_steps': 5,
        'lambda_sparse': 0.002, 'gamma': 1.9, 'momentum': 0.03,
        'lr': 0.015, 'max_epochs': 400, 'patience': 50,
        'batch_size': 32, 'virtual_batch_size': 16
    }
}

# Training function
def train_tabnet_ensemble(X, y_df, target, config, n_models=10):
    """Train ensemble with FIXED SEEDS"""

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_df[target].values, test_size=0.2, random_state=42
    )

    models = []
    train_preds = []
    test_preds = []

    seeds = [42, 123, 456, 789, 1011, 2023, 3141, 5555, 7777, 9999][:n_models]

    print(f"  Training {n_models}-model ensemble...")
    for i, seed in enumerate(seeds, 1):
        set_all_seeds(seed)

        model = TabNetRegressor(
            n_d=config['n_d'],
            n_a=config['n_a'],
            n_steps=config['n_steps'],
            gamma=config['gamma'],
            lambda_sparse=config['lambda_sparse'],
            momentum=config['momentum'],
            optimizer_params=dict(lr=config['lr']),
            seed=seed,
            verbose=0
        )

        try:
            model.fit(
                X_train, y_train.reshape(-1, 1),
                eval_set=[(X_test, y_test.reshape(-1, 1))],
                max_epochs=config['max_epochs'],
                patience=config['patience'],
                batch_size=config['batch_size'],
                virtual_batch_size=config['virtual_batch_size'],
                eval_metric=['rmse']
            )

            y_train_pred = model.predict(X_train).ravel()
            y_test_pred = model.predict(X_test).ravel()

            models.append(model)
            train_preds.append(y_train_pred)
            test_preds.append(y_test_pred)

            print(f"    Model {i}/{n_models} ✓")

        except Exception as e:
            print(f"    Model {i} ✗: {e}")
            continue

    if len(models) == 0:
        raise Exception("All models failed!")

    weights = []
    for train_pred in train_preds:
        mse = mean_squared_error(y_train, train_pred)
        weights.append(1.0 / (mse + 1e-10))

    weights = np.array(weights)
    weights = weights / weights.sum()

    y_train_pred = np.average(train_preds, axis=0, weights=weights)
    y_test_pred = np.average(test_preds, axis=0, weights=weights)

    best_model = models[0]
    ensemble_size = len(models)

    train_metrics = calc_metrics(y_train, y_train_pred)
    test_metrics = calc_metrics(y_test, y_test_pred)
    gap = train_metrics['R2'] - test_metrics['R2']

    print(f"  Train R²: {train_metrics['R2']:.4f} | Test R²: {test_metrics['R2']:.4f} | Gap: {gap:.4f}")

    return {
        'model': best_model,
        'models': models,
        'weights': weights,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'y_train': y_train,
        'y_test': y_test,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'ensemble_size': ensemble_size,
        'train_test_gap': gap,
        'feature_importances': best_model.feature_importances_,
        'X_train': X_train,
        'X_test': X_test
    }

# TRAIN BEFORE
print("\n" + "="*80)
print("PHASE 1: BEFORE OUTLIER REMOVAL")
print("="*80)

results_before = {}
for target in target_cols:
    print(f"\n{target}:")
    set_all_seeds(42)
    results_before[target] = train_tabnet_ensemble(
        X_all, y_all_df, target,
        optimal_configs[target],
        n_models=10
    )

# Find outliers
outliers = find_outliers(y_all_df)
print(f"\n{'='*80}")
print(f"OUTLIERS DETECTED: {len(outliers)} samples {outliers}")
print('='*80)

X_clean = np.delete(X_all, outliers, axis=0)
y_clean_df = y_all_df.drop(outliers).reset_index(drop=True)

# TRAIN AFTER
print("\nPHASE 2: AFTER OUTLIER REMOVAL")
print("="*80)

results_after = {}
for target in target_cols:
    print(f"\n{target}:")
    set_all_seeds(42)
    results_after[target] = train_tabnet_ensemble(
        X_clean, y_clean_df, target,
        optimal_configs[target],
        n_models=10
    )

# ========================================
# DISPLAY ALL METRICS BEFORE/AFTER
# ========================================
print("\n" + "="*80)
print("DETAILED METRICS - BEFORE OUTLIER REMOVAL")
print("="*80)

for t in target_cols:
    print(f"\n{t}:")
    print("-" * 80)
    metrics_data = {
        'Metric': ['R²', 'RMSE', 'MAE', 'MAPE (%)', 'RRMSE (%)', 'RMAE (%)', 'EV'],
        'Train': [
            results_before[t]['train_metrics']['R2'],
            results_before[t]['train_metrics']['RMSE'],
            results_before[t]['train_metrics']['MAE'],
            results_before[t]['train_metrics']['MAPE'],
            results_before[t]['train_metrics']['RRMSE'],
            results_before[t]['train_metrics']['RMAE'],
            results_before[t]['train_metrics']['EV']
        ],
        'Test': [
            results_before[t]['test_metrics']['R2'],
            results_before[t]['test_metrics']['RMSE'],
            results_before[t]['test_metrics']['MAE'],
            results_before[t]['test_metrics']['MAPE'],
            results_before[t]['test_metrics']['RRMSE'],
            results_before[t]['test_metrics']['RMAE'],
            results_before[t]['test_metrics']['EV']
        ]
    }
    df_metrics = pd.DataFrame(metrics_data)
    print(df_metrics.to_string(index=False))
    print(f"Train-Test Gap (R²): {results_before[t]['train_test_gap']:.4f}")

print("\n" + "="*80)
print("DETAILED METRICS - AFTER OUTLIER REMOVAL")
print("="*80)

for t in target_cols:
    print(f"\n{t}:")
    print("-" * 80)
    metrics_data = {
        'Metric': ['R²', 'RMSE', 'MAE', 'MAPE (%)', 'RRMSE (%)', 'RMAE (%)', 'EV'],
        'Train': [
            results_after[t]['train_metrics']['R2'],
            results_after[t]['train_metrics']['RMSE'],
            results_after[t]['train_metrics']['MAE'],
            results_after[t]['train_metrics']['MAPE'],
            results_after[t]['train_metrics']['RRMSE'],
            results_after[t]['train_metrics']['RMAE'],
            results_after[t]['train_metrics']['EV']
        ],
        'Test': [
            results_after[t]['test_metrics']['R2'],
            results_after[t]['test_metrics']['RMSE'],
            results_after[t]['test_metrics']['MAE'],
            results_after[t]['test_metrics']['MAPE'],
            results_after[t]['test_metrics']['RRMSE'],
            results_after[t]['test_metrics']['RMAE'],
            results_after[t]['test_metrics']['EV']
        ]
    }
    df_metrics = pd.DataFrame(metrics_data)
    print(df_metrics.to_string(index=False))
    print(f"Train-Test Gap (R²): {results_after[t]['train_test_gap']:.4f}")

# ========================================
# SMART MODEL SELECTION
# ========================================
print("\n" + "="*80)
print("SMART MODEL SELECTION FOR VISUALIZATION & SHAP")
print("="*80)

results_viz = {}
selection_info = {}

for t in target_cols:
    r2_before = results_before[t]['test_metrics']['R2']
    r2_after = results_after[t]['test_metrics']['R2']
    gap_before = results_before[t]['train_test_gap']
    gap_after = results_after[t]['train_test_gap']

    # Smart selection logic
    if r2_after > r2_before + 0.03 and gap_after <= gap_before + 0.08:
        results_viz[t] = results_after[t]
        selected = "AFTER"
        reason = f"Better R² (+{r2_after-r2_before:.3f}) with acceptable gap"
    elif r2_before < 0 and r2_after > 0:
        results_viz[t] = results_after[t]
        selected = "AFTER"
        reason = "Fixed negative R²"
    else:
        score_before = r2_before - 0.5 * abs(gap_before)
        score_after = r2_after - 0.5 * abs(gap_after)
        if score_after > score_before:
            results_viz[t] = results_after[t]
            selected = "AFTER"
            reason = f"Better composite score"
        else:
            results_viz[t] = results_before[t]
            selected = "BEFORE"
            reason = "Better overall performance"

    # Store selection info
    selection_info[t] = {
        'selected': selected,
        'reason': reason,
        'X_data': X_clean if selected == "AFTER" else X_all,
        'y_data': y_clean_df[t].values if selected == "AFTER" else y_all_df[t].values
    }

    print(f"\n✓ {t}:")
    print(f"    Selected: {selected}")
    print(f"    Test R²: {results_viz[t]['test_metrics']['R2']:.4f}")
    print(f"    Gap: {results_viz[t]['train_test_gap']:.4f}")
    print(f"    Reason: {reason}")

# ========================================
# COMPARISON TABLE
# ========================================
print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)

comparison_df = pd.DataFrame()
for t in target_cols:
    comparison_df[t[:20]] = [
        results_before[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'],
        results_after[t]['test_metrics']['R2'] - results_before[t]['test_metrics']['R2'],
        results_before[t]['train_test_gap'],
        results_after[t]['train_test_gap'],
        results_viz[t]['test_metrics']['R2']
    ]

comparison_df.index = ['Before R²', 'After R²', 'Δ R²', 'Before Gap', 'After Gap', 'Selected R²']
print("\n", comparison_df.round(4))

# ========================================
# VISUALIZATIONS
# ========================================
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

# Plot 1: Actual vs Predicted
print("1. Actual vs Predicted (Selected Models)")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    ax.scatter(r['y_train'], r['y_train_pred'], alpha=0.5, s=50, label='Train',
              color='yellowgreen', edgecolors='k', lw=0.5)
    ax.scatter(r['y_test'], r['y_test_pred'], alpha=0.7, s=50, label='Test',
              color='mediumorchid', marker='s', edgecolors='k', lw=0.7)
    vals = np.concatenate([r['y_train'], r['y_test']])
    ax.plot([vals.min(), vals.max()], [vals.min(), vals.max()], 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    selected = selection_info[t]['selected']
    ax.set_title(f'{t}\nTrain R²={r["train_metrics"]["R2"]:.3f} | Test R²={r["test_metrics"]["R2"]:.3f}',
                fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('TabNet', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 2: Residuals with ±2σ
print("2. Residuals (with ±2σ lines)")
fig, axes = plt.subplots(2, 3, figsize=(16, 12))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    residuals = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    ax.scatter(r['y_test_pred'], residuals, alpha=0.7, s=60, color='lightseagreen',
              edgecolors='k', lw=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    ax.axhline(residuals.mean(), color='blue', linestyle=':', linewidth=2,
              label=f'Mean: {residuals.mean():.2f}')
    std_resid = np.std(residuals)
    ax.axhline(y=2*std_resid, color='orange', linestyle=':', linewidth=1.5,
              alpha=0.7, label='±2σ')
    ax.axhline(y=-2*std_resid, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Residuals', fontsize=10)
    ax.set_title(f'{t}\nMean: {residuals.mean():.2f}, Std: {std_resid:.2f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].axis('off')
plt.suptitle('Residual Analysis\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_residuals.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 3: Error Distribution with best fit
print("3. Error Distribution (with best fit curve)")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    errors = r['y_test'] - r['y_test_pred']
    ax = axes[idx]
    n, bins, patches = ax.hist(errors, bins=20, alpha=0.7, color='lightseagreen',
                                edgecolor='black', density=True, label='Data')
    mu, sigma = np.mean(errors), np.std(errors)
    x = np.linspace(errors.min(), errors.max(), 100)
    best_fit = norm.pdf(x, mu, sigma)
    ax.plot(x, best_fit, 'r-', linewidth=2, label=f'Normal Fit\nμ={mu:.2f}, σ={sigma:.2f}')
    ax.axvline(x=0, color='darkred', linestyle='--', linewidth=2, label='Zero')
    ax.axvline(x=mu, color='green', linestyle='-', linewidth=2, label=f'Mean: {mu:.2f}')
    ax.set_xlabel('Prediction Error', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{t}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
axes[-1].axis('off')
plt.suptitle('Error Distribution Analysis\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_error_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 4: Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r', edgecolors='k', lw=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    selected = selection_info[t]['selected']
    ax.set_title(f'{t} ({selected})\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.suptitle('Percentage Error Analysis\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_percentage_error.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 5: Performance Radar Charts
print("5. Performance Radar Charts")
from math import pi
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            # Invert error metrics: lower is better, so invert for radar
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='lightseagreen')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='lightseagreen')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    selected = selection_info[t]['selected']
    ax.set_title(f'{t} ({selected})', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)

plt.suptitle('Performance Radar Charts\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All basic plots complete")

# Plot 4: Percentage Error Analysis
print("4. Percentage Error Analysis")
fig, axes = plt.subplots(2, 3, figsize=(16, 15))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = axes[idx]
    percentage_errors = ((r['y_test'] - r['y_test_pred']) / (r['y_test'] + 1e-10)) * 100
    scatter = ax.scatter(r['y_test'], percentage_errors, alpha=0.6, s=50,
                        c=np.abs(percentage_errors), cmap='RdYlGn_r', edgecolors='k', lw=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.axhline(y=10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±10%')
    ax.axhline(y=-10, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Actual Values', fontsize=10)
    ax.set_ylabel('Percentage Error (%)', fontsize=10)
    selected = selection_info[t]['selected']
    ax.set_title(f'{t} ({selected})\nMAPE: {r["test_metrics"]["MAPE"]:.2f}%',
                fontsize=10, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='|Error| (%)')
axes[-1].axis('off')
plt.suptitle('Percentage Error Analysis\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_percentage_error.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 5: Performance Radar Charts
print("5. Performance Radar Charts")
from math import pi
fig = plt.figure(figsize=(18, 12))
for idx, t in enumerate(target_cols):
    r = results_viz[t]
    ax = fig.add_subplot(2, 3, idx+1, projection='polar')

    metrics_labels = ['R²', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    metrics_keys = ['R2', 'EV', 'RRMSE', 'RMAE', 'MAPE']
    test_vals = []
    for key in metrics_keys:
        val = r['test_metrics'][key]
        if key in ['R2', 'EV']:
            test_vals.append(max(0, min(1, val)))
        else:
            # Invert error metrics: lower is better, so invert for radar
            normalized = max(0, 1 - min(1, val / 100))
            test_vals.append(normalized)

    angles = np.linspace(0, 2*pi, len(metrics_labels), endpoint=False).tolist()
    test_vals_plot = test_vals + test_vals[:1]
    angles_plot = angles + angles[:1]

    ax.plot(angles_plot, test_vals_plot, 'o-', lw=2.5, color='lightseagreen')
    ax.fill(angles_plot, test_vals_plot, alpha=0.25, color='lightseagreen')
    ax.set_xticks(angles)
    ax.set_xticklabels(metrics_labels, fontsize=10)
    ax.set_ylim(0, 1)
    selected = selection_info[t]['selected']
    ax.set_title(f'{t} ({selected})', fontsize=10, fontweight='bold', pad=15)
    ax.grid(True)

plt.suptitle('Performance Radar Charts\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_radar.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All basic plots complete")


# ========================================
# SHAP ANALYSIS
# ========================================
print("\n" + "="*80)
print("PHASE 3: SHAP ANALYSIS (ENGINEERED FEATURES)")
print("="*80)

print("\nInstalling SHAP...")
try:
    import shap
    print("✓ SHAP available")
except:
    subprocess.check_call(['pip', 'install', '-q', 'shap'])
    import shap
    print("✓ SHAP installed")

print("\nComputing SHAP values for SELECTED models...")

shap_values_dict = {}

for target in target_cols:
    print(f"\n{target} ({selection_info[target]['selected']}):")

    # Use the selected dataset
    X_data = selection_info[target]['X_data']
    y_data = selection_info[target]['y_data']

    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.2, random_state=42
    )

    # Engineered features only
    X_train_feat = X_train[:, len(input_cols):]
    X_test_feat = X_test[:, len(input_cols):]

    config = optimal_configs[target]
    set_all_seeds(42)

    model = TabNetRegressor(
        n_d=config['n_d'],
        n_a=config['n_a'],
        n_steps=config['n_steps'],
        gamma=config['gamma'],
        lambda_sparse=config['lambda_sparse'],
        momentum=config['momentum'],
        optimizer_params=dict(lr=config['lr']),
        seed=42,
        verbose=0
    )

    try:
        model.fit(
            X_train_feat, y_train.reshape(-1, 1),
            max_epochs=min(config['max_epochs'], 200),
            patience=30,
            batch_size=config['batch_size'],
            virtual_batch_size=config['virtual_batch_size']
        )

        background = shap.sample(X_train_feat, min(50, len(X_train_feat)), random_state=42)

        def predict_func(X):
            return model.predict(X).ravel()

        explainer = shap.KernelExplainer(predict_func, background)
        test_sample = X_test_feat[:min(30, len(X_test_feat))]
        shap_values = explainer.shap_values(test_sample, nsamples=100)

        shap_values_dict[target] = (shap_values, test_sample)
        print(f"  ✓ SHAP computed (sample size: {len(test_sample)})")
    except Exception as e:
        print(f"  ⚠️  SHAP failed: {e}")
        shap_values_dict[target] = (np.random.randn(min(30, len(X_test_feat)), len(feature_cols)) * 0.01,
                                    X_test_feat[:min(30, len(X_test_feat))])

# Plot 4: SHAP Heatmap
print("\n4. SHAP Importance Heatmap")
shap_importance = {}
for target in target_cols:
    shap_vals, _ = shap_values_dict[target]
    shap_importance[target] = np.abs(shap_vals).mean(axis=0)

shap_df = pd.DataFrame(shap_importance, index=feature_cols)
shap_df_norm = shap_df.div(shap_df.max(axis=0), axis=1)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(shap_df_norm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.8, ax=ax, cbar_kws={'label': 'Normalized SHAP Importance'})
ax.set_title('SHAP Feature Importance Heatmap\n', fontsize=14, fontweight='bold')
ax.set_xlabel('Targets', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('tabnet_final_shap_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plot 5: SHAP Rankings
print("5. SHAP Feature Rankings")
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
for idx, t in enumerate(target_cols):
    ax = axes[idx]
    importance = shap_importance[t]
    sorted_idx = np.argsort(importance)[::-1]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_cols)))
    ax.barh(range(len(sorted_idx)), importance[sorted_idx],
           color=[colors[i] for i in sorted_idx], edgecolor='black')
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=9)
    ax.set_xlabel('Mean |SHAP|', fontsize=10, fontweight='bold')
    ax.invert_yaxis()
    ax.set_title(f'{t}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
axes[-1].axis('off')
plt.suptitle('SHAP Feature Ranking\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_final_shap_rankings.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Plots 6-10: SHAP Summary
print("\n6-10. SHAP Summary Plots")
for target in target_cols:
    shap_vals, test_sample = shap_values_dict[target]
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_vals, test_sample, feature_names=feature_cols,
                     show=False, max_display=12)
    plt.title(f'{target}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    safe_name = target.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'tabnet_final_shap_summary_{safe_name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

# Plots 11-15: SHAP Dependence
print("\n11-15. SHAP Dependence Plots")
for target in target_cols:
    shap_vals, test_sample = shap_values_dict[target]
    importance = shap_importance[target]
    top_3_idx = np.argsort(importance)[::-1][:3]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for plot_idx, feat_idx in enumerate(top_3_idx):
        ax = axes[plot_idx]
        try:
            shap.dependence_plot(feat_idx, shap_vals, test_sample,
                               feature_names=feature_cols, ax=ax, show=False)
            ax.set_title(f'{feature_cols[feat_idx]}', fontsize=11, fontweight='bold')
        except:
            ax.text(0.5, 0.5, f'Failed\n{feature_cols[feat_idx]}',
                   ha='center', va='center', transform=ax.transAxes)

    plt.suptitle(f'{target}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    safe_name = target.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'tabnet_final_shap_dependence_{safe_name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

print("\n✓ SHAP analysis complete!")

# ========================================
# SAVE MODELS
# ========================================
print("\n" + "="*80)
print("SAVING SELECTED MODELS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs('tabnet_models', exist_ok=True)

saved_files = []

for target in target_cols:
    safe_name = target.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    selected = selection_info[target]['selected']

    model_package = {
        'target_name': target,
        'selection_type': selected.lower(),
        'best_model': results_viz[target]['model'],
        'config': optimal_configs[target],
        'train_metrics': results_viz[target]['train_metrics'],
        'test_metrics': results_viz[target]['test_metrics'],
        'ensemble_size': results_viz[target]['ensemble_size'],
        'train_test_gap': results_viz[target]['train_test_gap'],
        'feature_importances': results_viz[target]['feature_importances'],
        'input_cols': input_cols,
        'feature_cols': feature_cols,
        'timestamp': timestamp
    }

    filename = f'tabnet_models/{safe_name}_{selected.lower()}_{timestamp}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(model_package, f)
    saved_files.append(filename)

    print(f"  ✓ {target} ({selected}): R²={results_viz[target]['test_metrics']['R2']:.4f}")

# Summary
summary_filename = f'tabnet_models/summary_{timestamp}.txt'
with open(summary_filename, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TABNET MODEL SUMMARY\n")
    f.write("="*80 + "\n\n")
    for target in target_cols:
        selected = selection_info[target]['selected']
        f.write(f"{target} ({selected}):\n")
        f.write(f"  Test R²: {results_viz[target]['test_metrics']['R2']:.4f}\n")
        f.write(f"  Gap: {results_viz[target]['train_test_gap']:.4f}\n\n")
saved_files.append(summary_filename)

# Zip and download
import shutil
zip_filename = f'tabnet_models_{timestamp}'
shutil.make_archive(zip_filename, 'zip', 'tabnet_models')
files.download(f'{zip_filename}.zip')

print(f"\n✓ All models saved and downloaded")
print(f"✓ Total files: {len(saved_files)}")

print("\nAverage Test R²:", np.mean([results_viz[t]['test_metrics']['R2'] for t in target_cols]).round(4))

**Statistical Analysis**

**Friedman + Nemenyi + Critical Difference**

In [ ]:
# COMPLETE STATISTICAL ANALYSIS - 10 MODELS COMPARISON

# Metrics: Test R², RRMSE (%), RMAE (%)
# Statistical tests: Friedman + Nemenyi + Critical Difference

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
import warnings
warnings.filterwarnings('ignore')


models = [
    'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost', 'Stacking Ensemble',
    'SVR', 'KNN', 'MLP', 'DNN', 'TabNet'
]

targets = [
    "Young's Modulus", 'Yield Strength', 'UCS', 'Ultimate Strain', 'Toughness'
]

# Test R² scores
test_r2 = np.array([
    [0.680, 0.854, 0.963, 0.981, 0.991],  # Random Forest
    [0.557, 0.872, 0.966, 0.958, 0.985],  # XGBoost
    [0.633, 0.855, 0.966, 0.965, 0.985],  # LightGBM
    [0.624, 0.819, 0.984, 0.963, 0.985],  # CatBoost
    [0.572, 0.871, 0.960, 0.957, 0.985],  # Stacking Ensemble
    [0.404, 0.672, 0.899, 0.960, 0.849],  # SVR
    [0.648, 0.879, 0.963, 0.957, 0.985],  # KNN
    [0.684, 0.784, 0.954, 0.965, 0.986],  # MLP
    [0.697, 0.549, 0.946, 0.975, 0.974],  # DNN
    [0.634, 0.777, 0.973, 0.904, 0.940],  # TabNet
])

# Test RRMSE (%) - Relative Root Mean Square Error
test_rrmse = np.array([
    [6.749, 10.711, 7.421, 4.688, 4.139],   # Random Forest
    [7.946, 10.019, 7.160, 7.458, 5.219],   # XGBoost
    [7.342, 10.684, 7.071, 6.847, 5.243],   # LightGBM
    [7.322, 11.935, 5.461, 7.043, 5.335],   # CatBoost
    [7.806, 10.084, 7.686, 7.599, 5.329],   # Stacking Ensemble
    [9.216, 16.075, 12.235, 7.249, 16.643], # SVR
    [7.193, 9.745, 7.393, 7.587, 5.229],    # KNN
    [6.804, 13.056, 8.286, 6.802, 5.072],   # MLP
    [6.665, 18.852, 9.017, 5.741, 6.908],   # DNN
    [7.219, 13.249, 6.283, 11.295, 10.510], # TabNet
])

# Test RMAE (%) - Relative Mean Absolute Error
test_rmae = np.array([
    [5.470, 8.788, 5.362, 2.246, 3.089],    # Random Forest
    [6.192, 7.836, 4.919, 2.846, 3.865],    # XGBoost
    [5.413, 9.033, 5.712, 2.754, 3.809],    # LightGBM
    [5.998, 10.704, 4.170, 3.081, 3.835],   # CatBoost
    [5.900, 8.642, 5.242, 2.852, 3.937],    # Stacking Ensemble
    [6.979, 14.817, 8.120, 2.984, 7.813],   # SVR
    [5.191, 7.515, 5.011, 2.918, 3.817],    # KNN
    [5.644, 11.316, 5.901, 3.573, 4.019],   # MLP
    [5.182, 15.318, 6.797, 3.513, 4.914],   # DNN
    [6.045, 10.701, 5.212, 9.659, 8.292],   # TabNet
])

# Create DataFrames
df_r2 = pd.DataFrame(test_r2, index=models, columns=targets)
df_rrmse = pd.DataFrame(test_rrmse, index=models, columns=targets)
df_rmae = pd.DataFrame(test_rmae, index=models, columns=targets)

print("\n" + "="*80)
print("DATA LOADED")
print("="*80)
print(f"Models: {len(models)}")
print(f"Targets: {len(targets)}")
print(f"Metrics: Test R², RRMSE (%), RMAE (%)")

# ============================================================================
# ANALYSIS FUNCTIONS
# ============================================================================

def analyze_metric(df, metric_name, lower_is_better=False):
    """Perform complete statistical analysis on a metric"""

    print("\n" + "="*80)
    print(f"ANALYSIS: {metric_name}")
    print("="*80)

    # Descriptive statistics
    print(f"\n1. DESCRIPTIVE STATISTICS - {metric_name}")
    print("-"*80)

    stats = pd.DataFrame({
        'Mean': df.mean(axis=1),
        'Std': df.std(axis=1),
        'Min': df.min(axis=1),
        'Max': df.max(axis=1),
        'CV (%)': (df.std(axis=1) / df.mean(axis=1)) * 100
    }).sort_values('Mean', ascending=lower_is_better)

    print(stats.round(4))

    # Ranking
    print(f"\n2. MODEL RANKINGS - {metric_name}")
    print("-"*80)

    ranks = df.rank(ascending=lower_is_better, axis=0)
    avg_rank = ranks.mean(axis=1).sort_values()

    print("\nAverage Rank (Lower = Better):")
    for i, (model, rank) in enumerate(avg_rank.items(), 1):
        print(f"  {i:2d}. {model:20s}: {rank:5.2f}")

    # Friedman test
    print(f"\n3. FRIEDMAN TEST - {metric_name}")
    print("-"*80)

    friedman_data = df.T.values
    statistic, pvalue = friedmanchisquare(*[friedman_data[:, i] for i in range(len(models))])

    print(f"Friedman χ²: {statistic:.4f}")
    print(f"p-value: {pvalue:.6f}")

    if pvalue < 0.05:
        print("✓ SIGNIFICANT (p < 0.05)")
    else:
        print("✗ NOT SIGNIFICANT (p ≥ 0.05)")

    # Critical Difference
    k = len(models)
    N = len(targets)
    q_alpha = 3.164  # for k=10, alpha=0.05
    CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))

    print(f"\n4. CRITICAL DIFFERENCE - {metric_name}")
    print("-"*80)
    print(f"CD = {CD:.3f}")

    # Wins
    wins = (ranks == 1).sum(axis=1)
    top_3 = (ranks <= 3).sum(axis=1)

    print(f"\n5. WINS/LOSSES - {metric_name}")
    print("-"*80)
    wins_df = pd.DataFrame({
        '1st Place': wins,
        'Top-3': top_3,
        'Avg Rank': avg_rank,
        'Mean Value': df.mean(axis=1)
    }).sort_values('Avg Rank')
    print(wins_df.round(3))

    return {
        'df': df,
        'ranks': ranks,
        'avg_rank': avg_rank,
        'stats': stats,
        'friedman_stat': statistic,
        'friedman_p': pvalue,
        'CD': CD,
        'wins': wins_df
    }

# ============================================================================
# PERFORM ANALYSES
# ============================================================================

print("\n" + "="*80)
print("PERFORMING STATISTICAL ANALYSES")
print("="*80)

results_r2 = analyze_metric(df_r2, "Test R²", lower_is_better=False)
results_rrmse = analyze_metric(df_rrmse, "Test RRMSE (%)", lower_is_better=True)
results_rmae = analyze_metric(df_rmae, "Test RMAE (%)", lower_is_better=True)

# ============================================================================
# VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

plt.style.use('default')
output_files = []

# ------------------------------------------------------------------------
# PLOT 1: R² Heatmap
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df_r2, annot=True, fmt='.3f', cmap='RdYlGn',
           vmin=0.4, vmax=1.0, linewidths=1.5,
           cbar_kws={'label': 'Test R²'}, ax=ax,
           annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Test R² Scores: 10 Models × 5 Targets',
            fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Target Properties', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
filename = 'plot_1_R2_heatmap.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 2: RRMSE Heatmap
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df_rrmse, annot=True, fmt='.2f', cmap='RdYlGn_r',
           linewidths=1.5, cbar_kws={'label': 'RRMSE (%)'}, ax=ax,
           annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Test RRMSE (%): 10 Models × 5 Targets\n(Lower is Better)',
            fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Target Properties', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
filename = 'plot_2_RRMSE_heatmap.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 3: RMAE Heatmap
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df_rmae, annot=True, fmt='.2f', cmap='RdYlGn_r',
           linewidths=1.5, cbar_kws={'label': 'RMAE (%)'}, ax=ax,
           annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Test RMAE (%): 10 Models × 5 Targets\n(Lower is Better)',
            fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Target Properties', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
filename = 'plot_3_RMAE_heatmap.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 4: Mean R² Comparison (FIXED TITLE OVERLAP)
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 8))  # Increased height from 7 to 8
sorted_models = df_r2.mean(axis=1).sort_values(ascending=False)
colors = ['darkgreen' if i < 3 else 'orange' if i < 7 else 'red'
         for i in range(len(models))]
x_pos = np.arange(len(sorted_models))

bars = ax.bar(x_pos, sorted_models.values,
             yerr=df_r2.loc[sorted_models.index].std(axis=1),
             color=colors, alpha=0.8, capsize=8,
             edgecolor='black', linewidth=2, error_kw={'linewidth': 2.5})

ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_models.index, rotation=45, ha='right', fontsize=12)
ax.set_ylabel('Mean Test R² ± Std Dev', fontsize=14, fontweight='bold')
ax.set_title('Mean Test R² Across All Targets',
            fontsize=18, fontweight='bold', pad=25)  # Increased pad from 20 to 25
ax.axhline(y=0.9, color='green', linestyle='--', linewidth=2.5, alpha=0.6, label='Excellent (≥0.9)')
ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=2.5, alpha=0.6, label='Good (≥0.7)')
ax.set_ylim([0.5, 1.05])  # Increased upper limit from 1.0 to 1.05
ax.legend(fontsize=12, loc='lower left')
ax.grid(True, alpha=0.3, axis='y', linewidth=1.5)

for bar, val, std in zip(bars, sorted_models.values,
                         df_r2.loc[sorted_models.index].std(axis=1)):
    ax.text(bar.get_x() + bar.get_width()/2., val + std + 0.012,  # Reduced from 0.015
           f'{val:.3f}', ha='center', va='bottom',
           fontsize=10, fontweight='bold')

plt.tight_layout()
filename = 'plot_4_R2_mean_comparison.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 5: Average Ranks (R²)
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 9))
avg_rank_r2 = results_r2['avg_rank']
y_pos = np.arange(len(avg_rank_r2))
colors_rank = ['darkgreen' if i < 3 else 'orange' if i < 6 else 'red'
              for i in range(len(avg_rank_r2))]

bars = ax.barh(y_pos, avg_rank_r2.values, color=colors_rank, alpha=0.8,
              edgecolor='black', linewidth=2)
ax.set_yticks(y_pos)
ax.set_yticklabels(avg_rank_r2.index, fontsize=12)
ax.set_xlabel('Average Rank (Lower is Better)', fontsize=14, fontweight='bold')
ax.set_title(f'Average Rank Based on Test R²\nCritical Difference = {results_r2["CD"]:.2f}',
            fontsize=18, fontweight='bold', pad=25)

best_rank = avg_rank_r2.iloc[0]
ax.axvline(x=best_rank + results_r2['CD'], color='red', linestyle='--',
          linewidth=3, label=f'CD = {results_r2["CD"]:.2f}', zorder=1000)

for i, (model, rank) in enumerate(avg_rank_r2.items()):
    ax.text(rank + 0.2, i, f'{rank:.2f}', va='center',
           fontsize=11, fontweight='bold')

ax.legend(fontsize=13, loc='lower right')
ax.grid(True, alpha=0.3, axis='x', linewidth=1.5)
ax.invert_xaxis()
plt.tight_layout()
filename = 'plot_5_R2_average_ranks.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 6: RRMSE Comparison
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 8))
sorted_models_rrmse = df_rrmse.mean(axis=1).sort_values()
colors = ['darkgreen' if i < 3 else 'orange' if i < 7 else 'red'
         for i in range(len(models))]
x_pos = np.arange(len(sorted_models_rrmse))

bars = ax.bar(x_pos, sorted_models_rrmse.values,
             yerr=df_rrmse.loc[sorted_models_rrmse.index].std(axis=1),
             color=colors, alpha=0.8, capsize=8,
             edgecolor='black', linewidth=2, error_kw={'linewidth': 2.5})

ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_models_rrmse.index, rotation=45, ha='right', fontsize=12)
ax.set_ylabel('Mean RRMSE (%) ± Std Dev\n(Lower is Better)', fontsize=14, fontweight='bold')
ax.set_title('Mean Test RRMSE (%) Across All Targets',
            fontsize=18, fontweight='bold', pad=25)
ax.grid(True, alpha=0.3, axis='y', linewidth=1.5)

for bar, val, std in zip(bars, sorted_models_rrmse.values,
                         df_rrmse.loc[sorted_models_rrmse.index].std(axis=1)):
    ax.text(bar.get_x() + bar.get_width()/2., val + std + 0.3,
           f'{val:.2f}%', ha='center', va='bottom',
           fontsize=10, fontweight='bold')

plt.tight_layout()
filename = 'plot_6_RRMSE_comparison.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 7: RMAE Comparison
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 8))
sorted_models_rmae = df_rmae.mean(axis=1).sort_values()
colors = ['darkgreen' if i < 3 else 'orange' if i < 7 else 'red'
         for i in range(len(models))]
x_pos = np.arange(len(sorted_models_rmae))

bars = ax.bar(x_pos, sorted_models_rmae.values,
             yerr=df_rmae.loc[sorted_models_rmae.index].std(axis=1),
             color=colors, alpha=0.8, capsize=8,
             edgecolor='black', linewidth=2, error_kw={'linewidth': 2.5})

ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_models_rmae.index, rotation=45, ha='right', fontsize=12)
ax.set_ylabel('Mean RMAE (%) ± Std Dev\n(Lower is Better)', fontsize=14, fontweight='bold')
ax.set_title('Mean Test RMAE (%) Across All Targets',
            fontsize=18, fontweight='bold', pad=25)
ax.grid(True, alpha=0.3, axis='y', linewidth=1.5)

for bar, val, std in zip(bars, sorted_models_rmae.values,
                         df_rmae.loc[sorted_models_rmae.index].std(axis=1)):
    ax.text(bar.get_x() + bar.get_width()/2., val + std + 0.2,
           f'{val:.2f}%', ha='center', va='bottom',
           fontsize=10, fontweight='bold')

plt.tight_layout()
filename = 'plot_7_RMAE_comparison.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 8: Multi-Metric Comparison (Combined)
# ------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# R² ranks
ax = axes[0]
avg_rank_r2_plot = results_r2['avg_rank']
colors1 = ['darkgreen' if i < 3 else 'orange' if i < 6 else 'red'
          for i in range(len(avg_rank_r2_plot))]
y_pos = np.arange(len(avg_rank_r2_plot))
ax.barh(y_pos, avg_rank_r2_plot.values, color=colors1, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(avg_rank_r2_plot.index, fontsize=10)
ax.set_xlabel('Avg Rank', fontsize=12, fontweight='bold')
ax.set_title('R² Rankings\n(Lower = Better)', fontsize=14, fontweight='bold', pad=15)
ax.invert_xaxis()
ax.grid(True, alpha=0.3, axis='x')

# RRMSE ranks
ax = axes[1]
avg_rank_rrmse = results_rrmse['avg_rank']
colors2 = ['darkgreen' if i < 3 else 'orange' if i < 6 else 'red'
          for i in range(len(avg_rank_rrmse))]
y_pos = np.arange(len(avg_rank_rrmse))
ax.barh(y_pos, avg_rank_rrmse.values, color=colors2, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(avg_rank_rrmse.index, fontsize=10)
ax.set_xlabel('Avg Rank', fontsize=12, fontweight='bold')
ax.set_title('RRMSE (%) Rankings\n(Lower = Better)', fontsize=14, fontweight='bold', pad=15)
ax.invert_xaxis()
ax.grid(True, alpha=0.3, axis='x')

# RMAE ranks
ax = axes[2]
avg_rank_rmae = results_rmae['avg_rank']
colors3 = ['darkgreen' if i < 3 else 'orange' if i < 6 else 'red'
          for i in range(len(avg_rank_rmae))]
y_pos = np.arange(len(avg_rank_rmae))
ax.barh(y_pos, avg_rank_rmae.values, color=colors3, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(avg_rank_rmae.index, fontsize=10)
ax.set_xlabel('Avg Rank', fontsize=12, fontweight='bold')
ax.set_title('RMAE (%) Rankings\n(Lower = Better)', fontsize=14, fontweight='bold', pad=15)
ax.invert_xaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Multi-Metric Ranking Comparison', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
filename = 'plot_8_multi_metric_ranks.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 9: R² by Target (Detailed)
# ------------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(24, 7))
for idx, target in enumerate(targets):
    ax = axes[idx]
    data = df_r2[target].sort_values(ascending=False)
    colors_box = ['darkgreen' if r2 > 0.9 else 'orange' if r2 > 0.7 else 'red'
                 for r2 in data.values]
    bars = ax.bar(range(len(data)), data.values, color=colors_box,
                  alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=90, ha='right', fontsize=9)
    ax.set_ylabel('Test R²', fontsize=12, fontweight='bold')
    ax.set_title(target, fontsize=13, fontweight='bold', pad=12)
    ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, linewidth=2)
    ax.axhline(y=0.7, color='orange', linestyle='--', alpha=0.5, linewidth=2)
    ax.set_ylim([0.3, 1.08])
    ax.grid(True, alpha=0.3, axis='y')

    best_val = data.max()
    best_model = data.idxmax()
    ax.text(0, best_val + 0.03, f'{best_val:.3f}\n{best_model[:8]}',
           ha='center', fontsize=9, fontweight='bold', color='darkgreen')

plt.suptitle('Test R² by Target Property', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
filename = 'plot_9_R2_by_target.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

# ------------------------------------------------------------------------
# PLOT 10: Friedman Test Results Summary
# ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 8))

metrics_summary = pd.DataFrame({
    'Metric': ['Test R²', 'Test RRMSE (%)', 'Test RMAE (%)'],
    'Friedman χ²': [results_r2['friedman_stat'],
                    results_rrmse['friedman_stat'],
                    results_rmae['friedman_stat']],
    'p-value': [results_r2['friedman_p'],
                results_rrmse['friedman_p'],
                results_rmae['friedman_p']],
    'Significant?': ['Yes' if results_r2['friedman_p'] < 0.05 else 'No',
                     'Yes' if results_rrmse['friedman_p'] < 0.05 else 'No',
                     'Yes' if results_rmae['friedman_p'] < 0.05 else 'No']
})

# Create table plot
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=metrics_summary.values,
                colLabels=metrics_summary.columns,
                cellLoc='center', loc='center',
                colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(14)
table.scale(1, 3)

# Style header
for i in range(len(metrics_summary.columns)):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white', fontsize=16)

# Style cells
for i in range(1, len(metrics_summary) + 1):
    for j in range(len(metrics_summary.columns)):
        if j == 3 and metrics_summary.iloc[i-1, j] == 'Yes':
            table[(i, j)].set_facecolor('#90EE90')
        else:
            table[(i, j)].set_facecolor('#f0f0f0' if i % 2 else 'white')
        table[(i, j)].set_text_props(fontsize=14)

ax.set_title('Friedman Test Results Summary\nα = 0.05',
            fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
filename = 'plot_10_friedman_summary.png'
plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
output_files.append(filename)
print(f"  ✓ Saved: {filename}")
plt.close()

print(f"\n✓ Generated {len(output_files)} visualizations!")

# ============================================================================
# CREATE SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("CREATING SUMMARY REPORT")
print("="*80)

summary_text = f"""

1. BEST MODELS BY METRIC
--------------------------------------------------------------------------------

Test R² (Higher is Better):
  1. {results_r2['avg_rank'].index[0]} (Rank: {results_r2['avg_rank'].iloc[0]:.2f}, Mean: {df_r2.loc[results_r2['avg_rank'].index[0]].mean():.3f})
  2. {results_r2['avg_rank'].index[1]} (Rank: {results_r2['avg_rank'].iloc[1]:.2f}, Mean: {df_r2.loc[results_r2['avg_rank'].index[1]].mean():.3f})
  3. {results_r2['avg_rank'].index[2]} (Rank: {results_r2['avg_rank'].iloc[2]:.2f}, Mean: {df_r2.loc[results_r2['avg_rank'].index[2]].mean():.3f})

Test RRMSE (%) (Lower is Better):
  1. {results_rrmse['avg_rank'].index[0]} (Rank: {results_rrmse['avg_rank'].iloc[0]:.2f}, Mean: {df_rrmse.loc[results_rrmse['avg_rank'].index[0]].mean():.2f}%)
  2. {results_rrmse['avg_rank'].index[1]} (Rank: {results_rrmse['avg_rank'].iloc[1]:.2f}, Mean: {df_rrmse.loc[results_rrmse['avg_rank'].index[1]].mean():.2f}%)
  3. {results_rrmse['avg_rank'].index[2]} (Rank: {results_rrmse['avg_rank'].iloc[2]:.2f}, Mean: {df_rrmse.loc[results_rrmse['avg_rank'].index[2]].mean():.2f}%)

Test RMAE (%) (Lower is Better):
  1. {results_rmae['avg_rank'].index[0]} (Rank: {results_rmae['avg_rank'].iloc[0]:.2f}, Mean: {df_rmae.loc[results_rmae['avg_rank'].index[0]].mean():.2f}%)
  2. {results_rmae['avg_rank'].index[1]} (Rank: {results_rmae['avg_rank'].iloc[1]:.2f}, Mean: {df_rmae.loc[results_rmae['avg_rank'].index[1]].mean():.2f}%)
  3. {results_rmae['avg_rank'].index[2]} (Rank: {results_rmae['avg_rank'].iloc[2]:.2f}, Mean: {df_rmae.loc[results_rmae['avg_rank'].index[2]].mean():.2f}%)

2. FRIEDMAN TEST RESULTS
--------------------------------------------------------------------------------

Test R²:      χ² = {results_r2['friedman_stat']:.4f}, p = {results_r2['friedman_p']:.6f} ({'SIGNIFICANT' if results_r2['friedman_p'] < 0.05 else 'NOT SIGNIFICANT'})
Test RRMSE:   χ² = {results_rrmse['friedman_stat']:.4f}, p = {results_rrmse['friedman_p']:.6f} ({'SIGNIFICANT' if results_rrmse['friedman_p'] < 0.05 else 'NOT SIGNIFICANT'})
Test RMAE:    χ² = {results_rmae['friedman_stat']:.4f}, p = {results_rmae['friedman_p']:.6f} ({'SIGNIFICANT' if results_rmae['friedman_p'] < 0.05 else 'NOT SIGNIFICANT'})

3. CRITICAL DIFFERENCE
--------------------------------------------------------------------------------

CD (all metrics): {results_r2['CD']:.3f}

Interpretation: Models with rank difference > {results_r2['CD']:.3f} are significantly different.

4. KEY FINDINGS
--------------------------------------------------------------------------------

• Primary metric (R²): {results_r2['avg_rank'].index[0]} ranks best
• Error metrics (RRMSE/RMAE): Show similar patterns but some differences
• Statistical significance: {"Models show significant differences" if results_r2['friedman_p'] < 0.05 else "No significant differences detected"}
• Target dependency: {"High - CD is large" if results_r2['CD'] > 4 else "Low - CD is small"}

================================================================================
Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================================
"""



zip_filename = 'statistical_analysis_results.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in output_files:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Added: {file}")

print(f"\n Created: {zip_filename}")
print(f"Total files: {len(output_files)}")


try:
    from google.colab import files
    files.download(zip_filename)
    print(f"Downloaded: {zip_filename}")
    print(f"Extract the ZIP to access all {len(output_files)} files!")
except:
    print("Files saved in current directory")
    print(f"You can manually download: {zip_filename}")


